# 04_02 - Panel SER barrio-intervalo

Este notebook construye progresivamente el panel `SER_barrio_intervalo` a partir de la base de tiques SER depurada en `04_01_ser_joins_base.ipynb`. El punto de partida no es una tabla agregada, sino eventos individuales de estacionamiento regulado con `fecha_inicio`, `fecha_fin` y `barrio_key`.

El objetivo de este notebook es transformar esos eventos individuales en una estructura espacio-temporal agregada por barrio e intervalo, calculando métricas de uso pagado mediante solapes temporales exactos. Esta transformación es necesaria porque la dificultad real de aparcar en superficie no se observa directamente: el TFM construye un proxy histórico basado en señal pagada SER, capacidad espacial y restricciones temporales observables.

SER es el núcleo principal de este notebook. EMT/off-street y las fuentes contextuales externas no se incorporan aquí. El calendario laboral limpio se utiliza únicamente para delimitar las ventanas SER observables y evitar construir intervalos fuera del régimen de servicio.


## 0. Configuración, raíz del repositorio y banderas

La raíz del repositorio se detecta con `data_catalog.csv`. Las banderas se fijan de forma conservadora: la construcción completa del panel queda bloqueada mientras `EXECUTE_HEAVY_STEPS=False`, y cualquier escritura de salidas queda bloqueada mientras `WRITE_OUTPUTS=False`.


In [28]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

EXECUTE_HEAVY_STEPS = False
WRITE_OUTPUTS = False
OVERWRITE_OUTPUTS = False
SMOKE_TEST_N_PARTS = 2
GRANULARITIES_MIN = [15, 30, 45, 60]

In [29]:
def find_repo_root(start: Path | None = None) -> Path:
    """Busca hacia arriba la raíz del repositorio usando data_catalog.csv."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado data_catalog.csv al ascender desde el directorio actual.")


ROOT = find_repo_root()


def relpath(path: Path | str) -> str:
    """Devuelve una ruta relativa a ROOT cuando es posible."""
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


def safe_parquet_metadata(path: Path | str) -> dict:
    """Obtiene esquema y metadatos Parquet sin cargar datasets completos."""
    path = Path(path)
    result = {
        "path": relpath(path),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
        "n_files": pd.NA,
        "n_rows_metadata": pd.NA,
        "n_columns_metadata": pd.NA,
        "columns": [],
        "read_error": None,
    }
    if not path.exists():
        result["read_error"] = "No existe"
        return result

    try:
        if path.is_dir():
            parquet_files = sorted(path.glob("**/*.parquet"))
            result["n_files"] = len(parquet_files)
            dataset = ds.dataset(path, format="parquet", partitioning="hive")
            columns = list(dataset.schema.names)
            result["columns"] = columns
            result["n_columns_metadata"] = len(columns)
            if parquet_files:
                sample_meta = pq.ParquetFile(parquet_files[0]).metadata
                result["sample_file"] = relpath(parquet_files[0])
                result["sample_rows_metadata"] = sample_meta.num_rows
        else:
            meta = pq.ParquetFile(path).metadata
            columns = meta.schema.names
            result["n_files"] = 1
            result["n_rows_metadata"] = meta.num_rows
            result["n_columns_metadata"] = len(columns)
            result["columns"] = columns
            result["size_mb"] = round(path.stat().st_size / (1024**2), 3)
    except Exception as exc:
        result["read_error"] = f"{type(exc).__name__}: {exc}"

    return result


def require_columns(columns, required, dataset_id: str) -> pd.DataFrame:
    """Comprueba columnas mínimas y devuelve una tabla de control."""
    available = set(columns)
    return pd.DataFrame([
        {
            "dataset": dataset_id,
            "columna": column,
            "presente": column in available,
        }
        for column in required
    ])


def normalize_text_key(value):
    """Normaliza texto para futuras claves de comparación, sin ejecutar joins pesados."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"\s+", " ", text)
    return text


flags_status = pd.DataFrame([
    {"bandera": "EXECUTE_HEAVY_STEPS", "valor": EXECUTE_HEAVY_STEPS, "efecto": "bloquea la construcción completa del panel"},
    {"bandera": "WRITE_OUTPUTS", "valor": WRITE_OUTPUTS, "efecto": "bloquea cualquier escritura de salidas de producción"},
    {"bandera": "OVERWRITE_OUTPUTS", "valor": OVERWRITE_OUTPUTS, "efecto": "impide sobreescrituras salvo activación explícita"},
    {"bandera": "SMOKE_TEST_N_PARTS", "valor": SMOKE_TEST_N_PARTS, "efecto": "limita futuras pruebas de humo por particiones"},
    {"bandera": "GRANULARITIES_MIN", "valor": GRANULARITIES_MIN, "efecto": "granularidades candidatas que se compararán más adelante"},
])

display(Markdown(f"**Root detectado:** `{ROOT}`"))
display(flags_status)

**Root detectado:** `/Users/hugo/TFM_parking_madrid`

,bandera,valor,efecto
0,EXECUTE_HEAVY_STEPS,False,bloquea la construcción completa del panel
1,WRITE_OUTPUTS,False,bloquea cualquier escritura de salidas de producción
2,OVERWRITE_OUTPUTS,False,impide sobreescrituras salvo activación explícita
3,SMOKE_TEST_N_PARTS,2,limita futuras pruebas de humo por particiones
4,GRANULARITIES_MIN,"[15, 30, 45, 60]",granularidades candidatas que se compararán más adelante


## 1. Rutas de entrada y salidas previstas

La base `ser_tiques_barrio_base` procede de `04_01_ser_joins_base.ipynb` y constituye la entrada nuclear: conserva los tiques a escala barrio y la información temporal necesaria para calcular solapes. La tabla `ser_barrio_capacidad_anio.parquet` aporta el denominador estructural anual por barrio (`plazas_barrio_anio`). El calendario laboral limpio delimita los días y ventanas donde el SER es observable.

Las salidas previstas del notebook son el panel global candidato para las granularidades comparadas y el panel global final a 30 minutos. No se construyen variantes desagregadas adicionales en este notebook.


In [30]:
TIQUES_BARRIO_BASE_DIR = ROOT / "data/processed/core/ser/ser_tiques_barrio_base"
BARRIO_CAPACITY_PATH = ROOT / "data/processed/core/ser/ser_barrio_capacidad_anio.parquet"
CALENDAR_PATH = ROOT / "data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet"

OUTPUT_GLOBAL_CANDIDATES_DIR = ROOT / "data/processed/core/ser/ser_barrio_intervalo_global_candidates"
OUTPUT_GLOBAL_FINAL_PATH = ROOT / "data/processed/core/ser/ser_barrio_intervalo_global_final.parquet"

input_sources = pd.DataFrame([
    {"dataset": "ser_tiques_barrio_base", "path": relpath(TIQUES_BARRIO_BASE_DIR), "existe": TIQUES_BARRIO_BASE_DIR.exists(), "critico": True, "rol": "entrada crítica", "uso_en_04_02": "eventos SER depurados a escala barrio; unidad de cálculo inicial"},
    {"dataset": "ser_barrio_capacidad_anio", "path": relpath(BARRIO_CAPACITY_PATH), "existe": BARRIO_CAPACITY_PATH.exists(), "critico": True, "rol": "entrada crítica", "uso_en_04_02": "denominador anual global para métricas normalizadas"},
    {"dataset": "contexto_calendario_laboral_clean", "path": relpath(CALENDAR_PATH), "existe": CALENDAR_PATH.exists(), "critico": True, "rol": "entrada crítica", "uso_en_04_02": "restricción de malla a ventanas SER observables; no es variable exógena"},
])

display(input_sources)

missing_inputs = input_sources.loc[
    (~input_sources["existe"]) & (input_sources["critico"]),
    ["dataset", "path", "critico", "rol", "uso_en_04_02"],
]
if not missing_inputs.empty:
    display(Markdown("**Entradas críticas ausentes detectadas.**"))
    display(missing_inputs)
    raise FileNotFoundError("Faltan entradas críticas para iniciar 04_02. Revisa la tabla `missing_inputs`.")

output_plan = pd.DataFrame([
    {"salida_prevista": "ser_barrio_intervalo_global_candidates", "path": relpath(OUTPUT_GLOBAL_CANDIDATES_DIR), "se_escribe_ahora": False},
    {"salida_prevista": "ser_barrio_intervalo_global_final", "path": relpath(OUTPUT_GLOBAL_FINAL_PATH), "se_escribe_ahora": False},
])
display(output_plan)


,dataset,path,existe,critico,rol,uso_en_04_02
0,ser_tiques_barrio_base,data/processed/core/ser/ser_tiques_barrio_base,True,True,entrada crítica,eventos SER depurados a escala barrio; unidad de cálculo inicial
1,ser_barrio_capacidad_anio,data/processed/core/ser/ser_barrio_capacidad_anio.parquet,True,True,entrada crítica,denominador anual global para métricas normalizadas
2,contexto_calendario_laboral_clean,data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet,True,True,entrada crítica,restricción de malla a ventanas SER observables; no es variable exógena


,salida_prevista,path,se_escribe_ahora
0,ser_barrio_intervalo_global_candidates,data/processed/core/ser/ser_barrio_intervalo_global_candidates,False
1,ser_barrio_intervalo_global_final,data/processed/core/ser/ser_barrio_intervalo_global_final.parquet,False


## 2. Unidad de cálculo, unidad analítica y malla común

La unidad de cálculo será el tique individual, porque solo a ese nivel se conservan `fecha_inicio`, `fecha_fin` y `barrio_key`. Esta información permite calcular minutos exactos de solape entre cada tique y los intervalos temporales candidatos, evitando usar número bruto de tiques como sustituto directo de ocupación.

La unidad analítica final será `barrio_key × intervalo_inicio × granularidad_min`. La elección de barrio como unidad espacial principal responde a una limitación metodológica central: los pagos app/digitales no pueden imputarse de forma verificable a una escala más fina con las fuentes públicas disponibles. Trabajar a escala barrio permite conservar tanto pagos físicos como digitales sin introducir una hipótesis espacial no defendible.

El panel se construirá sobre una malla común barrio-intervalo. Esto evita que solo existan filas cuando hay tiques y permite representar intervalos observables sin señal pagada como filas explícitas. Cuando exista capacidad válida, esos intervalos podrán tener métricas de señal pagada igual a cero, pero esa ausencia de tiques no debe interpretarse como facilidad real para aparcar: los tiques SER observan demanda pagada, no ocupación real de todas las plazas.

Las granularidades de 15, 30, 45 y 60 minutos se mantienen como candidatas. La granularidad final deberá justificarse con diagnósticos de variabilidad, sparsity, estabilidad, coste computacional e interpretabilidad.


## 3. Lectura ligera de `ser_tiques_barrio_base`

No se carga el dataset completo ni se construye el cruce tique × intervalo. La inspección se limita al esquema Parquet y a una muestra de metadatos de ficheros, suficiente para verificar que las columnas mínimas existen antes de plantear la fase pesada.


In [31]:
required_tickets_columns = [
    "fecha_inicio",
    "fecha_fin",
    "barrio_key",
    "anio",
]

tickets_metadata = safe_parquet_metadata(TIQUES_BARRIO_BASE_DIR)
tickets_columns = tickets_metadata["columns"]

tickets_metadata_view = pd.DataFrame([
    {
        "dataset": "ser_tiques_barrio_base",
        "path": tickets_metadata["path"],
        "n_files": tickets_metadata["n_files"],
        "n_columns_metadata": tickets_metadata["n_columns_metadata"],
        "sample_file": tickets_metadata.get("sample_file", pd.NA),
        "sample_rows_metadata": tickets_metadata.get("sample_rows_metadata", pd.NA),
        "read_error": tickets_metadata["read_error"],
    }
])

tickets_column_checks = require_columns(tickets_columns, required_tickets_columns, "ser_tiques_barrio_base")

display(tickets_metadata_view)
display(tickets_column_checks)

if not tickets_column_checks["presente"].all():
    missing = tickets_column_checks.loc[~tickets_column_checks["presente"], "columna"].tolist()
    raise ValueError(f"Faltan columnas mínimas en ser_tiques_barrio_base: {missing}")

,dataset,path,n_files,n_columns_metadata,sample_file,sample_rows_metadata,read_error
0,ser_tiques_barrio_base,data/processed/core/ser/ser_tiques_barrio_base,1002,20,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2023_q1/part_000000__raw_2023_q1.parquet,236923,None


,dataset,columna,presente
0,ser_tiques_barrio_base,fecha_inicio,True
1,ser_tiques_barrio_base,fecha_fin,True
2,ser_tiques_barrio_base,barrio_key,True
3,ser_tiques_barrio_base,anio,True


## 4. Lectura ligera de capacidad anual por barrio

La capacidad anual por barrio aporta el denominador estructural del target global (`plazas_barrio_anio`). Si falta denominador o el denominador no es positivo, las métricas normalizadas no deben calcularse como cero.


In [32]:
required_capacity_columns = [
    "anio",
    "barrio_key",
    "plazas_barrio_anio",
]

capacity_metadata = safe_parquet_metadata(BARRIO_CAPACITY_PATH)
capacity_column_checks = require_columns(capacity_metadata["columns"], required_capacity_columns, "ser_barrio_capacidad_anio")

display(pd.DataFrame([
    {
        "dataset": "ser_barrio_capacidad_anio",
        "path": capacity_metadata["path"],
        "n_rows_metadata": capacity_metadata["n_rows_metadata"],
        "n_columns_metadata": capacity_metadata["n_columns_metadata"],
        "size_mb": capacity_metadata.get("size_mb", pd.NA),
        "read_error": capacity_metadata["read_error"],
    }
]))
display(capacity_column_checks)

if not capacity_column_checks["presente"].all():
    missing = capacity_column_checks.loc[~capacity_column_checks["presente"], "columna"].tolist()
    raise ValueError(f"Faltan columnas mínimas en ser_barrio_capacidad_anio: {missing}")

MAX_LIGHT_READ_MB = 100
capacity_df = None
capacity_read_note = None

if capacity_metadata.get("size_mb", np.inf) <= MAX_LIGHT_READ_MB:
    try:
        capacity_df = pd.read_parquet(BARRIO_CAPACITY_PATH)
        capacity_summary = pd.DataFrame([
            {
                "filas": len(capacity_df),
                "columnas": capacity_df.shape[1],
                "anio_min": capacity_df["anio"].min(),
                "anio_max": capacity_df["anio"].max(),
                "n_anios": capacity_df["anio"].nunique(dropna=True),
                "n_barrios": capacity_df["barrio_key"].nunique(dropna=True),
                "n_barrios_anio": capacity_df[["anio", "barrio_key"]].drop_duplicates().shape[0],
                "motor_resumen": "pandas/pyarrow",
            }
        ])
        capacity_years = pd.DataFrame({"anios_disponibles": sorted(capacity_df["anio"].dropna().unique().tolist())})
    except Exception as exc:
        capacity_read_note = f"Lectura pandas/pyarrow no disponible para resumen completo: {type(exc).__name__}: {exc}"
        try:
            import duckdb

            parquet_sql_path = str(BARRIO_CAPACITY_PATH).replace("'", "''")
            capacity_summary = duckdb.sql(f"""
                select
                    count(*) as filas,
                    {capacity_metadata['n_columns_metadata']} as columnas,
                    min(anio) as anio_min,
                    max(anio) as anio_max,
                    count(distinct anio) as n_anios,
                    count(distinct barrio_key) as n_barrios,
                    count(distinct cast(anio as varchar) || '|' || barrio_key) as n_barrios_anio,
                    'duckdb/parquet_scan' as motor_resumen
                from '{parquet_sql_path}'
            """).fetchdf()
            capacity_years = duckdb.sql(f"""
                select distinct anio as anios_disponibles
                from '{parquet_sql_path}'
                where anio is not null
                order by anio
            """).fetchdf()
        except Exception as fallback_exc:
            display(Markdown(
                "No se ha podido leer la tabla de capacidad con pandas/pyarrow ni con DuckDB. "
                "Se conservan los checks de esquema mediante metadatos Parquet."
            ))
            capacity_summary = pd.DataFrame([
                {
                    "filas": capacity_metadata["n_rows_metadata"],
                    "columnas": capacity_metadata["n_columns_metadata"],
                    "anio_min": pd.NA,
                    "anio_max": pd.NA,
                    "n_anios": pd.NA,
                    "n_barrios": pd.NA,
                    "n_barrios_anio": pd.NA,
                    "motor_resumen": "solo metadatos",
                    "read_error": f"{capacity_read_note}; fallback DuckDB: {type(fallback_exc).__name__}: {fallback_exc}",
                }
            ])
            capacity_years = pd.DataFrame({"anios_disponibles": []})

    if capacity_read_note:
        display(Markdown(f"**Advertencia de lectura:** {capacity_read_note}"))
    display(capacity_summary)
    display(capacity_years)
else:
    display(Markdown(f"La tabla de capacidad supera {MAX_LIGHT_READ_MB} MB; se evita la lectura completa en esta validación ligera."))

,dataset,path,n_rows_metadata,n_columns_metadata,size_mb,read_error
0,ser_barrio_capacidad_anio,data/processed/core/ser/ser_barrio_capacidad_anio.parquet,253,13,0.014,None


,dataset,columna,presente
0,ser_barrio_capacidad_anio,anio,True
1,ser_barrio_capacidad_anio,barrio_key,True
2,ser_barrio_capacidad_anio,plazas_barrio_anio,True


,filas,columnas,anio_min,anio_max,n_anios,n_barrios,n_barrios_anio,motor_resumen
0,253,13,2023,2026,4,65,253,pandas/pyarrow


,anios_disponibles
0,2023
1,2024
2,2025
3,2026


## 5. Régimen SER observable heredado de `04_01`

El panel solo debe generar intervalos dentro de ventanas SER observables. Esta regla mantiene la coherencia con `04_01_ser_joins_base.ipynb`, donde los tiques ya fueron filtrados según calendario y régimen horario. En `04_02` no se reabre esa limpieza, pero sí se reutiliza su lógica para construir la malla temporal.

La tabla siguiente resume la regla operativa aplicada: días laborables de lunes a viernes, sábados no festivos, régimen especial de agosto, 24 y 31 de diciembre, y exclusión de domingos y festivos. Esta tabla no mide ocupación ni dificultad; define únicamente cuándo la señal SER es comparable. No se crearán intervalos nocturnos, dominicales, festivos o fuera de servicio para absorber duración restante de tiques.

In [6]:
regimen_ser_observable = pd.DataFrame([
    {"caso": "lunes-viernes no festivos", "ventana": "09:00-21:00", "entra_en_base": True},
    {"caso": "sabados no festivos", "ventana": "09:00-15:00", "entra_en_base": True},
    {"caso": "agosto lunes-sabado no festivo", "ventana": "09:00-15:00", "entra_en_base": True},
    {"caso": "24 y 31 de diciembre", "ventana": "09:00-15:00", "entra_en_base": True},
    {"caso": "domingos y festivos", "ventana": "sin servicio", "entra_en_base": False},
])

display(regimen_ser_observable)

,caso,ventana,entra_en_base
0,lunes-viernes no festivos,09:00-21:00,True
1,sabados no festivos,09:00-15:00,True
2,agosto lunes-sabado no festivo,09:00-15:00,True
3,24 y 31 de diciembre,09:00-15:00,True
4,domingos y festivos,sin servicio,False


## 6. Control de ejecución pesada y escritura

La construcción completa del panel requiere cruzar temporalmente tiques con los intervalos que solapan y agregar de inmediato. Esta operación puede multiplicar el volumen intermedio de datos, por lo que queda protegida mediante banderas explícitas. La tabla expandida tique × intervalo no se guardará como salida final completa.


In [8]:
def build_panel_candidates_guard():
    if not EXECUTE_HEAVY_STEPS:
        raise RuntimeError(
            "Construcción completa bloqueada: EXECUTE_HEAVY_STEPS=False. "
            "La construcción completa del panel debe activarse explícitamente."
        )


def write_outputs_guard():
    if not WRITE_OUTPUTS:
        raise RuntimeError(
            "Escritura de outputs bloqueada: WRITE_OUTPUTS=False. "
            "No se escriben salidas de producción mientras `WRITE_OUTPUTS=False`."
        )
    if not OVERWRITE_OUTPUTS:
        display(Markdown("`OVERWRITE_OUTPUTS=False`: cualquier salida existente requerirá revisión explícita antes de sobrescribir."))


display(Markdown("Construcción pesada bloqueada correctamente porque `EXECUTE_HEAVY_STEPS=False`."))
display(Markdown("Escritura de outputs bloqueada correctamente porque `WRITE_OUTPUTS=False`."))

Construcción pesada bloqueada correctamente porque `EXECUTE_HEAVY_STEPS=False`.

Escritura de outputs bloqueada correctamente porque `WRITE_OUTPUTS=False`.

## 7. Calendario y malla temporal observable

La construcción de `SER_barrio_intervalo` requiere una malla temporal común que represente todos los intervalos SER observables por barrio, no solo aquellos donde existen tiques. Esta malla permite distinguir entre ausencia de señal pagada y ausencia de fila, evitando que el panel final quede condicionado por la existencia previa de registros.

El calendario laboral limpio se utiliza únicamente para delimitar qué fechas son comparables dentro del régimen SER. No se incorpora como variable explicativa en este notebook. Su función es restringir la generación de intervalos a ventanas donde el estacionamiento regulado es observable: días laborables ordinarios, sábados no festivos, régimen especial de agosto y jornadas reducidas de 24 y 31 de diciembre, de acuerdo con la lógica ya aplicada en `04_01_ser_joins_base.ipynb`.

En esta sección no se cruzan todavía tiques con intervalos. Primero se valida el calendario, se construye una función de generación de ventanas SER observables y se prueba sobre una muestra reducida. La construcción completa de la malla para todos los barrios, años y granularidades queda protegida por las banderas de ejecución pesada.


In [9]:
calendar_read_note = None
try:
    calendar_raw = pd.read_parquet(CALENDAR_PATH)
    calendar_read_engine = "pandas/pyarrow"
except Exception as exc:
    calendar_read_note = f"Lectura pandas/pyarrow no disponible: {type(exc).__name__}: {exc}"
    try:
        import duckdb

        calendar_sql_path = str(CALENDAR_PATH).replace("'", "''")
        calendar_raw = duckdb.sql(f"select * from '{calendar_sql_path}'").fetchdf()
        calendar_read_engine = "duckdb/parquet_scan"
    except Exception as fallback_exc:
        raise RuntimeError(
            "No se ha podido leer CALENDAR_PATH con pd.read_parquet ni con fallback DuckDB."
        ) from fallback_exc

if calendar_read_note:
    display(Markdown(f"**Advertencia de lectura:** {calendar_read_note}"))

calendar_columns = calendar_raw.columns.tolist()
display(Markdown(f"Calendario leído con `{calendar_read_engine}`."))
fecha_candidates = ["fecha", "date", "dia", "fecha_dia"]
fecha_col = next((col for col in fecha_candidates if col in calendar_raw.columns), None)
if fecha_col is None:
    raise ValueError(f"No se ha encontrado columna de fecha. Candidatas esperadas: {fecha_candidates}")

calendar_raw = calendar_raw.copy()
calendar_raw[fecha_col] = pd.to_datetime(calendar_raw[fecha_col])

holiday_direct_candidates = ["es_festivo", "festivo", "is_holiday"]
laborable_candidates = ["laborable", "es_laborable", "is_workday"]
holiday_col = next((col for col in holiday_direct_candidates if col in calendar_raw.columns), None)
laborable_col = next((col for col in laborable_candidates if col in calendar_raw.columns), None)

if holiday_col is not None:
    festivo_source = holiday_col
elif laborable_col is not None:
    festivo_source = laborable_col
else:
    festivo_source = None

def coerce_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int).astype(bool)
    normalized = series.map(normalize_text_key)
    return normalized.isin(["TRUE", "T", "1", "SI", "S", "YES", "Y"])

if festivo_source is None:
    display(Markdown(
        "**Advertencia:** no se ha encontrado una columna clara de festivo/laborable; "
        "se detiene la generación de ventanas SER observables."
    ))
    raise ValueError("No se puede generar `calendar_ser_windows` sin columna de festivo o laborable.")

if holiday_col is not None:
    calendar_raw["es_festivo_detectado"] = coerce_bool_series(calendar_raw[holiday_col])
else:
    calendar_raw["es_festivo_detectado"] = ~coerce_bool_series(calendar_raw[laborable_col])

aux_calendar_candidates = [
    "anio",
    "mes",
    "dia_semana_num",
    "dia_semana_nombre",
    "es_sabado",
    "es_domingo",
    "tipo_regimen_ser_dia_base",
]
aux_calendar_columns = [col for col in aux_calendar_candidates if col in calendar_raw.columns]
calendar_required_column_checks = pd.DataFrame([
    {
        "elemento": "fecha",
        "columna_detectada": fecha_col,
        "presente": fecha_col is not None,
        "uso": "ordenar fechas y derivar año, mes y día de semana",
    },
    {
        "elemento": "festivo/laborable",
        "columna_detectada": festivo_source,
        "presente": festivo_source is not None,
        "uso": "excluir festivos de las ventanas SER observables",
    },
    {
        "elemento": "columnas auxiliares calendario",
        "columna_detectada": ", ".join(aux_calendar_columns) if aux_calendar_columns else pd.NA,
        "presente": bool(aux_calendar_columns),
        "uso": "contraste informativo; la clasificación operativa se deriva de fecha y festivo/laborable",
    },
])
display(calendar_required_column_checks)

calendar_summary = pd.DataFrame([
    {
        "filas": len(calendar_raw),
        "columnas": calendar_raw.shape[1],
        "columna_fecha": fecha_col,
        "fecha_min": calendar_raw[fecha_col].min(),
        "fecha_max": calendar_raw[fecha_col].max(),
        "n_anios": calendar_raw[fecha_col].dt.year.nunique(dropna=True),
        "columna_festivo_laborable": festivo_source,
        "interpretacion_festivo": "directa" if holiday_col is not None else "inversa_laborable",
    }
])

display(calendar_summary)

Calendario leído con `pandas/pyarrow`.

,elemento,columna_detectada,presente,uso
0,fecha,fecha,True,"ordenar fechas y derivar año, mes y día de semana"
1,festivo/laborable,es_festivo,True,excluir festivos de las ventanas SER observables
2,columnas auxiliares calendario,"anio, mes, dia_semana_num, dia_semana_nombre, es_sabado, es_domingo, tipo_regimen_ser_dia_base",True,contraste informativo; la clasificación operativa se deriva de fecha y festivo/laborable


,filas,columnas,columna_fecha,fecha_min,fecha_max,n_anios,columna_festivo_laborable,interpretacion_festivo
0,1461,18,fecha,2023-01-01,2026-12-31,4,es_festivo,directa


In [10]:
def classify_ser_window(row) -> pd.Series:
    fecha = pd.Timestamp(row["fecha"])
    es_festivo = bool(row["es_festivo"])
    dia_semana = int(row["dia_semana"])

    if es_festivo or dia_semana == 6:
        return pd.Series({
            "ser_observable": False,
            "hora_inicio_ser": pd.NA,
            "hora_fin_ser": pd.NA,
            "regimen_ser": "domingo_o_festivo_sin_servicio",
        })
    if fecha.month == 8 and dia_semana <= 5:
        return pd.Series({
            "ser_observable": True,
            "hora_inicio_ser": "09:00",
            "hora_fin_ser": "15:00",
            "regimen_ser": "agosto_lunes_sabado_no_festivo",
        })
    if fecha.month == 12 and fecha.day in [24, 31]:
        return pd.Series({
            "ser_observable": True,
            "hora_inicio_ser": "09:00",
            "hora_fin_ser": "15:00",
            "regimen_ser": "24_31_diciembre_no_festivo",
        })
    if dia_semana == 5:
        return pd.Series({
            "ser_observable": True,
            "hora_inicio_ser": "09:00",
            "hora_fin_ser": "15:00",
            "regimen_ser": "sabado_no_festivo",
        })
    return pd.Series({
        "ser_observable": True,
        "hora_inicio_ser": "09:00",
        "hora_fin_ser": "21:00",
        "regimen_ser": "lunes_viernes_no_festivo",
    })

calendar_base = pd.DataFrame({
    "fecha": calendar_raw[fecha_col].dt.normalize(),
    "anio": calendar_raw[fecha_col].dt.year,
    "mes": calendar_raw[fecha_col].dt.month,
    "dia_semana": calendar_raw[fecha_col].dt.dayofweek,
    "es_festivo": calendar_raw["es_festivo_detectado"].astype(bool),
})

calendar_ser_windows = pd.concat(
    [calendar_base, calendar_base.apply(classify_ser_window, axis=1)],
    axis=1,
).sort_values("fecha").reset_index(drop=True)

regimen_counts = (
    calendar_ser_windows["regimen_ser"]
    .value_counts(dropna=False)
    .rename_axis("regimen_ser")
    .reset_index(name="n_fechas")
)
observable_by_year = (
    calendar_ser_windows
    .groupby(["anio", "ser_observable"], dropna=False)
    .size()
    .reset_index(name="n_fechas")
)
calendar_samples = pd.concat([
    calendar_ser_windows.loc[calendar_ser_windows["ser_observable"]].head(5),
    calendar_ser_windows.loc[~calendar_ser_windows["ser_observable"]].head(5),
], ignore_index=True)

display(regimen_counts)

,regimen_ser,n_fechas
0,lunes_viernes_no_festivo,905
1,domingo_o_festivo_sin_servicio,265
2,sabado_no_festivo,183
3,agosto_lunes_sabado_no_festivo,102
4,24_31_diciembre_no_festivo,6


In [11]:
def generate_intervals_for_day(fecha, hora_inicio, hora_fin, granularidad_min: int) -> pd.DataFrame:
    if pd.isna(hora_inicio) or pd.isna(hora_fin):
        return pd.DataFrame(columns=["intervalo_inicio", "intervalo_fin", "granularidad_min"])

    fecha = pd.Timestamp(fecha).normalize()
    start = pd.Timestamp(f"{fecha.date()} {hora_inicio}")
    end = pd.Timestamp(f"{fecha.date()} {hora_fin}")
    if end <= start:
        raise ValueError(f"Ventana SER inválida para {fecha.date()}: {hora_inicio}-{hora_fin}")

    starts = pd.date_range(start=start, end=end, freq=f"{granularidad_min}min", inclusive="left")
    intervals = pd.DataFrame({
        "intervalo_inicio": starts,
        "intervalo_fin": starts + pd.to_timedelta(granularidad_min, unit="min"),
        "granularidad_min": granularidad_min,
    })
    return intervals.loc[intervals["intervalo_fin"] <= end].reset_index(drop=True)


def first_matching_window(mask: pd.Series, label: str) -> pd.DataFrame:
    rows = calendar_ser_windows.loc[mask].head(1).copy()
    if rows.empty:
        display(Markdown(f"No hay fecha disponible para la muestra `{label}`."))
        return rows
    rows["tipo_muestra"] = label
    return rows

ordinary_mask = (
    calendar_ser_windows["ser_observable"]
    & (calendar_ser_windows["regimen_ser"] == "lunes_viernes_no_festivo")
    & (calendar_ser_windows["mes"] != 8)
    & ~((calendar_ser_windows["mes"] == 12) & (calendar_ser_windows["fecha"].dt.day.isin([24, 31])))
)
saturday_mask = calendar_ser_windows["regimen_ser"].eq("sabado_no_festivo")
august_mask = calendar_ser_windows["regimen_ser"].eq("agosto_lunes_sabado_no_festivo")
december_special_mask = calendar_ser_windows["regimen_ser"].eq("24_31_diciembre_no_festivo")

sample_dates = pd.concat([
    first_matching_window(ordinary_mask, "laborable_ordinario"),
    first_matching_window(saturday_mask, "sabado"),
    first_matching_window(august_mask, "agosto"),
    first_matching_window(december_special_mask, "24_31_diciembre"),
], ignore_index=True)

sample_grid_parts = []
for _, row in sample_dates.iterrows():
    for granularidad_min in GRANULARITIES_MIN:
        day_intervals = generate_intervals_for_day(
            row["fecha"],
            row["hora_inicio_ser"],
            row["hora_fin_ser"],
            granularidad_min,
        )
        if day_intervals.empty:
            continue
        day_intervals.insert(0, "tipo_muestra", row["tipo_muestra"])
        day_intervals.insert(1, "fecha", row["fecha"])
        day_intervals.insert(2, "regimen_ser", row["regimen_ser"])
        day_intervals["ventana_inicio_ser"] = pd.Timestamp(f"{pd.Timestamp(row['fecha']).date()} {row['hora_inicio_ser']}")
        day_intervals["ventana_fin_ser"] = pd.Timestamp(f"{pd.Timestamp(row['fecha']).date()} {row['hora_fin_ser']}")
        sample_grid_parts.append(day_intervals)

sample_time_grid = pd.concat(sample_grid_parts, ignore_index=True) if sample_grid_parts else pd.DataFrame()

interval_counts = (
    sample_time_grid
    .groupby(["tipo_muestra", "fecha", "regimen_ser", "granularidad_min"], dropna=False)
    .size()
    .reset_index(name="n_intervalos")
)

if sample_time_grid.empty:
    sample_interval_bounds = pd.DataFrame(columns=[
        "tipo_muestra",
        "fecha",
        "regimen_ser",
        "granularidad_min",
        "primer_intervalo_inicio",
        "primer_intervalo_fin",
        "ultimo_intervalo_inicio",
        "ultimo_intervalo_fin",
        "n_intervalos",
    ])
    interval_checks = pd.DataFrame([{
        "check": "sample_time_grid_no_vacia",
        "ok": False,
        "detalle": "No se han generado intervalos para las fechas de muestra.",
    }])
else:
    sample_interval_bounds = (
        sample_time_grid
        .groupby(["tipo_muestra", "fecha", "regimen_ser", "granularidad_min"], dropna=False)
        .agg(
            primer_intervalo_inicio=("intervalo_inicio", "min"),
            primer_intervalo_fin=("intervalo_fin", "first"),
            ultimo_intervalo_inicio=("intervalo_inicio", "last"),
            ultimo_intervalo_fin=("intervalo_fin", "max"),
            n_intervalos=("intervalo_inicio", "size"),
        )
        .reset_index()
    )
    interval_checks = pd.DataFrame([
        {
            "check": "intervalos_semiabiertos_inicio_menor_fin",
            "ok": bool((sample_time_grid["intervalo_inicio"] < sample_time_grid["intervalo_fin"]).all()),
            "detalle": "Cada intervalo cumple inicio < fin.",
        },
        {
            "check": "intervalos_dentro_de_ventana_ser",
            "ok": bool(
                (
                    (sample_time_grid["intervalo_inicio"] >= sample_time_grid["ventana_inicio_ser"])
                    & (sample_time_grid["intervalo_fin"] <= sample_time_grid["ventana_fin_ser"])
                ).all()
            ),
            "detalle": "Ningún intervalo excede la ventana SER de su fecha.",
        },
        {
            "check": "duracion_igual_granularidad",
            "ok": bool(
                (
                    (sample_time_grid["intervalo_fin"] - sample_time_grid["intervalo_inicio"]).dt.total_seconds().div(60)
                    == sample_time_grid["granularidad_min"]
                ).all()
            ),
            "detalle": "La duración de cada intervalo coincide con la granularidad declarada.",
        },
    ])

display(sample_dates[["tipo_muestra", "fecha", "regimen_ser", "hora_inicio_ser", "hora_fin_ser"]])
display(sample_interval_bounds)
display(interval_checks)

panel_horizon_rule = pd.DataFrame([
    {
        "criterio": "horizonte efectivo del panel",
        "regla": "intersección entre calendario limpio, cobertura real de tiques y capacidad disponible",
        "estado": "pendiente de calcular antes de construir la malla completa",
        "motivo": "evitar intervalos fuera de la cobertura real de señal SER",
    }
])
display(panel_horizon_rule)

if not interval_checks["ok"].all():
    raise ValueError("Alguna comprobación de la malla temporal de muestra ha fallado.")

,tipo_muestra,fecha,regimen_ser,hora_inicio_ser,hora_fin_ser
0,laborable_ordinario,2023-01-02,lunes_viernes_no_festivo,09:00,21:00
1,sabado,2023-01-07,sabado_no_festivo,09:00,15:00
2,agosto,2023-08-01,agosto_lunes_sabado_no_festivo,09:00,15:00
3,24_31_diciembre,2024-12-24,24_31_diciembre_no_festivo,09:00,15:00


,tipo_muestra,fecha,regimen_ser,granularidad_min,primer_intervalo_inicio,primer_intervalo_fin,ultimo_intervalo_inicio,ultimo_intervalo_fin,n_intervalos
0,24_31_diciembre,2024-12-24,24_31_diciembre_no_festivo,15,2024-12-24 09:00:00,2024-12-24 09:15:00,2024-12-24 14:45:00,2024-12-24 15:00:00,24
1,24_31_diciembre,2024-12-24,24_31_diciembre_no_festivo,30,2024-12-24 09:00:00,2024-12-24 09:30:00,2024-12-24 14:30:00,2024-12-24 15:00:00,12
2,24_31_diciembre,2024-12-24,24_31_diciembre_no_festivo,45,2024-12-24 09:00:00,2024-12-24 09:45:00,2024-12-24 14:15:00,2024-12-24 15:00:00,8
3,24_31_diciembre,2024-12-24,24_31_diciembre_no_festivo,60,2024-12-24 09:00:00,2024-12-24 10:00:00,2024-12-24 14:00:00,2024-12-24 15:00:00,6
4,agosto,2023-08-01,agosto_lunes_sabado_no_festivo,15,2023-08-01 09:00:00,2023-08-01 09:15:00,2023-08-01 14:45:00,2023-08-01 15:00:00,24
5,agosto,2023-08-01,agosto_lunes_sabado_no_festivo,30,2023-08-01 09:00:00,2023-08-01 09:30:00,2023-08-01 14:30:00,2023-08-01 15:00:00,12
6,agosto,2023-08-01,agosto_lunes_sabado_no_festivo,45,2023-08-01 09:00:00,2023-08-01 09:45:00,2023-08-01 14:15:00,2023-08-01 15:00:00,8
7,agosto,2023-08-01,agosto_lunes_sabado_no_festivo,60,2023-08-01 09:00:00,2023-08-01 10:00:00,2023-08-01 14:00:00,2023-08-01 15:00:00,6
8,laborable_ordinario,2023-01-02,lunes_viernes_no_festivo,15,2023-01-02 09:00:00,2023-01-02 09:15:00,2023-01-02 20:45:00,2023-01-02 21:00:00,48
9,laborable_ordinario,2023-01-02,lunes_viernes_no_festivo,30,2023-01-02 09:00:00,2023-01-02 09:30:00,2023-01-02 20:30:00,2023-01-02 21:00:00,24


,check,ok,detalle
0,intervalos_semiabiertos_inicio_menor_fin,True,Cada intervalo cumple inicio < fin.
1,intervalos_dentro_de_ventana_ser,True,Ningún intervalo excede la ventana SER de su fecha.
2,duracion_igual_granularidad,True,La duración de cada intervalo coincide con la granularidad declarada.


,criterio,regla,estado,motivo
0,horizonte efectivo del panel,"intersección entre calendario limpio, cobertura real de tiques y capacidad disponible",pendiente de calcular antes de construir la malla completa,evitar intervalos fuera de la cobertura real de señal SER


La cobertura completa del calendario no implica que el panel final se genere hasta la última fecha disponible en el calendario. El horizonte efectivo de `SER_barrio_intervalo` se acotará posteriormente mediante la intersección entre calendario limpio, cobertura real de tiques y capacidad disponible. Esta decisión evita generar intervalos sin posibilidad de señal SER observada y mantiene el notebook preparado para futuras ampliaciones de datos.

## 8. Horizonte efectivo del panel

La cobertura del calendario limpio alcanza todo el periodo 2023–2026, pero el panel `SER_barrio_intervalo` no debe generarse automáticamente hasta la última fecha del calendario. La malla temporal debe limitarse al periodo donde exista señal SER potencialmente observable y capacidad estructural disponible.

Por tanto, el horizonte efectivo del panel se definirá como la intersección entre la cobertura del calendario, la cobertura real de los tiques SER disponibles y los años con capacidad barrio-año. Esta regla evita generar intervalos sin posibilidad de señal observada y mantiene el diseño preparado para futuras ampliaciones de datos: si se incorporan nuevos tiques, el horizonte podrá extenderse sin cambiar la metodología.

En esta sección no se construye todavía la malla completa. Solo se calcula y valida el rango temporal que deberá utilizarse posteriormente.

In [12]:
MANUAL_PANEL_START = None
MANUAL_PANEL_END = None

calendar_min_date = pd.Timestamp(calendar_ser_windows["fecha"].min()).normalize()
calendar_max_date = pd.Timestamp(calendar_ser_windows["fecha"].max()).normalize()

if "capacity_df" in globals() and capacity_df is not None:
    capacity_min_year = int(capacity_df["anio"].min())
    capacity_max_year = int(capacity_df["anio"].max())
elif "capacity_summary" in globals() and not capacity_summary.empty:
    if capacity_summary["anio_min"].isna().all() or capacity_summary["anio_max"].isna().all():
        raise ValueError("No se puede derivar el horizonte de capacidad desde `capacity_summary`.")
    capacity_min_year = int(capacity_summary["anio_min"].iloc[0])
    capacity_max_year = int(capacity_summary["anio_max"].iloc[0])
else:
    raise ValueError("No existe `capacity_df` ni `capacity_summary` para derivar el horizonte de capacidad.")

capacity_min_date = pd.Timestamp(year=capacity_min_year, month=1, day=1)
capacity_max_date = pd.Timestamp(year=capacity_max_year, month=12, day=31)


def parquet_column_stats_for_temporal_coverage(dataset_dir: Path, columns: list[str]) -> dict:
    parquet_files = sorted(Path(dataset_dir).glob("**/*.parquet"))
    file_stats = []
    n_files_with_stats = 0
    n_files_without_stats = 0
    read_errors = 0

    for path in parquet_files:
        file_result = {"path": path, "has_all_stats": True}
        try:
            parquet_file = pq.ParquetFile(path)
            schema_names = parquet_file.schema.names
            column_indexes = {name: schema_names.index(name) for name in columns if name in schema_names}
            if set(column_indexes) != set(columns):
                file_result["has_all_stats"] = False
            else:
                for column in columns:
                    values_min = []
                    values_max = []
                    column_index = column_indexes[column]
                    for row_group_idx in range(parquet_file.metadata.num_row_groups):
                        column_chunk = parquet_file.metadata.row_group(row_group_idx).column(column_index)
                        stats = column_chunk.statistics
                        if stats is None or not stats.has_min_max:
                            file_result["has_all_stats"] = False
                            break
                        values_min.append(pd.Timestamp(stats.min))
                        values_max.append(pd.Timestamp(stats.max))
                    if file_result["has_all_stats"]:
                        file_result[f"{column}_min"] = min(values_min)
                        file_result[f"{column}_max"] = max(values_max)
        except Exception as exc:
            read_errors += 1
            file_result["has_all_stats"] = False
            file_result["read_error"] = f"{type(exc).__name__}: {exc}"

        if file_result["has_all_stats"]:
            n_files_with_stats += 1
        else:
            n_files_without_stats += 1
        file_stats.append(file_result)

    stats_df = pd.DataFrame(file_stats)
    coverage_confirmed = bool(parquet_files) and n_files_with_stats == len(parquet_files)
    min_max = {
        "tickets_min_fecha_inicio": pd.NaT,
        "tickets_max_fecha_inicio": pd.NaT,
        "tickets_min_fecha_fin": pd.NaT,
        "tickets_max_fecha_fin": pd.NaT,
    }
    if n_files_with_stats > 0:
        stats_ok = stats_df.loc[stats_df["has_all_stats"]].copy()
        min_max = {
            "tickets_min_fecha_inicio": stats_ok["fecha_inicio_min"].min(),
            "tickets_max_fecha_inicio": stats_ok["fecha_inicio_max"].max(),
            "tickets_min_fecha_fin": stats_ok["fecha_fin_min"].min(),
            "tickets_max_fecha_fin": stats_ok["fecha_fin_max"].max(),
        }

    return {
        "n_files": len(parquet_files),
        "n_files_with_stats": n_files_with_stats,
        "n_files_without_stats": n_files_without_stats,
        "read_errors": read_errors,
        "coverage_confirmed": coverage_confirmed,
        **min_max,
    }


tickets_coverage = parquet_column_stats_for_temporal_coverage(
    TIQUES_BARRIO_BASE_DIR,
    ["fecha_inicio", "fecha_fin"],
)

tickets_coverage_confirmed = tickets_coverage["coverage_confirmed"]
if not tickets_coverage_confirmed:
    display(Markdown(
        "**Advertencia:** la cobertura temporal de tiques no queda confirmada solo con estadísticas Parquet. "
        "Antes de construir la malla completa habrá que calcular la cobertura real mediante una lectura controlada "
        "de columnas temporales."
    ))

manual_start_date = pd.Timestamp(MANUAL_PANEL_START).normalize() if MANUAL_PANEL_START is not None else pd.NaT
manual_end_date = pd.Timestamp(MANUAL_PANEL_END).normalize() if MANUAL_PANEL_END is not None else pd.NaT

panel_horizon_components = pd.DataFrame([
    {
        "componente": "calendario limpio",
        "inicio": calendar_min_date,
        "fin": calendar_max_date,
        "estado": "confirmado",
        "observacion": "derivado de `calendar_ser_windows`",
    },
    {
        "componente": "capacidad barrio-año",
        "inicio": capacity_min_date,
        "fin": capacity_max_date,
        "estado": "confirmado",
        "observacion": f"años disponibles {capacity_min_year}-{capacity_max_year}",
    },
    {
        "componente": "tiques SER",
        "inicio": tickets_coverage["tickets_min_fecha_inicio"],
        "fin": tickets_coverage["tickets_max_fecha_fin"],
        "estado": "confirmado" if tickets_coverage_confirmed else "no confirmado",
        "observacion": (
            f"estadísticas temporales en {tickets_coverage['n_files_with_stats']} de {tickets_coverage['n_files']} ficheros; "
            f"sin estadísticas completas: {tickets_coverage['n_files_without_stats']}"
        ),
    },
    {
        "componente": "corte manual opcional",
        "inicio": manual_start_date,
        "fin": manual_end_date,
        "estado": "no aplicado" if MANUAL_PANEL_START is None and MANUAL_PANEL_END is None else "aplicado",
        "observacion": "solo se aplicará si MANUAL_PANEL_START o MANUAL_PANEL_END dejan de ser None",
    },
])

if tickets_coverage_confirmed:
    start_candidates = [calendar_min_date, capacity_min_date, tickets_coverage["tickets_min_fecha_inicio"]]
    end_candidates = [calendar_max_date, capacity_max_date, tickets_coverage["tickets_max_fecha_fin"]]
    if MANUAL_PANEL_START is not None:
        start_candidates.append(pd.Timestamp(MANUAL_PANEL_START).normalize())
    if MANUAL_PANEL_END is not None:
        end_candidates.append(pd.Timestamp(MANUAL_PANEL_END).normalize())

    panel_start_date = max(start_candidates)
    panel_end_date = min(end_candidates)
    horizonte_confirmado = bool(panel_start_date <= panel_end_date)
    lectura = (
        "El horizonte efectivo queda delimitado por la intersección de calendario, capacidad y cobertura temporal confirmada de tiques."
        if horizonte_confirmado
        else "La intersección de coberturas no produce un intervalo temporal válido."
    )
else:
    panel_start_date = pd.NaT
    panel_end_date = pd.NaT
    horizonte_confirmado = False
    lectura = (
        "La cobertura temporal de tiques no queda confirmada con metadatos Parquet; antes de construir la malla completa "
        "habrá que calcular la cobertura real mediante lectura controlada de `fecha_inicio` y `fecha_fin`."
    )

panel_horizon_decision = pd.DataFrame([
    {
        "panel_start_date": panel_start_date,
        "panel_end_date": panel_end_date,
        "horizonte_confirmado": horizonte_confirmado,
        "criterio": "intersección entre calendario limpio, capacidad disponible, cobertura real de tiques y corte manual opcional",
        "lectura": lectura,
    }
])

display(panel_horizon_components)
display(panel_horizon_decision)

,componente,inicio,fin,estado,observacion
0,calendario limpio,2023-01-01 00:00:00,2026-12-31 00:00:00,confirmado,derivado de `calendar_ser_windows`
1,capacidad barrio-año,2023-01-01 00:00:00,2026-12-31 00:00:00,confirmado,años disponibles 2023-2026
2,tiques SER,2023-01-02 09:00:00,2026-03-31 21:00:00,confirmado,estadísticas temporales en 1002 de 1002 ficheros; sin estadísticas completas: 0
3,corte manual opcional,NaT,NaT,no aplicado,solo se aplicará si MANUAL_PANEL_START o MANUAL_PANEL_END dejan de ser None


,panel_start_date,panel_end_date,horizonte_confirmado,criterio,lectura
0,2023-01-02 09:00:00,2026-03-31 21:00:00,True,"intersección entre calendario limpio, capacidad disponible, cobertura real de tiques y corte manual opcional","El horizonte efectivo queda delimitado por la intersección de calendario, capacidad y cobertura temporal confirmada de tiques."


## 9. Dimensión temporal observable y estimación de tamaño del panel

Una vez definido el horizonte efectivo del panel, el siguiente paso es construir la dimensión temporal observable sobre la que se apoyará la malla `SER_barrio_intervalo`. Esta dimensión contiene únicamente intervalos incluidos en ventanas SER observables y acotados por la cobertura real de tiques y capacidad.

En esta sección todavía no se cruza la dimensión temporal con barrios ni con tiques individuales. Primero se estima el volumen potencial de la malla para cada granularidad candidata. Esta comprobación permite anticipar el coste computacional de construir el panel completo y evita activar una expansión pesada sin conocer antes su tamaño aproximado.

La estimación de tamaño se calcula a partir del número de intervalos observables por granularidad y del número de barrios con capacidad disponible. El resultado no es todavía el panel final, sino un diagnóstico previo de viabilidad computacional.

In [13]:
if not horizonte_confirmado:
    raise RuntimeError(
        "No se puede construir la dimensión temporal observable porque `panel_horizon_decision` "
        "no confirma todavía el horizonte efectivo del panel."
    )

panel_start_bound = pd.Timestamp(panel_start_date)
panel_end_bound = pd.Timestamp(panel_end_date)
panel_start_date_norm = panel_start_bound.normalize()
panel_end_date_norm = panel_end_bound.normalize()

calendar_horizon_observable = calendar_ser_windows.loc[
    (calendar_ser_windows["fecha"] >= panel_start_date_norm)
    & (calendar_ser_windows["fecha"] <= panel_end_date_norm)
    & (calendar_ser_windows["ser_observable"])
].copy()

if calendar_horizon_observable.empty:
    raise ValueError("No hay fechas SER observables dentro del horizonte efectivo calculado.")

time_dimension_parts = []
for _, row in calendar_horizon_observable.iterrows():
    for granularidad_min in GRANULARITIES_MIN:
        day_intervals = generate_intervals_for_day(
            row["fecha"],
            row["hora_inicio_ser"],
            row["hora_fin_ser"],
            granularidad_min,
        )
        if day_intervals.empty:
            continue
        day_intervals.insert(0, "fecha", row["fecha"])
        day_intervals.insert(1, "regimen_ser", row["regimen_ser"])
        time_dimension_parts.append(day_intervals)

time_dimension_candidates = (
    pd.concat(time_dimension_parts, ignore_index=True)
    if time_dimension_parts
    else pd.DataFrame(columns=["fecha", "regimen_ser", "intervalo_inicio", "intervalo_fin", "granularidad_min"])
)

time_dimension_candidates = time_dimension_candidates.loc[
    (time_dimension_candidates["intervalo_inicio"] >= panel_start_bound)
    & (time_dimension_candidates["intervalo_fin"] <= panel_end_bound)
].copy()

time_dimension_candidates = time_dimension_candidates[
    ["granularidad_min", "fecha", "regimen_ser", "intervalo_inicio", "intervalo_fin"]
].sort_values(["granularidad_min", "intervalo_inicio"]).reset_index(drop=True)

time_dimension_candidates["anio_intervalo"] = time_dimension_candidates["intervalo_inicio"].dt.year
time_dimension_candidates["mes_intervalo"] = time_dimension_candidates["intervalo_inicio"].dt.month
time_dimension_candidates["dia_semana_intervalo"] = time_dimension_candidates["intervalo_inicio"].dt.dayofweek

time_dimension_summary = (
    time_dimension_candidates
    .groupby("granularidad_min", dropna=False)
    .agg(
        n_intervalos=("intervalo_inicio", "size"),
        fecha_min=("fecha", "min"),
        fecha_max=("fecha", "max"),
        primer_intervalo=("intervalo_inicio", "min"),
        ultimo_intervalo=("intervalo_fin", "max"),
    )
    .reset_index()
)

if "capacity_df" in globals() and capacity_df is not None:
    capacity_barrio_year = capacity_df[["anio", "barrio_key"]].drop_duplicates().copy()
else:
    try:
        import duckdb

        capacity_sql_path = str(BARRIO_CAPACITY_PATH).replace("'", "''")
        capacity_barrio_year = duckdb.sql(f"""
            select distinct anio, barrio_key
            from '{capacity_sql_path}'
            where anio is not null and barrio_key is not null
        """).fetchdf()
    except Exception as exc:
        raise RuntimeError(
            "No se ha podido obtener `anio` y `barrio_key` de la tabla de capacidad para estimar tamaño del panel."
        ) from exc

capacity_barrio_year["anio"] = capacity_barrio_year["anio"].astype(int)
capacity_barrio_summary = pd.DataFrame([
    {
        "n_barrios_total": capacity_barrio_year["barrio_key"].nunique(dropna=True),
        "n_anio_barrio_key": capacity_barrio_year[["anio", "barrio_key"]].drop_duplicates().shape[0],
        "anios_disponibles": sorted(capacity_barrio_year["anio"].dropna().unique().tolist()),
    }
])

barrios_by_year = (
    capacity_barrio_year
    .groupby("anio", dropna=False)["barrio_key"]
    .nunique()
    .rename("n_barrios_anio")
    .reset_index()
    .rename(columns={"anio": "anio_intervalo"})
)
intervals_by_year = (
    time_dimension_candidates
    .groupby(["granularidad_min", "anio_intervalo"], dropna=False)
    .size()
    .reset_index(name="n_intervalos_anio")
)
panel_size_estimate = intervals_by_year.merge(barrios_by_year, on="anio_intervalo", how="left")
panel_size_estimate["n_filas_estimadas"] = (
    panel_size_estimate["n_intervalos_anio"] * panel_size_estimate["n_barrios_anio"]
)
panel_size_estimate_total = (
    panel_size_estimate
    .groupby("granularidad_min", dropna=False)["n_filas_estimadas"]
    .sum()
    .reset_index(name="n_filas_estimadas_total")
)

observable_dates = set(calendar_horizon_observable["fecha"].dt.normalize())
interval_dates = set(time_dimension_candidates["fecha"].dt.normalize())
interval_duration_min = (
    time_dimension_candidates["intervalo_fin"] - time_dimension_candidates["intervalo_inicio"]
).dt.total_seconds().div(60)

time_dimension_checks = pd.DataFrame([
    {
        "check": "intervalos_dentro_del_horizonte",
        "ok": bool(
            (time_dimension_candidates["intervalo_inicio"] >= panel_start_bound).all()
            and (time_dimension_candidates["intervalo_fin"] <= panel_end_bound).all()
        ),
        "detalle": "Los intervalos no exceden el horizonte efectivo por fecha de servicio.",
    },
    {
        "check": "intervalo_inicio_menor_fin",
        "ok": bool((time_dimension_candidates["intervalo_inicio"] < time_dimension_candidates["intervalo_fin"]).all()),
        "detalle": "Todos los intervalos cumplen la convención semiabierta [inicio, fin).",
    },
    {
        "check": "duracion_igual_granularidad",
        "ok": bool((interval_duration_min == time_dimension_candidates["granularidad_min"]).all()),
        "detalle": "La duración de cada intervalo coincide con su granularidad candidata.",
    },
    {
        "check": "solo_fechas_observables",
        "ok": bool(interval_dates.issubset(observable_dates)),
        "detalle": "No se generan intervalos en fechas no observables del calendario SER.",
    },
])

display(time_dimension_summary)
display(panel_size_estimate_total)
display(time_dimension_checks)

if not time_dimension_checks["ok"].all():
    raise ValueError("Alguna comprobación de la dimensión temporal observable ha fallado.")

,granularidad_min,n_intervalos,fecha_min,fecha_max,primer_intervalo,ultimo_intervalo
0,15,41112,2023-01-02,2026-03-31,2023-01-02 09:00:00,2026-03-31 21:00:00
1,30,20556,2023-01-02,2026-03-31,2023-01-02 09:00:00,2026-03-31 21:00:00
2,45,13704,2023-01-02,2026-03-31,2023-01-02 09:00:00,2026-03-31 21:00:00
3,60,10278,2023-01-02,2026-03-31,2023-01-02 09:00:00,2026-03-31 21:00:00


,granularidad_min,n_filas_estimadas_total
0,15,2584296
1,30,1292148
2,45,861432
3,60,646074


,check,ok,detalle
0,intervalos_dentro_del_horizonte,True,Los intervalos no exceden el horizonte efectivo por fecha de servicio.
1,intervalo_inicio_menor_fin,True,"Todos los intervalos cumplen la convención semiabierta [inicio, fin)."
2,duracion_igual_granularidad,True,La duración de cada intervalo coincide con su granularidad candidata.
3,solo_fechas_observables,True,No se generan intervalos en fechas no observables del calendario SER.


### Lectura metodológica de viabilidad

La dimensión temporal observable queda correctamente acotada por el horizonte efectivo del panel: desde `2023-01-02 09:00:00` hasta `2026-03-31 21:00:00`. Dentro de ese rango, la generación de intervalos respeta las ventanas SER observables, no crea registros fuera del horizonte y mantiene la convención semiabierta `[inicio, fin)`.

La estimación de tamaño indica que la malla global barrio-intervalo es viable para las cuatro granularidades candidatas. Incluso en el caso más exigente, 15 minutos, el panel agregado tendría aproximadamente 2,58 millones de filas. Las granularidades de 30, 45 y 60 minutos reducen progresivamente ese volumen hasta alrededor de 1,29 millones, 0,86 millones y 0,65 millones de filas, respectivamente.

Por tanto, el tamaño de la malla final no parece ser el principal cuello de botella computacional. El coste relevante se concentrará en la fase posterior de cruce entre tiques individuales e intervalos solapados, porque esa operación parte de una base de más de 141 millones de tiques. Para controlar ese riesgo, la construcción del panel deberá realizarse por particiones temporales o bloques de datos, agregando inmediatamente los solapes y evitando conservar una tabla expandida tique × intervalo como salida persistente.

Con esta evidencia, se considera metodológicamente razonable mantener las cuatro granularidades candidatas —15, 30, 45 y 60 minutos— para la siguiente fase. La decisión final de granularidad no se toma aquí: deberá basarse posteriormente en variabilidad temporal, sparsity, estabilidad, interpretabilidad y coste real de cómputo.

## 10. Estrategia incremental de ejecución y selección de particiones de prueba

La construcción del panel completo no debe activarse directamente sobre todos los tiques. Aunque la malla barrio-intervalo estimada es viable como tabla agregada, el cálculo de solapes parte de una base individual de gran tamaño y puede generar un volumen intermedio elevado si no se controla.

Por ello, la ejecución se organizará de forma incremental. Primero se identificará una muestra de prueba formada por el fichero Parquet más grande de cada partición temporal `periodo_inicio`. Esta muestra permite comprobar el cálculo con datos reales, cubrir distintos años y trimestres, y someter el procedimiento a un caso exigente sin procesar todavía todo el dataset.

La estrategia de escalado será: primero metadatos y muestra de particiones; después un smoke test con `largest_part_per_period`; posteriormente un periodo completo; y solo al final el conjunto completo de tiques. En todos los casos, el cruce tique-intervalo deberá agregarse inmediatamente y no persistir una tabla expandida completa.

In [14]:
def extract_hive_partition_from_path(path: Path, partition_name: str):
    prefix = f"{partition_name}="
    for part in path.parts:
        if part.startswith(prefix):
            return part.split("=", 1)[1]
    return pd.NA


def parquet_file_has_stats_for_columns(parquet_file: pq.ParquetFile, columns: list[str]) -> bool:
    schema_names = parquet_file.schema.names
    if not set(columns).issubset(schema_names):
        return False
    column_indexes = {column: schema_names.index(column) for column in columns}
    for row_group_idx in range(parquet_file.metadata.num_row_groups):
        row_group = parquet_file.metadata.row_group(row_group_idx)
        for column in columns:
            stats = row_group.column(column_indexes[column]).statistics
            if stats is None or not stats.has_min_max:
                return False
    return True

part_rows = []
for path in sorted(TIQUES_BARRIO_BASE_DIR.glob("**/*.parquet")):
    try:
        parquet_file = pq.ParquetFile(path)
        metadata = parquet_file.metadata
        n_rows_metadata = metadata.num_rows
        n_columns = metadata.num_columns
        has_temporal_stats = parquet_file_has_stats_for_columns(parquet_file, ["fecha_inicio", "fecha_fin"])
        read_error = None
    except Exception as exc:
        n_rows_metadata = pd.NA
        n_columns = pd.NA
        has_temporal_stats = False
        read_error = f"{type(exc).__name__}: {exc}"

    part_rows.append({
        "path": relpath(path),
        "path_obj": path,
        "periodo_inicio": extract_hive_partition_from_path(path, "periodo_inicio"),
        "size_mb": round(path.stat().st_size / (1024**2), 3),
        "n_rows_metadata": n_rows_metadata,
        "n_columns": n_columns,
        "has_temporal_stats": has_temporal_stats,
        "read_error": read_error,
    })

ticket_parts_inventory = pd.DataFrame(part_rows)

if ticket_parts_inventory.empty:
    ticket_parts_summary_by_period = pd.DataFrame(columns=[
        "periodo_inicio", "n_files", "n_rows_metadata_total", "size_mb_total",
        "min_rows_file", "max_rows_file", "mean_rows_file",
    ])
    largest_part_per_period = pd.DataFrame(columns=[
        "periodo_inicio", "path", "n_rows_metadata", "size_mb", "has_temporal_stats",
    ])
else:
    ticket_parts_inventory["n_rows_metadata"] = pd.to_numeric(ticket_parts_inventory["n_rows_metadata"], errors="coerce")
    ticket_parts_summary_by_period = (
        ticket_parts_inventory
        .groupby("periodo_inicio", dropna=False)
        .agg(
            n_files=("path", "size"),
            n_rows_metadata_total=("n_rows_metadata", "sum"),
            size_mb_total=("size_mb", "sum"),
            min_rows_file=("n_rows_metadata", "min"),
            max_rows_file=("n_rows_metadata", "max"),
            mean_rows_file=("n_rows_metadata", "mean"),
        )
        .reset_index()
        .sort_values("periodo_inicio")
    )
    ticket_parts_summary_by_period["size_mb_total"] = ticket_parts_summary_by_period["size_mb_total"].round(3)
    ticket_parts_summary_by_period["mean_rows_file"] = ticket_parts_summary_by_period["mean_rows_file"].round(1)

    largest_part_per_period = (
        ticket_parts_inventory
        .sort_values(["periodo_inicio", "n_rows_metadata", "size_mb", "path"], ascending=[True, False, False, True])
        .groupby("periodo_inicio", dropna=False)
        .head(1)
        .sort_values("periodo_inicio")
        .reset_index(drop=True)
    )

ticket_parts_inventory_summary = pd.DataFrame([
    {
        "n_files_total": len(ticket_parts_inventory),
        "n_periodos": ticket_parts_inventory["periodo_inicio"].nunique(dropna=True) if not ticket_parts_inventory.empty else 0,
        "n_rows_metadata_total": ticket_parts_inventory["n_rows_metadata"].sum() if not ticket_parts_inventory.empty else 0,
        "size_mb_total": round(ticket_parts_inventory["size_mb"].sum(), 3) if not ticket_parts_inventory.empty else 0.0,
        "n_files_with_temporal_stats": int(ticket_parts_inventory["has_temporal_stats"].sum()) if not ticket_parts_inventory.empty else 0,
        "n_files_without_temporal_stats": int((~ticket_parts_inventory["has_temporal_stats"]).sum()) if not ticket_parts_inventory.empty else 0,
    }
])

execution_scale_plan = pd.DataFrame([
    {
        "nivel": 0,
        "scope": "metadatos",
        "descripcion": "validaciones de entradas, calendario, horizonte y estimación de tamaño",
        "objetivo": "dimensionar el problema sin procesar tiques",
        "se_ejecuta_ahora": True,
    },
    {
        "nivel": 1,
        "scope": "largest_part_per_period",
        "descripcion": "una partición física representativa por cada periodo_inicio",
        "objetivo": "próximo smoke test de solapes y agregación incremental",
        "se_ejecuta_ahora": False,
    },
    {
        "nivel": 2,
        "scope": "periodo completo",
        "descripcion": "todos los ficheros de un periodo_inicio seleccionado",
        "objetivo": "validar coste y consistencia antes del dataset completo",
        "se_ejecuta_ahora": False,
    },
    {
        "nivel": 3,
        "scope": "dataset completo",
        "descripcion": "todos los periodos y particiones de ser_tiques_barrio_base",
        "objetivo": "construcción controlada de candidatos de panel",
        "se_ejecuta_ahora": False,
    },
])

n_periods_inventory = ticket_parts_inventory["periodo_inicio"].nunique(dropna=True) if not ticket_parts_inventory.empty else 0
n_periods_selected = largest_part_per_period["periodo_inicio"].nunique(dropna=True) if not largest_part_per_period.empty else 0
partition_selection_checks = pd.DataFrame([
    {
        "check": "existe_al_menos_un_parquet",
        "ok": bool(len(ticket_parts_inventory) > 0),
        "detalle": f"ficheros localizados: {len(ticket_parts_inventory)}",
    },
    {
        "check": "todos_los_ficheros_tienen_periodo_inicio",
        "ok": bool((~ticket_parts_inventory["periodo_inicio"].isna()).all()) if not ticket_parts_inventory.empty else False,
        "detalle": "la partición Hive periodo_inicio se extrae desde la ruta",
    },
    {
        "check": "largest_part_per_period_una_fila_por_periodo",
        "ok": bool(n_periods_inventory == n_periods_selected and len(largest_part_per_period) == n_periods_inventory),
        "detalle": f"periodos inventariados: {n_periods_inventory}; periodos seleccionados: {n_periods_selected}",
    },
    {
        "check": "seleccion_con_filas_positivas",
        "ok": bool((largest_part_per_period["n_rows_metadata"] > 0).all()) if not largest_part_per_period.empty else False,
        "detalle": "todas las particiones seleccionadas tienen n_rows_metadata > 0",
    },
])

display(ticket_parts_inventory_summary)
display(largest_part_per_period[["periodo_inicio", "path", "n_rows_metadata", "size_mb", "has_temporal_stats"]])
display(execution_scale_plan)
display(partition_selection_checks)

if not partition_selection_checks["ok"].all():
    raise ValueError("Alguna comprobación de selección de particiones de prueba ha fallado.")

,n_files_total,n_periodos,n_rows_metadata_total,size_mb_total,n_files_with_temporal_stats,n_files_without_temporal_stats
0,1002,13,141286515,3786.459,1002,0


,periodo_inicio,path,n_rows_metadata,size_mb,has_temporal_stats
0,2023_q1,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2023_q1/part_000004__raw_2023_q1.parquet,237233,6.436,True
1,2023_q2,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2023_q2/part_000148__raw_2023_q2.parquet,239284,4.728,True
2,2023_q3,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2023_q3/part_000196__raw_2023_q3.parquet,235275,6.558,True
3,2023_q4,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2023_q4/part_000304__raw_2023_q4.parquet,237431,6.635,True
4,2024_q1,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2024_q1/part_000346__raw_2024_q1.parquet,237543,6.701,True
5,2024_q2,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2024_q2/part_000516__raw_2024_q2.parquet,235485,6.665,True
6,2024_q3,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2024_q3/part_000542__raw_2024_q3.parquet,235933,6.187,True
7,2024_q4,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2024_q4/part_000609__raw_2024_q4.parquet,235693,6.672,True
8,2025_q1,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2025_q1/part_000687__raw_2025_q1.parquet,235938,6.151,True
9,2025_q2,data/processed/core/ser/ser_tiques_barrio_base/periodo_inicio=2025_q2/part_000781__raw_2025_q2.parquet,234519,6.488,True


,nivel,scope,descripcion,objetivo,se_ejecuta_ahora
0,0,metadatos,"validaciones de entradas, calendario, horizonte y estimación de tamaño",dimensionar el problema sin procesar tiques,True
1,1,largest_part_per_period,una partición física representativa por cada periodo_inicio,próximo smoke test de solapes y agregación incremental,False
2,2,periodo completo,todos los ficheros de un periodo_inicio seleccionado,validar coste y consistencia antes del dataset completo,False
3,3,dataset completo,todos los periodos y particiones de ser_tiques_barrio_base,construcción controlada de candidatos de panel,False


,check,ok,detalle
0,existe_al_menos_un_parquet,True,ficheros localizados: 1002
1,todos_los_ficheros_tienen_periodo_inicio,True,la partición Hive periodo_inicio se extrae desde la ruta
2,largest_part_per_period_una_fila_por_periodo,True,periodos inventariados: 13; periodos seleccionados: 13
3,seleccion_con_filas_positivas,True,todas las particiones seleccionadas tienen n_rows_metadata > 0


### Lectura metodológica de selección incremental

La selección incremental queda validada. El inventario localiza 1002 ficheros Parquet distribuidos en 13 particiones temporales `periodo_inicio`, con estadísticas temporales disponibles para todos los ficheros. La muestra `largest_part_per_period` conserva una partición física por cada periodo y permite iniciar el smoke test con datos reales de todos los trimestres disponibles, sin procesar todavía el conjunto completo de tiques.

Esta selección no se interpreta como muestra estadística representativa del comportamiento SER, sino como muestra técnica exigente para validar el cálculo de solapes, la agregación inmediata y el control de memoria. Si el procedimiento funciona correctamente sobre esta escala, el siguiente paso será ampliar la ejecución a un periodo completo antes de activar el dataset completo.


## 11. Smoke test de solapes tique-intervalo

Una vez definida la estrategia incremental, se ejecuta una primera prueba controlada del cálculo de solapes entre tiques individuales e intervalos SER observables. Esta prueba no construye todavía el panel completo ni escribe salidas de producción: su objetivo es validar la lógica de expansión temporal, cálculo de minutos solapados, clasificación A/B/C/D y agregación inmediata.

El smoke test se ejecuta inicialmente sobre un subconjunto reducido de las particiones seleccionadas en `largest_part_per_period` y con una granularidad conservadora. Esta decisión reduce el riesgo computacional y permite comprobar la corrección del procedimiento antes de ampliarlo a todos los periodos y granularidades candidatas.

La tabla expandida tique × intervalo se utiliza solo como objeto intermedio temporal. El resultado relevante de esta sección es la agregación por barrio, intervalo y granularidad, junto con checks de coherencia temporal y métricas diagnósticas.

### Mecánica del cálculo de solapes

El cálculo parte de tiques individuales definidos por `fecha_inicio`, `fecha_fin` y `barrio_key`. Cada tique puede solapar uno o varios intervalos de la malla temporal SER observable. Por este motivo, el número bruto de tiques no se utiliza como medida directa de ocupación: primero se calcula cuántos minutos reales de cada tique caen dentro de cada intervalo.

Los intervalos se tratan como semiabiertos, es decir, `[inicio, fin)`. Esta convención evita dobles conteos en los bordes: si un tique termina exactamente en el cierre de un intervalo, contribuye a ese intervalo, pero no al siguiente.

Para un intervalo `[a,b)`, cada solape se clasifica en una de cuatro categorías:

| Categoría | Condición                                  | Interpretación                                                                            |
| --------- | ------------------------------------------ | ----------------------------------------------------------------------------------------- |
| A         | `fecha_inicio < a` y `fecha_fin > b`       | El tique ya estaba activo antes del intervalo y continúa activo después.                  |
| B         | `fecha_inicio < a` y `a < fecha_fin <= b`  | El tique ya estaba activo al inicio y finaliza dentro del intervalo o justo en su cierre. |
| C         | `a <= fecha_inicio < b` y `fecha_fin <= b` | El tique empieza y termina dentro del intervalo o justo en su cierre.                     |
| D         | `a <= fecha_inicio < b` y `fecha_fin > b`  | El tique empieza dentro del intervalo y continúa activo después.                          |

Esta descomposición permite validar que la expansión temporal distingue correctamente situaciones de persistencia, entrada y salida dentro de cada intervalo. Las variables `activo_inicio`, `activo_fin`, `entra_intervalo`, `sale_intervalo` y `persistente_completo` se derivan de la misma lógica temporal y se agregan posteriormente por barrio e intervalo.

La tabla expandida tique-intervalo es solo un objeto intermedio de cálculo. No se conserva como salida final porque puede multiplicar el volumen de datos. La salida relevante de esta prueba es la agregación inmediata por `barrio_key`, `intervalo_inicio`, `intervalo_fin`, `anio_intervalo` y `granularidad_min`.

En este apartado todavía no se interpretan métricas proxy de dificultad. La unión con `ser_barrio_capacidad_anio.parquet` se utiliza únicamente como comprobación técnica de que cada fila agregada puede enlazar con un denominador estructural válido. La definición, cálculo sistemático e interpretación de las métricas normalizadas se reserva para una sección posterior.

In [ ]:
SMOKE_OVERLAP_N_PARTS = SMOKE_TEST_N_PARTS
SMOKE_OVERLAP_GRANULARITIES_MIN = [60]

smoke_overlap_parts = largest_part_per_period.head(SMOKE_OVERLAP_N_PARTS).copy()
smoke_required_columns = [
    "fecha_inicio",
    "fecha_fin",
    "barrio_key",
    "anio",
]


def read_ticket_part_columns(path: Path, columns: list[str]) -> pd.DataFrame:
    try:
        return pq.read_table(path, columns=columns).to_pandas()
    except Exception:
        import duckdb

        parquet_sql_path = str(path).replace("'", "''")
        column_sql = ", ".join(columns)
        return duckdb.sql(f"select {column_sql} from '{parquet_sql_path}'").fetchdf()

smoke_ticket_parts = []
for _, part in smoke_overlap_parts.iterrows():
    part_path = part.get("path_obj", pd.NA)
    if pd.isna(part_path):
        part_path = ROOT / part["path"]
    part_df = read_ticket_part_columns(Path(part_path), smoke_required_columns)
    part_df["source_periodo_inicio"] = part["periodo_inicio"]
    smoke_ticket_parts.append(part_df)

smoke_tickets_raw = pd.concat(smoke_ticket_parts, ignore_index=True) if smoke_ticket_parts else pd.DataFrame(columns=smoke_required_columns + ["source_periodo_inicio"])
n_rows_leidas = len(smoke_tickets_raw)

smoke_tickets = smoke_tickets_raw.copy()
smoke_tickets["fecha_inicio"] = pd.to_datetime(smoke_tickets["fecha_inicio"])
smoke_tickets["fecha_fin"] = pd.to_datetime(smoke_tickets["fecha_fin"])
smoke_tickets = smoke_tickets.loc[
    smoke_tickets["fecha_inicio"].notna()
    & smoke_tickets["fecha_fin"].notna()
    & (smoke_tickets["fecha_fin"] > smoke_tickets["fecha_inicio"])
    & smoke_tickets["barrio_key"].notna()
].copy()
smoke_tickets = smoke_tickets.reset_index(drop=True)
n_rows_validas = len(smoke_tickets)

smoke_overlap_input_summary = pd.DataFrame([
    {
        "n_files": len(smoke_overlap_parts),
        "periodos": ", ".join(smoke_overlap_parts["periodo_inicio"].astype(str).tolist()),
        "n_rows_leidas": n_rows_leidas,
        "n_rows_validas": n_rows_validas,
        "granularidades": SMOKE_OVERLAP_GRANULARITIES_MIN,
    }
])


def expand_tickets_to_intervals_smoke(tickets_df: pd.DataFrame, granularidad_min: int) -> pd.DataFrame:
    if tickets_df.empty:
        return pd.DataFrame(columns=[
            "granularidad_min",
            "fecha_inicio",
            "fecha_fin",
            "barrio_key",
            "intervalo_inicio",
            "intervalo_fin",
            "minutos_solapados",
        ])

    freq = pd.Timedelta(minutes=granularidad_min)
    ticket_base = tickets_df[["fecha_inicio", "fecha_fin", "barrio_key"]].copy().reset_index(drop=True)
    ticket_base["ticket_smoke_id"] = np.arange(len(ticket_base), dtype=np.int64)
    ticket_base["primer_intervalo_inicio"] = ticket_base["fecha_inicio"].dt.floor(f"{granularidad_min}min")
    span_min = (ticket_base["fecha_fin"] - ticket_base["primer_intervalo_inicio"]).dt.total_seconds().div(60)
    n_intervals = np.ceil(span_min / granularidad_min).astype("int64")
    n_intervals = np.maximum(n_intervals, 0)

    repeated_idx = np.repeat(ticket_base.index.to_numpy(), n_intervals)
    if len(repeated_idx) == 0:
        return pd.DataFrame(columns=[
            "granularidad_min",
            "fecha_inicio",
            "fecha_fin",
            "barrio_key",
            "intervalo_inicio",
            "intervalo_fin",
            "minutos_solapados",
        ])

    offsets = np.concatenate([np.arange(count, dtype=np.int64) for count in n_intervals if count > 0])
    expanded = ticket_base.loc[repeated_idx, ["fecha_inicio", "fecha_fin", "barrio_key"]].reset_index(drop=True)
    first_starts = ticket_base.loc[repeated_idx, "primer_intervalo_inicio"].reset_index(drop=True)
    expanded["intervalo_inicio"] = first_starts + offsets * freq
    expanded["intervalo_fin"] = expanded["intervalo_inicio"] + freq

    overlap_start = expanded[["fecha_inicio", "intervalo_inicio"]].max(axis=1)
    overlap_end = expanded[["fecha_fin", "intervalo_fin"]].min(axis=1)
    expanded["minutos_solapados"] = (overlap_end - overlap_start).dt.total_seconds().div(60)
    expanded = expanded.loc[expanded["minutos_solapados"] > 0].copy()
    expanded.insert(0, "granularidad_min", granularidad_min)
    return expanded[[
        "granularidad_min",
        "fecha_inicio",
        "fecha_fin",
        "barrio_key",
        "intervalo_inicio",
        "intervalo_fin",
        "minutos_solapados",
    ]]

expanded_parts = []
for granularidad_min in SMOKE_OVERLAP_GRANULARITIES_MIN:
    expanded_granularity = expand_tickets_to_intervals_smoke(smoke_tickets, granularidad_min)
    observable_intervals = time_dimension_candidates.loc[
        time_dimension_candidates["granularidad_min"].eq(granularidad_min),
        ["granularidad_min", "intervalo_inicio", "intervalo_fin", "anio_intervalo"],
    ].drop_duplicates()
    expanded_granularity = expanded_granularity.drop(columns=["intervalo_fin"]).merge(
        observable_intervals,
        on=["granularidad_min", "intervalo_inicio"],
        how="inner",
    )
    expanded_parts.append(expanded_granularity)

smoke_expanded = pd.concat(expanded_parts, ignore_index=True) if expanded_parts else pd.DataFrame()

if smoke_expanded.empty:
    smoke_aggregated = pd.DataFrame()
else:
    cond_A = (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_fin"])
    cond_B = (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_fin"] <= smoke_expanded["intervalo_fin"])
    cond_C = (smoke_expanded["fecha_inicio"] >= smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_fin"]) & (smoke_expanded["fecha_fin"] <= smoke_expanded["intervalo_fin"])
    cond_D = (smoke_expanded["fecha_inicio"] >= smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_fin"]) & (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_fin"])
    category_count = cond_A.astype(int) + cond_B.astype(int) + cond_C.astype(int) + cond_D.astype(int)

    smoke_expanded["categoria_solape"] = np.select([cond_A, cond_B, cond_C, cond_D], ["A", "B", "C", "D"], default=pd.NA)
    smoke_expanded["categoria_count"] = category_count
    smoke_expanded["activo_inicio"] = (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_inicio"])
    smoke_expanded["activo_fin"] = (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_fin"]) & (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_fin"])
    smoke_expanded["entra_intervalo"] = (smoke_expanded["fecha_inicio"] >= smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_fin"])
    smoke_expanded["sale_intervalo"] = (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_fin"] <= smoke_expanded["intervalo_fin"])
    smoke_expanded["persistente_completo"] = (smoke_expanded["fecha_inicio"] < smoke_expanded["intervalo_inicio"]) & (smoke_expanded["fecha_fin"] > smoke_expanded["intervalo_fin"])

    for category in ["A", "B", "C", "D"]:
        smoke_expanded[f"is_{category}"] = smoke_expanded["categoria_solape"].eq(category)

    smoke_aggregated = (
        smoke_expanded
        .groupby(["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"], dropna=False)
        .agg(
            n_tiques_solapados=("minutos_solapados", "size"),
            minutos_solapados=("minutos_solapados", "sum"),
            n_A=("is_A", "sum"),
            n_B=("is_B", "sum"),
            n_C=("is_C", "sum"),
            n_D=("is_D", "sum"),
            n_activo_inicio=("activo_inicio", "sum"),
            n_activo_fin=("activo_fin", "sum"),
            n_entra_intervalo=("entra_intervalo", "sum"),
            n_sale_intervalo=("sale_intervalo", "sum"),
            n_persistente_completo=("persistente_completo", "sum"),
        )
        .reset_index()
    )

if "capacity_df" in globals() and capacity_df is not None:
    capacity_for_smoke = capacity_df[["anio", "barrio_key", "plazas_barrio_anio"]].drop_duplicates().copy()
else:
    import duckdb

    capacity_sql_path = str(BARRIO_CAPACITY_PATH).replace("'", "''")
    capacity_for_smoke = duckdb.sql(f"""
        select distinct anio, barrio_key, plazas_barrio_anio
        from '{capacity_sql_path}'
    """).fetchdf()

if smoke_aggregated.empty:
    smoke_aggregated_with_capacity = smoke_aggregated.copy()
else:
    smoke_aggregated_with_capacity = smoke_aggregated.merge(
        capacity_for_smoke,
        left_on=["anio_intervalo", "barrio_key"],
        right_on=["anio", "barrio_key"],
        how="left",
    )

smoke_overlap_expansion_summary = (
    smoke_expanded
    .groupby("granularidad_min", dropna=False)
    .agg(
        n_rows_expandidas=("minutos_solapados", "size"),
        minutos_solapados_total=("minutos_solapados", "sum"),
        n_barrios=("barrio_key", "nunique"),
        intervalo_min=("intervalo_inicio", "min"),
        intervalo_max=("intervalo_fin", "max"),
    )
    .reset_index()
    if not smoke_expanded.empty
    else pd.DataFrame(columns=["granularidad_min", "n_rows_expandidas", "n_rows_agregadas", "minutos_solapados_total", "n_barrios", "intervalo_min", "intervalo_max"])
)
if not smoke_overlap_expansion_summary.empty:
    aggregated_rows_by_granularity = smoke_aggregated.groupby("granularidad_min").size().rename("n_rows_agregadas").reset_index()
    smoke_overlap_expansion_summary = smoke_overlap_expansion_summary.merge(aggregated_rows_by_granularity, on="granularidad_min", how="left")
    smoke_overlap_expansion_summary = smoke_overlap_expansion_summary[["granularidad_min", "n_rows_expandidas", "n_rows_agregadas", "minutos_solapados_total", "n_barrios", "intervalo_min", "intervalo_max"]]

smoke_overlap_category_summary = (
    smoke_expanded["categoria_solape"]
    .value_counts(dropna=False)
    .rename_axis("categoria_solape")
    .reset_index(name="n_rows")
    if not smoke_expanded.empty
    else pd.DataFrame(columns=["categoria_solape", "n_rows"])
)

observable_keys = time_dimension_candidates.loc[
    time_dimension_candidates["granularidad_min"].isin(SMOKE_OVERLAP_GRANULARITIES_MIN),
    ["granularidad_min", "intervalo_inicio"],
].drop_duplicates()
expanded_observable_check = (
    smoke_expanded[["granularidad_min", "intervalo_inicio"]]
    .merge(observable_keys.assign(_observable=True), on=["granularidad_min", "intervalo_inicio"], how="left")
    ["_observable"]
    .fillna(False)
    .all()
    if not smoke_expanded.empty
    else False
)
smoke_overlap_checks = pd.DataFrame([
    {
        "check": "filas_validas_leidas",
        "ok": bool(n_rows_validas > 0),
        "critico": True,
        "detalle": f"filas válidas: {n_rows_validas}",
    },
    {
        "check": "expansion_genero_filas",
        "ok": bool(len(smoke_expanded) > 0),
        "critico": True,
        "detalle": f"filas expandidas observables: {len(smoke_expanded)}",
    },
    {
        "check": "minutos_solapados_positivos",
        "ok": bool((smoke_expanded["minutos_solapados"] > 0).all()) if not smoke_expanded.empty else False,
        "critico": True,
        "detalle": "todos los solapes conservados son positivos",
    },
    {
        "check": "minutos_solapados_no_superan_granularidad",
        "ok": bool((smoke_expanded["minutos_solapados"] <= smoke_expanded["granularidad_min"]).all()) if not smoke_expanded.empty else False,
        "critico": True,
        "detalle": "ningún solape supera la duración del intervalo",
    },
    {
        "check": "intervalos_ser_observables",
        "ok": bool(expanded_observable_check),
        "critico": True,
        "detalle": "todas las filas expandidas pertenecen a time_dimension_candidates",
    },
    {
        "check": "categoria_unica_abcd",
        "ok": bool((smoke_expanded["categoria_count"] == 1).all()) if not smoke_expanded.empty else False,
        "critico": True,
        "detalle": "cada fila expandida tiene una y solo una categoría A/B/C/D",
    },
    {
        "check": "agregacion_sin_barrio_nulo",
        "ok": bool(smoke_aggregated["barrio_key"].notna().all()) if not smoke_aggregated.empty else False,
        "critico": True,
        "detalle": "la agregación conserva barrio_key no nulo",
    },
    {
        "check": "capacidad_no_nula_en_agregados",
        "ok": bool(smoke_aggregated_with_capacity["plazas_barrio_anio"].notna().all()) if not smoke_aggregated_with_capacity.empty else False,
        "critico": True,
        "detalle": "todas las filas agregadas enlazan con capacidad barrio-año",
    },
])

display(smoke_overlap_input_summary)
display(smoke_overlap_expansion_summary)
display(smoke_overlap_category_summary)
display(smoke_overlap_checks)

failed_critical_smoke_checks = smoke_overlap_checks.loc[smoke_overlap_checks["critico"] & ~smoke_overlap_checks["ok"]]
if not failed_critical_smoke_checks.empty:
    raise ValueError("Fallan checks críticos del smoke test de solapes. Revisa `smoke_overlap_checks`.")

### Lectura metodológica del smoke test

El smoke test valida correctamente la mecánica inicial de solapes. La prueba se ejecuta sobre dos ficheros seleccionados desde `largest_part_per_period`, correspondientes a `2023_q1` y `2023_q2`, con granularidad de 60 minutos. Se leen 476.517 tiques válidos y la expansión genera 957.452 solapes observables, que se agregan inmediatamente a 52.635 filas barrio-intervalo.

La expansión temporal es coherente: todos los solapes conservados tienen minutos positivos, ningún solape supera la duración del intervalo, todas las filas expandidas pertenecen a intervalos SER observables y cada fila queda asignada a una única categoría A/B/C/D. Además, la agregación conserva `barrio_key` y enlaza con capacidad barrio-año, lo que confirma que el denominador estructural estará disponible para el cálculo posterior de métricas normalizadas.

La distribución de categorías muestra que las cuatro situaciones temporales están presentes en la muestra. Por tanto, el smoke test no valida únicamente un caso trivial: aparecen tiques persistentes durante todo el intervalo, tiques que salen, tiques que entran y salen dentro del mismo intervalo, y tiques que entran y continúan después. Esto confirma que la descomposición A/B/C/D es operativa sobre datos reales.

Los resultados de esta sección no deben interpretarse todavía como indicadores de dificultad SER. La muestra se ha elegido por conveniencia técnica y cubre solo dos ficheros concretos, no el conjunto completo de tiques ni todos los barrios-periodos. Su función es validar la mecánica de cálculo, la agregación inmediata y la disponibilidad del denominador, no describir aún el patrón espacial o temporal de aparcamiento.

Con esta evidencia, el procedimiento puede escalarse al siguiente nivel: ejecutar el mismo cálculo sobre las 13 particiones de `largest_part_per_period`, manteniendo primero la granularidad de 60 minutos y sin escribir salidas de producción. Solo después de validar ese escalado debería ampliarse a las cuatro granularidades candidatas y pasar al cálculo formal de métricas proxy.

## 12. Cálculo de métricas proxy en muestra agregada

Una vez validada la mecánica de solapes y agregación inmediata, se calcula el primer conjunto de métricas proxy sobre la muestra agregada del smoke test. Esta sección no vuelve a expandir tiques ni procesa más particiones: parte de `smoke_aggregated_with_capacity`, que ya contiene una fila por `barrio_key`, intervalo y granularidad, junto con los conteos A/B/C/D, minutos solapados y denominador `plazas_barrio_anio`.

El objetivo no es todavía describir la dificultad SER de Madrid ni fijar el target final, sino comprobar que las fórmulas se pueden aplicar de forma coherente sobre una tabla agregada. Por tanto, las métricas calculadas aquí se interpretan como validación metodológica en muestra técnica.

La métrica nuclear es `ocupacion_pagada_proxy`, calculada como minutos pagados solapados divididos entre la capacidad-tiempo del barrio. Esta variable aproxima la intensidad de uso pagado durante el intervalo, pero no mide ocupación real de todas las plazas SER. Las demás métricas son auxiliares: permiten distinguir entre presión ya existente, entradas, salidas, persistencia y rotación.

| Métrica                  | Fórmula operativa                                             | Papel en el TFM                                              |
| ------------------------ | ------------------------------------------------------------- | ------------------------------------------------------------ |
| `ocupacion_pagada_proxy` | `minutos_solapados / (plazas_barrio_anio * granularidad_min)` | Núcleo del target candidato global.                          |
| `stock_inicio_proxy`     | `n_activo_inicio / plazas_barrio_anio`                        | Presión pagada activa al inicio del intervalo.               |
| `stock_fin_proxy`        | `n_activo_fin / plazas_barrio_anio`                           | Presión pagada activa al cierre del intervalo.               |
| `afluencia_tiques`       | `n_entra_intervalo / plazas_barrio_anio`                      | Entrada de nueva demanda pagada.                             |
| `liberacion_tiques`      | `n_sale_intervalo / plazas_barrio_anio`                       | Finalización de tiques; oportunidad potencial de renovación. |
| `persistencia_completa`  | `n_persistente_completo / plazas_barrio_anio`                 | Rigidez de la ocupación pagada durante todo el intervalo.    |
| `saldo_flujo`            | `(n_D - n_B) / plazas_barrio_anio`                            | Tendencia neta de presión pagada dentro del intervalo.       |
| `rotacion_bruta`         | `(n_B + 2*n_C + n_D) / plazas_barrio_anio`                    | Movimiento total de entradas y salidas pagadas.              |

`saldo_flujo` y `rotacion_bruta` se calculan porque facilitan la lectura conjunta del fenómeno, pero son métricas derivadas de las entradas y salidas. Por tanto, no deben tratarse automáticamente como variables independientes en modelado sin una revisión posterior de redundancia y colinealidad.

En esta sección no se construyen todavía `dificultad_SER_proxy`, `prob_aparcar_proxy`, clases de dificultad ni flags binarios. Esas decisiones quedan condicionadas al análisis posterior de distribuciones, estabilidad temporal, granularidad y sensibilidad.


In [16]:
if "smoke_aggregated_with_capacity" not in globals():
    raise NameError("No existe `smoke_aggregated_with_capacity`; ejecuta antes el smoke test agregado de la sección 13.")

required_smoke_metric_columns = [
    "granularidad_min",
    "barrio_key",
    "intervalo_inicio",
    "intervalo_fin",
    "anio_intervalo",
    "n_tiques_solapados",
    "minutos_solapados",
    "n_A",
    "n_B",
    "n_C",
    "n_D",
    "n_activo_inicio",
    "n_activo_fin",
    "n_entra_intervalo",
    "n_sale_intervalo",
    "n_persistente_completo",
    "plazas_barrio_anio",
]
missing_smoke_metric_columns = [
    column for column in required_smoke_metric_columns
    if column not in smoke_aggregated_with_capacity.columns
]
if missing_smoke_metric_columns:
    raise ValueError(
        "Faltan columnas mínimas para calcular métricas proxy en la muestra agregada: "
        f"{missing_smoke_metric_columns}"
    )

smoke_metrics_sample = smoke_aggregated_with_capacity.copy()
valid_capacity = smoke_metrics_sample["plazas_barrio_anio"] > 0

metric_formulas = {
    "ocupacion_pagada_proxy": smoke_metrics_sample["minutos_solapados"] / (smoke_metrics_sample["plazas_barrio_anio"] * smoke_metrics_sample["granularidad_min"]),
    "stock_inicio_proxy": smoke_metrics_sample["n_activo_inicio"] / smoke_metrics_sample["plazas_barrio_anio"],
    "stock_fin_proxy": smoke_metrics_sample["n_activo_fin"] / smoke_metrics_sample["plazas_barrio_anio"],
    "afluencia_tiques": smoke_metrics_sample["n_entra_intervalo"] / smoke_metrics_sample["plazas_barrio_anio"],
    "liberacion_tiques": smoke_metrics_sample["n_sale_intervalo"] / smoke_metrics_sample["plazas_barrio_anio"],
    "persistencia_completa": smoke_metrics_sample["n_persistente_completo"] / smoke_metrics_sample["plazas_barrio_anio"],
    "saldo_flujo": (smoke_metrics_sample["n_D"] - smoke_metrics_sample["n_B"]) / smoke_metrics_sample["plazas_barrio_anio"],
    "rotacion_bruta": (smoke_metrics_sample["n_B"] + 2 * smoke_metrics_sample["n_C"] + smoke_metrics_sample["n_D"]) / smoke_metrics_sample["plazas_barrio_anio"],
}

smoke_metric_columns = list(metric_formulas)
for metric, values in metric_formulas.items():
    smoke_metrics_sample[metric] = np.where(valid_capacity, values, np.nan)

smoke_metrics_calculation_summary = pd.DataFrame([
    {
        "n_rows_agregadas": len(smoke_metrics_sample),
        "n_rows_capacidad_valida": int(valid_capacity.sum()),
        "n_barrios": smoke_metrics_sample["barrio_key"].nunique(dropna=True),
        "granularidades": sorted(smoke_metrics_sample["granularidad_min"].dropna().unique().tolist()),
        "intervalo_min": smoke_metrics_sample["intervalo_inicio"].min(),
        "intervalo_max": smoke_metrics_sample["intervalo_fin"].max(),
        "n_metricas_calculadas": len(smoke_metric_columns),
    }
])

smoke_metrics_distribution_summary = pd.DataFrame([
    {
        "metrica": metric,
        "n_non_null": int(smoke_metrics_sample[metric].notna().sum()),
        "n_null": int(smoke_metrics_sample[metric].isna().sum()),
        "min": smoke_metrics_sample[metric].min(skipna=True),
        "p50": smoke_metrics_sample[metric].quantile(0.50),
        "p95": smoke_metrics_sample[metric].quantile(0.95),
        "p99": smoke_metrics_sample[metric].quantile(0.99),
        "max": smoke_metrics_sample[metric].max(skipna=True),
    }
    for metric in smoke_metric_columns
])

ocupacion = smoke_metrics_sample["ocupacion_pagada_proxy"]
saldo = smoke_metrics_sample["saldo_flujo"]
n_metric_rows = int(ocupacion.notna().sum())
smoke_metrics_signal_summary = pd.DataFrame([
    {
        "n_rows_ocupacion_gt_1": int((ocupacion > 1).sum()),
        "pct_rows_ocupacion_gt_1": (ocupacion > 1).sum() / n_metric_rows if n_metric_rows else np.nan,
        "n_rows_ocupacion_eq_0": int((ocupacion == 0).sum()),
        "pct_rows_ocupacion_eq_0": (ocupacion == 0).sum() / n_metric_rows if n_metric_rows else np.nan,
        "n_rows_saldo_positivo": int((saldo > 0).sum()),
        "n_rows_saldo_negativo": int((saldo < 0).sum()),
        "n_rows_saldo_cero": int((saldo == 0).sum()),
    }
])

metric_values = smoke_metrics_sample[smoke_metric_columns]
finite_values = np.isfinite(metric_values.to_numpy(dtype=float)) | metric_values.isna().to_numpy()
non_negative_metric_columns = [metric for metric in smoke_metric_columns if metric != "saldo_flujo"]
non_negative_values = smoke_metrics_sample[non_negative_metric_columns]

tolerance = 1e-10
saldo_expected = smoke_metrics_sample["afluencia_tiques"] - smoke_metrics_sample["liberacion_tiques"]
rotacion_expected = smoke_metrics_sample["afluencia_tiques"] + smoke_metrics_sample["liberacion_tiques"]
ocupacion_expected = smoke_metrics_sample["minutos_solapados"] / (smoke_metrics_sample["plazas_barrio_anio"] * smoke_metrics_sample["granularidad_min"])

smoke_metrics_checks = pd.DataFrame([
    {
        "check": "input_agregado_no_vacio",
        "ok": bool(not smoke_metrics_sample.empty),
        "critico": True,
        "detalle": f"filas agregadas: {len(smoke_metrics_sample)}",
    },
    {
        "check": "capacidad_valida_en_todas_las_filas",
        "ok": bool(valid_capacity.all()) if not smoke_metrics_sample.empty else False,
        "critico": True,
        "detalle": "todas las filas tienen plazas_barrio_anio > 0",
    },
    {
        "check": "metricas_finitas",
        "ok": bool(finite_values.all()) if not smoke_metrics_sample.empty else False,
        "critico": True,
        "detalle": "no hay inf ni -inf en las métricas calculadas",
    },
    {
        "check": "metricas_no_negativas_salvo_saldo",
        "ok": bool((non_negative_values.dropna(how="all") >= 0).all().all()) if not smoke_metrics_sample.empty else False,
        "critico": True,
        "detalle": "todas las métricas excepto saldo_flujo son >= 0 cuando no son nulas",
    },
    {
        "check": "saldo_equivale_afluencia_menos_liberacion",
        "ok": bool(np.nanmax(np.abs(smoke_metrics_sample["saldo_flujo"] - saldo_expected)) <= tolerance) if not smoke_metrics_sample.empty else False,
        "critico": True,
        "detalle": "saldo_flujo coincide con afluencia_tiques - liberacion_tiques",
    },
    {
        "check": "rotacion_equivale_afluencia_mas_liberacion",
        "ok": bool(np.nanmax(np.abs(smoke_metrics_sample["rotacion_bruta"] - rotacion_expected)) <= tolerance) if not smoke_metrics_sample.empty else False,
        "critico": True,
        "detalle": "rotacion_bruta coincide con afluencia_tiques + liberacion_tiques",
    },
    {
        "check": "ocupacion_usa_minutos_exactos",
        "ok": bool(np.nanmax(np.abs(smoke_metrics_sample["ocupacion_pagada_proxy"] - ocupacion_expected)) <= tolerance) if not smoke_metrics_sample.empty else False,
        "critico": True,
        "detalle": "ocupacion_pagada_proxy coincide con minutos solapados / capacidad-tiempo",
    },
])

display(smoke_metrics_calculation_summary)
display(smoke_metrics_distribution_summary)
display(smoke_metrics_signal_summary)
display(smoke_metrics_checks)

failed_critical_metric_checks = smoke_metrics_checks.loc[
    smoke_metrics_checks["critico"] & ~smoke_metrics_checks["ok"]
]
if not failed_critical_metric_checks.empty:
    raise ValueError("Fallan checks críticos del cálculo de métricas proxy en muestra agregada. Revisa `smoke_metrics_checks`.")

,n_rows_agregadas,n_rows_capacidad_valida,n_barrios,granularidades,intervalo_min,intervalo_max,n_metricas_calculadas
0,52635,52635,55,[60],2023-01-02 09:00:00,2023-06-29 21:00:00,8


,metrica,n_non_null,n_null,min,p50,p95,p99,max
0,ocupacion_pagada_proxy,52635,0,2.477946e-07,0.002313,0.009928,0.019294,0.044413
1,stock_inicio_proxy,52635,0,0.000000e+00,0.002186,0.009811,0.019227,0.044420
2,stock_fin_proxy,52635,0,0.000000e+00,0.002186,0.009811,0.019227,0.044420
3,afluencia_tiques,52635,0,0.000000e+00,0.001936,0.010628,0.020399,0.048704
4,liberacion_tiques,52635,0,0.000000e+00,0.001937,0.010521,0.020564,0.069705
5,persistencia_completa,52635,0,0.000000e+00,0.000720,0.003559,0.006496,0.037790
6,saldo_flujo,52635,0,-3.887399e-02,0.000000,0.003117,0.006498,0.039337
7,rotacion_bruta,52635,0,0.000000e+00,0.003931,0.020946,0.039654,0.100536


,n_rows_ocupacion_gt_1,pct_rows_ocupacion_gt_1,n_rows_ocupacion_eq_0,pct_rows_ocupacion_eq_0,n_rows_saldo_positivo,n_rows_saldo_negativo,n_rows_saldo_cero
0,0,0.0,0,0.0,23187,22650,6798


,check,ok,critico,detalle
0,input_agregado_no_vacio,True,True,filas agregadas: 52635
1,capacidad_valida_en_todas_las_filas,True,True,todas las filas tienen plazas_barrio_anio > 0
2,metricas_finitas,True,True,no hay inf ni -inf en las métricas calculadas
3,metricas_no_negativas_salvo_saldo,True,True,todas las métricas excepto saldo_flujo son >= 0 cuando no son nulas
4,saldo_equivale_afluencia_menos_liberacion,True,True,saldo_flujo coincide con afluencia_tiques - liberacion_tiques
5,rotacion_equivale_afluencia_mas_liberacion,True,True,rotacion_bruta coincide con afluencia_tiques + liberacion_tiques
6,ocupacion_usa_minutos_exactos,True,True,ocupacion_pagada_proxy coincide con minutos solapados / capacidad-tiempo


### Lectura metodológica de métricas en muestra

El cálculo de métricas proxy se realiza correctamente sobre la muestra agregada del smoke test. La tabla de entrada ya no contiene tiques individuales, sino registros barrio-intervalo con minutos solapados, categorías A/B/C/D y denominador anual de capacidad. Esto confirma que las métricas pueden calcularse sobre una estructura agregada y trazable, sin conservar la tabla expandida tique-intervalo como salida persistente.

`ocupacion_pagada_proxy` queda como métrica nuclear del target candidato global, porque utiliza minutos exactos de solape y capacidad-tiempo del barrio. Las métricas de stock, afluencia, liberación y persistencia permiten contextualizar esa ocupación pagada: no es equivalente una ocupación elevada con alta liberación que una ocupación elevada con persistencia completa.

`saldo_flujo` y `rotacion_bruta` se conservan como métricas derivadas de lectura. La primera resume si la presión pagada tiende a aumentar o disminuir dentro del intervalo; la segunda resume el volumen total de movimiento pagado. Sin embargo, ambas proceden algebraicamente de afluencia y liberación, por lo que no deben tratarse todavía como variables independientes para modelado sin una revisión posterior de redundancia y colinealidad.

Los checks confirman que todas las filas agregadas tienen capacidad válida, que las métricas son finitas, que no aparecen valores negativos salvo en `saldo_flujo` —donde el signo tiene interpretación de tendencia— y que las equivalencias algebraicas se cumplen. También se confirma que no hay valores de `ocupacion_pagada_proxy` superiores a 1 en esta muestra, por lo que no aparecen incidencias de saturación proxy o inconsistencia en este subconjunto técnico.

Debe tenerse en cuenta que esta muestra agregada contiene únicamente intervalos con señal pagada observada. Por tanto, el hecho de que no aparezcan filas con `ocupacion_pagada_proxy = 0` no debe interpretarse como ausencia de intervalos sin demanda pagada. Esos ceros solo podrán evaluarse cuando las métricas se integren sobre la malla común barrio-intervalo mediante un `left join`, conservando también intervalos observables sin tiques.

Los resultados siguen correspondiendo a una muestra técnica limitada a dos ficheros y granularidad de 60 minutos. Por tanto, no se interpretan como patrón final de dificultad SER. La sección valida las fórmulas, los denominadores y las comprobaciones de coherencia; la lectura analítica definitiva deberá hacerse cuando el cálculo se escale a más particiones y a las cuatro granularidades candidatas.


## 13. Escalado de prueba a `largest_part_per_period` con granularidad 60

Tras validar la mecánica de solapes y el cálculo de métricas sobre una muestra reducida, se escala la prueba a todos los ficheros seleccionados en `largest_part_per_period`, manteniendo una única granularidad de 60 minutos. Este paso permite comprobar si el procedimiento sigue siendo estable al cubrir todos los periodos temporales disponibles, sin activar todavía el dataset completo ni las cuatro granularidades candidatas.

La finalidad de esta sección es técnica y metodológica: validar lectura por particiones, expansión controlada, agregación inmediata, enlace con capacidad y cálculo de métricas en una muestra más amplia. No se escriben salidas de producción y no se conserva la tabla expandida tique-intervalo como resultado final.

Los objetos generados en esta sección utilizan nombres independientes de los apartados anteriores para no sobrescribir el smoke test inicial. De este modo, las lecturas previas siguen siendo válidas como evidencia de una prueba reducida, mientras que esta sección documenta el siguiente nivel de escalado.


In [ ]:
SCALE60_PARTS = largest_part_per_period.copy()
SCALE60_GRANULARITIES_MIN = [60]

scale60_required_columns = [
    "fecha_inicio",
    "fecha_fin",
    "barrio_key",
    "anio",
]

scale60_ticket_parts = []
for _, part in SCALE60_PARTS.iterrows():
    part_path = part.get("path_obj", pd.NA)
    if pd.isna(part_path):
        part_path = ROOT / part["path"]
    part_df = read_ticket_part_columns(Path(part_path), scale60_required_columns)
    part_df["source_periodo_inicio"] = part["periodo_inicio"]
    scale60_ticket_parts.append(part_df)

scale60_tickets_raw = (
    pd.concat(scale60_ticket_parts, ignore_index=True)
    if scale60_ticket_parts
    else pd.DataFrame(columns=scale60_required_columns + ["source_periodo_inicio"])
)
scale60_n_rows_leidas = len(scale60_tickets_raw)

scale60_tickets = scale60_tickets_raw.copy()
scale60_tickets["fecha_inicio"] = pd.to_datetime(scale60_tickets["fecha_inicio"])
scale60_tickets["fecha_fin"] = pd.to_datetime(scale60_tickets["fecha_fin"])
scale60_tickets = scale60_tickets.loc[
    scale60_tickets["fecha_inicio"].notna()
    & scale60_tickets["fecha_fin"].notna()
    & (scale60_tickets["fecha_fin"] > scale60_tickets["fecha_inicio"])
    & scale60_tickets["barrio_key"].notna()
].copy()
scale60_tickets = scale60_tickets.reset_index(drop=True)
scale60_n_rows_validas = len(scale60_tickets)

scale60_expanded_parts = []
for granularidad_min in SCALE60_GRANULARITIES_MIN:
    expanded_granularity = expand_tickets_to_intervals_smoke(scale60_tickets, granularidad_min)
    observable_intervals = time_dimension_candidates.loc[
        time_dimension_candidates["granularidad_min"].eq(granularidad_min),
        ["granularidad_min", "intervalo_inicio", "intervalo_fin", "anio_intervalo"],
    ].drop_duplicates()
    expanded_granularity = expanded_granularity.drop(columns=["intervalo_fin"]).merge(
        observable_intervals,
        on=["granularidad_min", "intervalo_inicio"],
        how="inner",
    )
    scale60_expanded_parts.append(expanded_granularity)

scale60_expanded = (
    pd.concat(scale60_expanded_parts, ignore_index=True)
    if scale60_expanded_parts
    else pd.DataFrame()
)

if scale60_expanded.empty:
    scale60_aggregated = pd.DataFrame()
else:
    scale60_cond_A = (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_fin"])
    scale60_cond_B = (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_fin"] <= scale60_expanded["intervalo_fin"])
    scale60_cond_C = (scale60_expanded["fecha_inicio"] >= scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_fin"]) & (scale60_expanded["fecha_fin"] <= scale60_expanded["intervalo_fin"])
    scale60_cond_D = (scale60_expanded["fecha_inicio"] >= scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_fin"]) & (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_fin"])
    scale60_category_count = scale60_cond_A.astype(int) + scale60_cond_B.astype(int) + scale60_cond_C.astype(int) + scale60_cond_D.astype(int)

    scale60_expanded["categoria_solape"] = np.select(
        [scale60_cond_A, scale60_cond_B, scale60_cond_C, scale60_cond_D],
        ["A", "B", "C", "D"],
        default=pd.NA,
    )
    scale60_expanded["categoria_count"] = scale60_category_count
    scale60_expanded["activo_inicio"] = (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_inicio"])
    scale60_expanded["activo_fin"] = (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_fin"]) & (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_fin"])
    scale60_expanded["entra_intervalo"] = (scale60_expanded["fecha_inicio"] >= scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_fin"])
    scale60_expanded["sale_intervalo"] = (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_fin"] <= scale60_expanded["intervalo_fin"])
    scale60_expanded["persistente_completo"] = (scale60_expanded["fecha_inicio"] < scale60_expanded["intervalo_inicio"]) & (scale60_expanded["fecha_fin"] > scale60_expanded["intervalo_fin"])

    for category in ["A", "B", "C", "D"]:
        scale60_expanded[f"is_{category}"] = scale60_expanded["categoria_solape"].eq(category)

    scale60_aggregated = (
        scale60_expanded
        .groupby(["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"], dropna=False)
        .agg(
            n_tiques_solapados=("minutos_solapados", "size"),
            minutos_solapados=("minutos_solapados", "sum"),
            n_A=("is_A", "sum"),
            n_B=("is_B", "sum"),
            n_C=("is_C", "sum"),
            n_D=("is_D", "sum"),
            n_activo_inicio=("activo_inicio", "sum"),
            n_activo_fin=("activo_fin", "sum"),
            n_entra_intervalo=("entra_intervalo", "sum"),
            n_sale_intervalo=("sale_intervalo", "sum"),
            n_persistente_completo=("persistente_completo", "sum"),
        )
        .reset_index()
    )

if scale60_aggregated.empty:
    scale60_aggregated_with_capacity = scale60_aggregated.copy()
else:
    scale60_aggregated_with_capacity = scale60_aggregated.merge(
        capacity_for_smoke,
        left_on=["anio_intervalo", "barrio_key"],
        right_on=["anio", "barrio_key"],
        how="left",
    )

scale60_metrics_sample = scale60_aggregated_with_capacity.copy()
if not scale60_metrics_sample.empty:
    scale60_valid_capacity = scale60_metrics_sample["plazas_barrio_anio"] > 0
    scale60_metric_formulas = {
        "ocupacion_pagada_proxy": scale60_metrics_sample["minutos_solapados"] / (scale60_metrics_sample["plazas_barrio_anio"] * scale60_metrics_sample["granularidad_min"]),
        "stock_inicio_proxy": scale60_metrics_sample["n_activo_inicio"] / scale60_metrics_sample["plazas_barrio_anio"],
        "stock_fin_proxy": scale60_metrics_sample["n_activo_fin"] / scale60_metrics_sample["plazas_barrio_anio"],
        "afluencia_tiques": scale60_metrics_sample["n_entra_intervalo"] / scale60_metrics_sample["plazas_barrio_anio"],
        "liberacion_tiques": scale60_metrics_sample["n_sale_intervalo"] / scale60_metrics_sample["plazas_barrio_anio"],
        "persistencia_completa": scale60_metrics_sample["n_persistente_completo"] / scale60_metrics_sample["plazas_barrio_anio"],
        "saldo_flujo": (scale60_metrics_sample["n_D"] - scale60_metrics_sample["n_B"]) / scale60_metrics_sample["plazas_barrio_anio"],
        "rotacion_bruta": (scale60_metrics_sample["n_B"] + 2 * scale60_metrics_sample["n_C"] + scale60_metrics_sample["n_D"]) / scale60_metrics_sample["plazas_barrio_anio"],
    }
    scale60_metric_columns = list(scale60_metric_formulas)
    for metric, values in scale60_metric_formulas.items():
        scale60_metrics_sample[metric] = np.where(scale60_valid_capacity, values, np.nan)
else:
    scale60_valid_capacity = pd.Series(dtype=bool)
    scale60_metric_columns = [
        "ocupacion_pagada_proxy",
        "stock_inicio_proxy",
        "stock_fin_proxy",
        "afluencia_tiques",
        "liberacion_tiques",
        "persistencia_completa",
        "saldo_flujo",
        "rotacion_bruta",
    ]

scale60_periods = sorted(SCALE60_PARTS["periodo_inicio"].dropna().astype(str).tolist())
scale60_input_summary = pd.DataFrame([
    {
        "n_files": len(SCALE60_PARTS),
        "n_periodos": SCALE60_PARTS["periodo_inicio"].nunique(dropna=True),
        "periodo_min": min(scale60_periods) if scale60_periods else pd.NA,
        "periodo_max": max(scale60_periods) if scale60_periods else pd.NA,
        "n_rows_leidas": scale60_n_rows_leidas,
        "n_rows_validas": scale60_n_rows_validas,
        "granularidades": SCALE60_GRANULARITIES_MIN,
    }
])

scale60_expansion_summary = (
    scale60_expanded
    .groupby("granularidad_min", dropna=False)
    .agg(
        n_rows_expandidas=("minutos_solapados", "size"),
        minutos_solapados_total=("minutos_solapados", "sum"),
        n_barrios=("barrio_key", "nunique"),
        intervalo_min=("intervalo_inicio", "min"),
        intervalo_max=("intervalo_fin", "max"),
    )
    .reset_index()
    if not scale60_expanded.empty
    else pd.DataFrame(columns=["granularidad_min", "n_rows_expandidas", "n_rows_agregadas", "minutos_solapados_total", "n_barrios", "intervalo_min", "intervalo_max"])
)
if not scale60_expansion_summary.empty:
    scale60_aggregated_rows_by_granularity = scale60_aggregated.groupby("granularidad_min").size().rename("n_rows_agregadas").reset_index()
    scale60_expansion_summary = scale60_expansion_summary.merge(scale60_aggregated_rows_by_granularity, on="granularidad_min", how="left")
    scale60_expansion_summary = scale60_expansion_summary[["granularidad_min", "n_rows_expandidas", "n_rows_agregadas", "minutos_solapados_total", "n_barrios", "intervalo_min", "intervalo_max"]]

scale60_category_summary = (
    scale60_expanded["categoria_solape"]
    .value_counts(dropna=False)
    .rename_axis("categoria_solape")
    .reset_index(name="n_rows")
    if not scale60_expanded.empty
    else pd.DataFrame(columns=["categoria_solape", "n_rows"])
)

scale60_metrics_distribution_summary = pd.DataFrame([
    {
        "metrica": metric,
        "n_non_null": int(scale60_metrics_sample[metric].notna().sum()) if metric in scale60_metrics_sample else 0,
        "n_null": int(scale60_metrics_sample[metric].isna().sum()) if metric in scale60_metrics_sample else len(scale60_metrics_sample),
        "min": scale60_metrics_sample[metric].min(skipna=True) if metric in scale60_metrics_sample else np.nan,
        "p50": scale60_metrics_sample[metric].quantile(0.50) if metric in scale60_metrics_sample else np.nan,
        "p95": scale60_metrics_sample[metric].quantile(0.95) if metric in scale60_metrics_sample else np.nan,
        "p99": scale60_metrics_sample[metric].quantile(0.99) if metric in scale60_metrics_sample else np.nan,
        "max": scale60_metrics_sample[metric].max(skipna=True) if metric in scale60_metrics_sample else np.nan,
    }
    for metric in scale60_metric_columns
])

scale60_ocupacion = scale60_metrics_sample["ocupacion_pagada_proxy"] if "ocupacion_pagada_proxy" in scale60_metrics_sample else pd.Series(dtype=float)
scale60_saldo = scale60_metrics_sample["saldo_flujo"] if "saldo_flujo" in scale60_metrics_sample else pd.Series(dtype=float)
scale60_n_metric_rows = int(scale60_ocupacion.notna().sum())
scale60_signal_summary = pd.DataFrame([
    {
        "n_rows_ocupacion_gt_1": int((scale60_ocupacion > 1).sum()),
        "pct_rows_ocupacion_gt_1": (scale60_ocupacion > 1).sum() / scale60_n_metric_rows if scale60_n_metric_rows else np.nan,
        "n_rows_ocupacion_eq_0": int((scale60_ocupacion == 0).sum()),
        "pct_rows_ocupacion_eq_0": (scale60_ocupacion == 0).sum() / scale60_n_metric_rows if scale60_n_metric_rows else np.nan,
        "n_rows_saldo_positivo": int((scale60_saldo > 0).sum()),
        "n_rows_saldo_negativo": int((scale60_saldo < 0).sum()),
        "n_rows_saldo_cero": int((scale60_saldo == 0).sum()),
    }
])

scale60_observable_keys = time_dimension_candidates.loc[
    time_dimension_candidates["granularidad_min"].isin(SCALE60_GRANULARITIES_MIN),
    ["granularidad_min", "intervalo_inicio"],
].drop_duplicates()
scale60_expanded_observable_check = (
    scale60_expanded[["granularidad_min", "intervalo_inicio"]]
    .merge(scale60_observable_keys.assign(_observable=True), on=["granularidad_min", "intervalo_inicio"], how="left")
    ["_observable"]
    .fillna(False)
    .all()
    if not scale60_expanded.empty
    else False
)
scale60_metric_values = scale60_metrics_sample[scale60_metric_columns] if not scale60_metrics_sample.empty else pd.DataFrame(columns=scale60_metric_columns)
scale60_finite_values = np.isfinite(scale60_metric_values.to_numpy(dtype=float)) | scale60_metric_values.isna().to_numpy()
scale60_non_negative_metric_columns = [metric for metric in scale60_metric_columns if metric != "saldo_flujo"]
scale60_non_negative_values = scale60_metrics_sample[scale60_non_negative_metric_columns] if not scale60_metrics_sample.empty else pd.DataFrame(columns=scale60_non_negative_metric_columns)

scale60_tolerance = 1e-10
if not scale60_metrics_sample.empty:
    scale60_saldo_expected = scale60_metrics_sample["afluencia_tiques"] - scale60_metrics_sample["liberacion_tiques"]
    scale60_rotacion_expected = scale60_metrics_sample["afluencia_tiques"] + scale60_metrics_sample["liberacion_tiques"]
    scale60_ocupacion_expected = scale60_metrics_sample["minutos_solapados"] / (scale60_metrics_sample["plazas_barrio_anio"] * scale60_metrics_sample["granularidad_min"])
    scale60_saldo_diff_ok = bool(np.nanmax(np.abs(scale60_metrics_sample["saldo_flujo"] - scale60_saldo_expected)) <= scale60_tolerance)
    scale60_rotacion_diff_ok = bool(np.nanmax(np.abs(scale60_metrics_sample["rotacion_bruta"] - scale60_rotacion_expected)) <= scale60_tolerance)
    scale60_ocupacion_diff_ok = bool(np.nanmax(np.abs(scale60_metrics_sample["ocupacion_pagada_proxy"] - scale60_ocupacion_expected)) <= scale60_tolerance)
else:
    scale60_saldo_diff_ok = False
    scale60_rotacion_diff_ok = False
    scale60_ocupacion_diff_ok = False

scale60_checks = pd.DataFrame([
    {
        "check": "filas_validas_leidas",
        "ok": bool(scale60_n_rows_validas > 0),
        "critico": True,
        "detalle": f"filas válidas: {scale60_n_rows_validas}",
    },
    {
        "check": "expansion_genero_filas",
        "ok": bool(len(scale60_expanded) > 0),
        "critico": True,
        "detalle": f"filas expandidas observables: {len(scale60_expanded)}",
    },
    {
        "check": "minutos_solapados_positivos",
        "ok": bool((scale60_expanded["minutos_solapados"] > 0).all()) if not scale60_expanded.empty else False,
        "critico": True,
        "detalle": "todos los solapes conservados son positivos",
    },
    {
        "check": "minutos_solapados_no_superan_granularidad",
        "ok": bool((scale60_expanded["minutos_solapados"] <= scale60_expanded["granularidad_min"]).all()) if not scale60_expanded.empty else False,
        "critico": True,
        "detalle": "ningún solape supera la duración del intervalo",
    },
    {
        "check": "intervalos_ser_observables",
        "ok": bool(scale60_expanded_observable_check),
        "critico": True,
        "detalle": "todas las filas expandidas pertenecen a time_dimension_candidates",
    },
    {
        "check": "categoria_unica_abcd",
        "ok": bool((scale60_expanded["categoria_count"] == 1).all()) if not scale60_expanded.empty else False,
        "critico": True,
        "detalle": "cada fila expandida tiene una y solo una categoría A/B/C/D",
    },
    {
        "check": "agregacion_sin_barrio_nulo",
        "ok": bool(scale60_aggregated["barrio_key"].notna().all()) if not scale60_aggregated.empty else False,
        "critico": True,
        "detalle": "la agregación conserva barrio_key no nulo",
    },
    {
        "check": "capacidad_no_nula_en_agregados",
        "ok": bool(scale60_aggregated_with_capacity["plazas_barrio_anio"].notna().all()) if not scale60_aggregated_with_capacity.empty else False,
        "critico": True,
        "detalle": "todas las filas agregadas enlazan con capacidad barrio-año",
    },
    {
        "check": "metricas_finitas",
        "ok": bool(scale60_finite_values.all()) if not scale60_metrics_sample.empty else False,
        "critico": True,
        "detalle": "no hay inf ni -inf en las métricas calculadas",
    },
    {
        "check": "metricas_no_negativas_salvo_saldo",
        "ok": bool((scale60_non_negative_values.dropna(how="all") >= 0).all().all()) if not scale60_metrics_sample.empty else False,
        "critico": True,
        "detalle": "todas las métricas excepto saldo_flujo son >= 0 cuando no son nulas",
    },
    {
        "check": "saldo_equivale_afluencia_menos_liberacion",
        "ok": bool(scale60_saldo_diff_ok),
        "critico": True,
        "detalle": "saldo_flujo coincide con afluencia_tiques - liberacion_tiques",
    },
    {
        "check": "rotacion_equivale_afluencia_mas_liberacion",
        "ok": bool(scale60_rotacion_diff_ok),
        "critico": True,
        "detalle": "rotacion_bruta coincide con afluencia_tiques + liberacion_tiques",
    },
    {
        "check": "ocupacion_usa_minutos_exactos",
        "ok": bool(scale60_ocupacion_diff_ok),
        "critico": True,
        "detalle": "ocupacion_pagada_proxy coincide con minutos solapados / capacidad-tiempo",
    },
])

display(scale60_input_summary)
display(scale60_expansion_summary)
display(scale60_category_summary)
display(scale60_metrics_distribution_summary)
display(scale60_signal_summary)
display(scale60_checks)

failed_critical_scale60_checks = scale60_checks.loc[scale60_checks["critico"] & ~scale60_checks["ok"]]
if not failed_critical_scale60_checks.empty:
    raise ValueError("Fallan checks críticos del escalado de prueba a largest_part_per_period con granularidad 60. Revisa `scale60_checks`.")

### Lectura de control del escalado a 60 minutos

El escalado técnico a `largest_part_per_period` con granularidad de 60 minutos queda validado. La prueba cubre los 13 periodos temporales disponibles mediante un fichero físico por periodo, procesa más de tres millones de tiques válidos, genera solapes observables, agrega por barrio-intervalo y calcula métricas sin fallos críticos.

La prueba no representa todavía el panel completo ni debe interpretarse como resultado analítico de dificultad SER. Su función es confirmar que la lógica validada en el smoke test reducido escala correctamente a una muestra más amplia, manteniendo agregación inmediata, enlace con capacidad y controles de coherencia.

Con esta comprobación superada, el siguiente paso es repetir el proceso sobre la misma muestra `largest_part_per_period`, pero comparando las cuatro granularidades candidatas: 15, 30, 45 y 60 minutos.


## 14. Comparación técnica de granularidades sobre `largest_part_per_period`

Una vez validado el escalado a 60 minutos sobre la muestra `largest_part_per_period`, se repite el procedimiento para las cuatro granularidades candidatas: 15, 30, 45 y 60 minutos. Esta sección no busca elegir todavía la granularidad final, sino comprobar que el cálculo de solapes, agregación, enlace con capacidad y métricas funciona de forma estable en todas ellas.

La comparación se mantiene sobre la misma muestra técnica de particiones, sin procesar periodos completos ni el dataset completo. Para controlar memoria, la expansión se ejecuta granularidad por granularidad y se agrega inmediatamente. La tabla expandida tique-intervalo no se conserva como salida persistente.

Los resultados de esta sección se interpretan como diagnóstico de viabilidad computacional y coherencia técnica. La elección final de granularidad deberá apoyarse más adelante en criterios de variabilidad, sparsity, estabilidad temporal, interpretabilidad y coste real de ejecución.


In [ ]:
GRANULARITY_SCALE_PARTS = largest_part_per_period.copy()
GRANULARITY_SCALE_MIN = GRANULARITIES_MIN

if "scale60_tickets" in globals() and not scale60_tickets.empty:
    granularity_scale_tickets = scale60_tickets.copy()
else:
    granularity_required_columns = [
        "fecha_inicio",
        "fecha_fin",
        "barrio_key",
        "anio",
            ]
    granularity_scale_ticket_parts = []
    for _, part in GRANULARITY_SCALE_PARTS.iterrows():
        part_path = part.get("path_obj", pd.NA)
        if pd.isna(part_path):
            part_path = ROOT / part["path"]
        part_df = read_ticket_part_columns(Path(part_path), granularity_required_columns)
        part_df["source_periodo_inicio"] = part["periodo_inicio"]
        granularity_scale_ticket_parts.append(part_df)

    granularity_scale_tickets_raw = (
        pd.concat(granularity_scale_ticket_parts, ignore_index=True)
        if granularity_scale_ticket_parts
        else pd.DataFrame(columns=granularity_required_columns + ["source_periodo_inicio"])
    )
    granularity_scale_tickets = granularity_scale_tickets_raw.copy()
    granularity_scale_tickets["fecha_inicio"] = pd.to_datetime(granularity_scale_tickets["fecha_inicio"])
    granularity_scale_tickets["fecha_fin"] = pd.to_datetime(granularity_scale_tickets["fecha_fin"])
    granularity_scale_tickets = granularity_scale_tickets.loc[
        granularity_scale_tickets["fecha_inicio"].notna()
        & granularity_scale_tickets["fecha_fin"].notna()
        & (granularity_scale_tickets["fecha_fin"] > granularity_scale_tickets["fecha_inicio"])
        & granularity_scale_tickets["barrio_key"].notna()
    ].copy().reset_index(drop=True)

granularity_scale_input_summary = pd.DataFrame([
    {
        "n_files": len(GRANULARITY_SCALE_PARTS),
        "n_periodos": GRANULARITY_SCALE_PARTS["periodo_inicio"].nunique(dropna=True),
        "n_rows_validas": len(granularity_scale_tickets),
        "granularidades": GRANULARITY_SCALE_MIN,
    }
])

granularity_scale_aggregated_parts = []
granularity_scale_metrics_parts = []
granularity_scale_expansion_summary_rows = []
granularity_scale_category_summary_parts = []
granularity_scale_metrics_summary_parts = []
granularity_scale_signal_summary_rows = []
granularity_scale_check_rows = []
granularity_scale_failed_critical = []

granularity_metric_columns = [
    "ocupacion_pagada_proxy",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]

def add_granularity_check(granularidad_min: int, check: str, ok: bool, detalle: str, critico: bool = True):
    granularity_scale_check_rows.append({
        "granularidad_min": granularidad_min,
        "check": check,
        "ok": bool(ok),
        "critico": critico,
        "detalle": detalle,
    })
    if critico and not ok:
        granularity_scale_failed_critical.append((granularidad_min, check))

for granularidad_min in GRANULARITY_SCALE_MIN:
    granularity_expanded_current = expand_tickets_to_intervals_smoke(granularity_scale_tickets, granularidad_min)
    granularity_observable_intervals = time_dimension_candidates.loc[
        time_dimension_candidates["granularidad_min"].eq(granularidad_min),
        ["granularidad_min", "intervalo_inicio", "intervalo_fin", "anio_intervalo"],
    ].drop_duplicates()
    granularity_expanded_current = granularity_expanded_current.drop(columns=["intervalo_fin"]).merge(
        granularity_observable_intervals,
        on=["granularidad_min", "intervalo_inicio"],
        how="inner",
    )

    if granularity_expanded_current.empty:
        granularity_aggregated_current = pd.DataFrame()
        granularity_metrics_current = pd.DataFrame()
        granularity_category_count = pd.Series(dtype=int)
    else:
        cond_A = (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_fin"])
        cond_B = (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_fin"] <= granularity_expanded_current["intervalo_fin"])
        cond_C = (granularity_expanded_current["fecha_inicio"] >= granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_fin"]) & (granularity_expanded_current["fecha_fin"] <= granularity_expanded_current["intervalo_fin"])
        cond_D = (granularity_expanded_current["fecha_inicio"] >= granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_fin"]) & (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_fin"])
        granularity_expanded_current["categoria_count"] = cond_A.astype(int) + cond_B.astype(int) + cond_C.astype(int) + cond_D.astype(int)
        granularity_expanded_current["categoria_solape"] = np.select([cond_A, cond_B, cond_C, cond_D], ["A", "B", "C", "D"], default=pd.NA)
        granularity_expanded_current["activo_inicio"] = (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_inicio"])
        granularity_expanded_current["activo_fin"] = (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_fin"]) & (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_fin"])
        granularity_expanded_current["entra_intervalo"] = (granularity_expanded_current["fecha_inicio"] >= granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_fin"])
        granularity_expanded_current["sale_intervalo"] = (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_fin"] <= granularity_expanded_current["intervalo_fin"])
        granularity_expanded_current["persistente_completo"] = (granularity_expanded_current["fecha_inicio"] < granularity_expanded_current["intervalo_inicio"]) & (granularity_expanded_current["fecha_fin"] > granularity_expanded_current["intervalo_fin"])

        for category in ["A", "B", "C", "D"]:
            granularity_expanded_current[f"is_{category}"] = granularity_expanded_current["categoria_solape"].eq(category)

        granularity_aggregated_current = (
            granularity_expanded_current
            .groupby(["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"], dropna=False)
            .agg(
                n_tiques_solapados=("minutos_solapados", "size"),
                minutos_solapados=("minutos_solapados", "sum"),
                n_A=("is_A", "sum"),
                n_B=("is_B", "sum"),
                n_C=("is_C", "sum"),
                n_D=("is_D", "sum"),
                n_activo_inicio=("activo_inicio", "sum"),
                n_activo_fin=("activo_fin", "sum"),
                n_entra_intervalo=("entra_intervalo", "sum"),
                n_sale_intervalo=("sale_intervalo", "sum"),
                n_persistente_completo=("persistente_completo", "sum"),
            )
            .reset_index()
        )
        granularity_scale_aggregated_parts.append(granularity_aggregated_current)

        granularity_metrics_current = granularity_aggregated_current.merge(
            capacity_for_smoke,
            left_on=["anio_intervalo", "barrio_key"],
            right_on=["anio", "barrio_key"],
            how="left",
        )
        valid_capacity_current = granularity_metrics_current["plazas_barrio_anio"] > 0
        granularity_metric_formulas = {
            "ocupacion_pagada_proxy": granularity_metrics_current["minutos_solapados"] / (granularity_metrics_current["plazas_barrio_anio"] * granularity_metrics_current["granularidad_min"]),
            "stock_inicio_proxy": granularity_metrics_current["n_activo_inicio"] / granularity_metrics_current["plazas_barrio_anio"],
            "stock_fin_proxy": granularity_metrics_current["n_activo_fin"] / granularity_metrics_current["plazas_barrio_anio"],
            "afluencia_tiques": granularity_metrics_current["n_entra_intervalo"] / granularity_metrics_current["plazas_barrio_anio"],
            "liberacion_tiques": granularity_metrics_current["n_sale_intervalo"] / granularity_metrics_current["plazas_barrio_anio"],
            "persistencia_completa": granularity_metrics_current["n_persistente_completo"] / granularity_metrics_current["plazas_barrio_anio"],
            "saldo_flujo": (granularity_metrics_current["n_D"] - granularity_metrics_current["n_B"]) / granularity_metrics_current["plazas_barrio_anio"],
            "rotacion_bruta": (granularity_metrics_current["n_B"] + 2 * granularity_metrics_current["n_C"] + granularity_metrics_current["n_D"]) / granularity_metrics_current["plazas_barrio_anio"],
        }
        for metric, values in granularity_metric_formulas.items():
            granularity_metrics_current[metric] = np.where(valid_capacity_current, values, np.nan)
        granularity_scale_metrics_parts.append(granularity_metrics_current)

        granularity_category_count = granularity_expanded_current["categoria_solape"].value_counts(dropna=False)
        granularity_scale_category_summary_parts.append(
            granularity_category_count.rename_axis("categoria_solape").reset_index(name="n_rows").assign(granularidad_min=granularidad_min)
        )

        granularity_scale_metrics_summary_parts.append(pd.DataFrame([
            {
                "granularidad_min": granularidad_min,
                "metrica": metric,
                "n_non_null": int(granularity_metrics_current[metric].notna().sum()),
                "n_null": int(granularity_metrics_current[metric].isna().sum()),
                "min": granularity_metrics_current[metric].min(skipna=True),
                "p50": granularity_metrics_current[metric].quantile(0.50),
                "p95": granularity_metrics_current[metric].quantile(0.95),
                "p99": granularity_metrics_current[metric].quantile(0.99),
                "max": granularity_metrics_current[metric].max(skipna=True),
            }
            for metric in granularity_metric_columns
        ]))

    granularity_scale_expansion_summary_rows.append({
        "granularidad_min": granularidad_min,
        "n_rows_expandidas": len(granularity_expanded_current),
        "n_rows_agregadas": len(granularity_aggregated_current),
        "minutos_solapados_total": granularity_expanded_current["minutos_solapados"].sum() if not granularity_expanded_current.empty else 0.0,
        "n_barrios": granularity_expanded_current["barrio_key"].nunique(dropna=True) if not granularity_expanded_current.empty else 0,
        "intervalo_min": granularity_expanded_current["intervalo_inicio"].min() if not granularity_expanded_current.empty else pd.NaT,
        "intervalo_max": granularity_expanded_current["intervalo_fin"].max() if not granularity_expanded_current.empty else pd.NaT,
    })

    if not granularity_metrics_current.empty:
        ocupacion_current = granularity_metrics_current["ocupacion_pagada_proxy"]
        saldo_current = granularity_metrics_current["saldo_flujo"]
        n_metric_rows_current = int(ocupacion_current.notna().sum())
        granularity_scale_signal_summary_rows.append({
            "granularidad_min": granularidad_min,
            "n_rows_ocupacion_gt_1": int((ocupacion_current > 1).sum()),
            "pct_rows_ocupacion_gt_1": (ocupacion_current > 1).sum() / n_metric_rows_current if n_metric_rows_current else np.nan,
            "n_rows_ocupacion_eq_0": int((ocupacion_current == 0).sum()),
            "pct_rows_ocupacion_eq_0": (ocupacion_current == 0).sum() / n_metric_rows_current if n_metric_rows_current else np.nan,
            "n_rows_saldo_positivo": int((saldo_current > 0).sum()),
            "n_rows_saldo_negativo": int((saldo_current < 0).sum()),
            "n_rows_saldo_cero": int((saldo_current == 0).sum()),
        })
    else:
        granularity_scale_signal_summary_rows.append({
            "granularidad_min": granularidad_min,
            "n_rows_ocupacion_gt_1": 0,
            "pct_rows_ocupacion_gt_1": np.nan,
            "n_rows_ocupacion_eq_0": 0,
            "pct_rows_ocupacion_eq_0": np.nan,
            "n_rows_saldo_positivo": 0,
            "n_rows_saldo_negativo": 0,
            "n_rows_saldo_cero": 0,
        })

    observable_keys_current = granularity_observable_intervals[["granularidad_min", "intervalo_inicio"]].drop_duplicates()
    observable_check_current = (
        granularity_expanded_current[["granularidad_min", "intervalo_inicio"]]
        .merge(observable_keys_current.assign(_observable=True), on=["granularidad_min", "intervalo_inicio"], how="left")
        ["_observable"]
        .fillna(False)
        .all()
        if not granularity_expanded_current.empty
        else False
    )
    metric_values_current = granularity_metrics_current[granularity_metric_columns] if not granularity_metrics_current.empty else pd.DataFrame(columns=granularity_metric_columns)
    finite_values_current = np.isfinite(metric_values_current.to_numpy(dtype=float)) | metric_values_current.isna().to_numpy()
    non_negative_cols_current = [metric for metric in granularity_metric_columns if metric != "saldo_flujo"]
    non_negative_values_current = granularity_metrics_current[non_negative_cols_current] if not granularity_metrics_current.empty else pd.DataFrame(columns=non_negative_cols_current)

    if not granularity_metrics_current.empty:
        saldo_expected_current = granularity_metrics_current["afluencia_tiques"] - granularity_metrics_current["liberacion_tiques"]
        rotacion_expected_current = granularity_metrics_current["afluencia_tiques"] + granularity_metrics_current["liberacion_tiques"]
        ocupacion_expected_current = granularity_metrics_current["minutos_solapados"] / (granularity_metrics_current["plazas_barrio_anio"] * granularity_metrics_current["granularidad_min"])
        saldo_ok_current = bool(np.nanmax(np.abs(granularity_metrics_current["saldo_flujo"] - saldo_expected_current)) <= 1e-10)
        rotacion_ok_current = bool(np.nanmax(np.abs(granularity_metrics_current["rotacion_bruta"] - rotacion_expected_current)) <= 1e-10)
        ocupacion_ok_current = bool(np.nanmax(np.abs(granularity_metrics_current["ocupacion_pagada_proxy"] - ocupacion_expected_current)) <= 1e-10)
    else:
        saldo_ok_current = False
        rotacion_ok_current = False
        ocupacion_ok_current = False

    add_granularity_check(granularidad_min, "filas_validas_leidas", len(granularity_scale_tickets) > 0, f"filas válidas: {len(granularity_scale_tickets)}")
    add_granularity_check(granularidad_min, "expansion_genero_filas", len(granularity_expanded_current) > 0, f"filas expandidas observables: {len(granularity_expanded_current)}")
    add_granularity_check(granularidad_min, "minutos_solapados_positivos", (granularity_expanded_current["minutos_solapados"] > 0).all() if not granularity_expanded_current.empty else False, "todos los solapes conservados son positivos")
    add_granularity_check(granularidad_min, "minutos_solapados_no_superan_granularidad", (granularity_expanded_current["minutos_solapados"] <= granularity_expanded_current["granularidad_min"]).all() if not granularity_expanded_current.empty else False, "ningún solape supera la duración del intervalo")
    add_granularity_check(granularidad_min, "intervalos_ser_observables", observable_check_current, "todas las filas expandidas pertenecen a time_dimension_candidates")
    add_granularity_check(granularidad_min, "categoria_unica_abcd", (granularity_expanded_current["categoria_count"] == 1).all() if not granularity_expanded_current.empty else False, "cada fila expandida tiene una y solo una categoría A/B/C/D")
    add_granularity_check(granularidad_min, "agregacion_sin_barrio_nulo", granularity_aggregated_current["barrio_key"].notna().all() if not granularity_aggregated_current.empty else False, "la agregación conserva barrio_key no nulo")
    add_granularity_check(granularidad_min, "capacidad_no_nula_en_agregados", granularity_metrics_current["plazas_barrio_anio"].notna().all() if not granularity_metrics_current.empty else False, "todas las filas agregadas enlazan con capacidad barrio-año")
    add_granularity_check(granularidad_min, "metricas_finitas", finite_values_current.all() if not granularity_metrics_current.empty else False, "no hay inf ni -inf en las métricas calculadas")
    add_granularity_check(granularidad_min, "metricas_no_negativas_salvo_saldo", (non_negative_values_current.dropna(how="all") >= 0).all().all() if not granularity_metrics_current.empty else False, "todas las métricas excepto saldo_flujo son >= 0 cuando no son nulas")
    add_granularity_check(granularidad_min, "saldo_equivale_afluencia_menos_liberacion", saldo_ok_current, "saldo_flujo coincide con afluencia_tiques - liberacion_tiques")
    add_granularity_check(granularidad_min, "rotacion_equivale_afluencia_mas_liberacion", rotacion_ok_current, "rotacion_bruta coincide con afluencia_tiques + liberacion_tiques")
    add_granularity_check(granularidad_min, "ocupacion_usa_minutos_exactos", ocupacion_ok_current, "ocupacion_pagada_proxy coincide con minutos solapados / capacidad-tiempo")

granularity_scale_expansion_summary = pd.DataFrame(granularity_scale_expansion_summary_rows)
granularity_scale_category_summary = (
    pd.concat(granularity_scale_category_summary_parts, ignore_index=True)
    if granularity_scale_category_summary_parts
    else pd.DataFrame(columns=["categoria_solape", "n_rows", "granularidad_min"])
)
granularity_scale_metrics_summary = (
    pd.concat(granularity_scale_metrics_summary_parts, ignore_index=True)
    if granularity_scale_metrics_summary_parts
    else pd.DataFrame(columns=["granularidad_min", "metrica", "n_non_null", "n_null", "min", "p50", "p95", "p99", "max"])
)
granularity_scale_signal_summary = pd.DataFrame(granularity_scale_signal_summary_rows)
granularity_scale_checks = pd.DataFrame(granularity_scale_check_rows)

granularity_scale_failed_checks = granularity_scale_checks.loc[
    ~granularity_scale_checks["ok"],
    ["granularidad_min", "check", "ok", "critico", "detalle"],
].copy()

granularity_scale_checks_compact = (
    granularity_scale_checks
    .groupby(["check", "critico", "detalle"], dropna=False)
    .agg(
        ok=("ok", "all"),
        granularidades_validadas=(
            "granularidad_min",
            lambda values: ", ".join(str(value) for value in sorted(values.dropna().unique())),
        ),
        n_granularidades=("granularidad_min", "nunique"),
    )
    .reset_index()
    .loc[lambda df: df["ok"]]
    [["check", "ok", "critico", "granularidades_validadas", "n_granularidades", "detalle"]]
)

display(granularity_scale_input_summary)
display(granularity_scale_expansion_summary)
display(granularity_scale_signal_summary)
display(granularity_scale_checks_compact)
if not granularity_scale_failed_checks.empty:
    display(granularity_scale_failed_checks)
else:
    display(Markdown("No hay checks fallidos por granularidad."))

if granularity_scale_failed_critical:
    raise ValueError(
        "Fallan checks críticos en la comparación técnica de granularidades: "
        f"{granularity_scale_failed_critical}. Revisa `granularity_scale_checks`."
    )

### Lectura de control de granularidades

La comparación técnica de granularidades queda validada sobre la muestra `largest_part_per_period`. La prueba utiliza 13 ficheros, cubre los 13 periodos temporales disponibles y procesa 3.068.748 tiques válidos sobre las cuatro granularidades candidatas: 15, 30, 45 y 60 minutos.

La expansión se ejecuta correctamente para todas las granularidades y se agrega de forma inmediata por barrio, intervalo y granularidad. Como era esperable, la granularidad de 15 minutos genera el mayor volumen intermedio, mientras que 60 minutos produce el menor. En todos los casos se conserva el mismo total de minutos solapados, lo que confirma la coherencia del reparto temporal entre granularidades.

Los checks compactos confirman que no hay fallos por granularidad: los solapes son positivos, no superan la duración del intervalo, pertenecen a ventanas SER observables, se asignan a una única categoría A/B/C/D, enlazan con capacidad barrio-año y generan métricas finitas y coherentes.

## 15. Prueba sobre un periodo completo

Tras validar la comparación de granularidades sobre `largest_part_per_period`, se ejecuta una prueba sobre un periodo completo. Este paso aumenta la escala de ejecución: en lugar de utilizar un único fichero físico por trimestre, se procesan todos los ficheros Parquet de un `periodo_inicio` seleccionado.

La finalidad sigue siendo técnica. Esta sección comprueba si la lectura por bloques, la expansión temporal, la agregación inmediata, el enlace con capacidad y el cálculo de métricas se mantienen estables al procesar un trimestre completo. No se construye todavía el panel final, no se procesan todos los periodos y no se escriben salidas de producción.

El periodo seleccionado para esta prueba es `2025_q3`, porque permite validar un trimestre completo que incluye agosto y, por tanto, contiene ventanas SER reducidas. Si el coste computacional resulta excesivo, el periodo podrá sustituirse por otro trimestre, manteniendo la misma lógica de ejecución.


In [ ]:
FULL_PERIOD_TEST = "2025_q3"
FULL_PERIOD_GRANULARITIES_MIN = GRANULARITIES_MIN

full_period_parts = ticket_parts_inventory.loc[
    ticket_parts_inventory["periodo_inicio"].eq(FULL_PERIOD_TEST)
].copy()
if full_period_parts.empty:
    raise ValueError(f"No hay ficheros Parquet para periodo_inicio={FULL_PERIOD_TEST}.")

full_period_required_columns = [
    "fecha_inicio",
    "fecha_fin",
    "barrio_key",
    "anio",
]

full_period_ticket_parts = []
for _, part in full_period_parts.iterrows():
    part_path = part.get("path_obj", pd.NA)
    if pd.isna(part_path):
        part_path = ROOT / part["path"]
    part_df = read_ticket_part_columns(Path(part_path), full_period_required_columns)
    part_df["source_periodo_inicio"] = part["periodo_inicio"]
    full_period_ticket_parts.append(part_df)

full_period_tickets_raw = pd.concat(full_period_ticket_parts, ignore_index=True)
full_period_tickets = full_period_tickets_raw.copy()
full_period_tickets["fecha_inicio"] = pd.to_datetime(full_period_tickets["fecha_inicio"])
full_period_tickets["fecha_fin"] = pd.to_datetime(full_period_tickets["fecha_fin"])
full_period_tickets = full_period_tickets.loc[
    full_period_tickets["fecha_inicio"].notna()
    & full_period_tickets["fecha_fin"].notna()
    & (full_period_tickets["fecha_fin"] > full_period_tickets["fecha_inicio"])
    & full_period_tickets["barrio_key"].notna()
].copy().reset_index(drop=True)

full_period_input_summary = pd.DataFrame([
    {
        "periodo_inicio": FULL_PERIOD_TEST,
        "n_files": len(full_period_parts),
        "n_rows_leidas": len(full_period_tickets_raw),
        "n_rows_validas": len(full_period_tickets),
        "granularidades": FULL_PERIOD_GRANULARITIES_MIN,
    }
])

full_period_aggregated_parts = []
full_period_metrics_parts = []
full_period_expansion_summary_rows = []
full_period_signal_summary_rows = []
full_period_check_rows = []
full_period_failed_critical = []

full_period_metric_columns = [
    "ocupacion_pagada_proxy",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]


def add_full_period_check(granularidad_min: int, check: str, ok: bool, detalle: str, critico: bool = True):
    full_period_check_rows.append({
        "granularidad_min": granularidad_min,
        "check": check,
        "ok": bool(ok),
        "critico": critico,
        "detalle": detalle,
    })
    if critico and not ok:
        full_period_failed_critical.append((granularidad_min, check))

for granularidad_min in FULL_PERIOD_GRANULARITIES_MIN:
    full_period_expanded_current = expand_tickets_to_intervals_smoke(full_period_tickets, granularidad_min)
    full_period_observable_intervals = time_dimension_candidates.loc[
        time_dimension_candidates["granularidad_min"].eq(granularidad_min),
        ["granularidad_min", "intervalo_inicio", "intervalo_fin", "anio_intervalo"],
    ].drop_duplicates()
    full_period_expanded_current = full_period_expanded_current.drop(columns=["intervalo_fin"]).merge(
        full_period_observable_intervals,
        on=["granularidad_min", "intervalo_inicio"],
        how="inner",
    )

    if full_period_expanded_current.empty:
        full_period_aggregated_current = pd.DataFrame()
        full_period_metrics_current = pd.DataFrame()
    else:
        cond_A = (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_fin"])
        cond_B = (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_fin"] <= full_period_expanded_current["intervalo_fin"])
        cond_C = (full_period_expanded_current["fecha_inicio"] >= full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_fin"]) & (full_period_expanded_current["fecha_fin"] <= full_period_expanded_current["intervalo_fin"])
        cond_D = (full_period_expanded_current["fecha_inicio"] >= full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_fin"]) & (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_fin"])
        full_period_expanded_current["categoria_count"] = cond_A.astype(int) + cond_B.astype(int) + cond_C.astype(int) + cond_D.astype(int)
        full_period_expanded_current["categoria_solape"] = np.select([cond_A, cond_B, cond_C, cond_D], ["A", "B", "C", "D"], default=pd.NA)
        full_period_expanded_current["activo_inicio"] = (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_inicio"])
        full_period_expanded_current["activo_fin"] = (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_fin"]) & (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_fin"])
        full_period_expanded_current["entra_intervalo"] = (full_period_expanded_current["fecha_inicio"] >= full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_fin"])
        full_period_expanded_current["sale_intervalo"] = (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_fin"] <= full_period_expanded_current["intervalo_fin"])
        full_period_expanded_current["persistente_completo"] = (full_period_expanded_current["fecha_inicio"] < full_period_expanded_current["intervalo_inicio"]) & (full_period_expanded_current["fecha_fin"] > full_period_expanded_current["intervalo_fin"])

        for category in ["A", "B", "C", "D"]:
            full_period_expanded_current[f"is_{category}"] = full_period_expanded_current["categoria_solape"].eq(category)

        full_period_aggregated_current = (
            full_period_expanded_current
            .groupby(["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"], dropna=False)
            .agg(
                n_tiques_solapados=("minutos_solapados", "size"),
                minutos_solapados=("minutos_solapados", "sum"),
                n_A=("is_A", "sum"),
                n_B=("is_B", "sum"),
                n_C=("is_C", "sum"),
                n_D=("is_D", "sum"),
                n_activo_inicio=("activo_inicio", "sum"),
                n_activo_fin=("activo_fin", "sum"),
                n_entra_intervalo=("entra_intervalo", "sum"),
                n_sale_intervalo=("sale_intervalo", "sum"),
                n_persistente_completo=("persistente_completo", "sum"),
            )
            .reset_index()
        )
        full_period_aggregated_parts.append(full_period_aggregated_current)

        full_period_metrics_current = full_period_aggregated_current.merge(
            capacity_for_smoke,
            left_on=["anio_intervalo", "barrio_key"],
            right_on=["anio", "barrio_key"],
            how="left",
        )
        valid_capacity_current = full_period_metrics_current["plazas_barrio_anio"] > 0
        full_period_metric_formulas = {
            "ocupacion_pagada_proxy": full_period_metrics_current["minutos_solapados"] / (full_period_metrics_current["plazas_barrio_anio"] * full_period_metrics_current["granularidad_min"]),
            "stock_inicio_proxy": full_period_metrics_current["n_activo_inicio"] / full_period_metrics_current["plazas_barrio_anio"],
            "stock_fin_proxy": full_period_metrics_current["n_activo_fin"] / full_period_metrics_current["plazas_barrio_anio"],
            "afluencia_tiques": full_period_metrics_current["n_entra_intervalo"] / full_period_metrics_current["plazas_barrio_anio"],
            "liberacion_tiques": full_period_metrics_current["n_sale_intervalo"] / full_period_metrics_current["plazas_barrio_anio"],
            "persistencia_completa": full_period_metrics_current["n_persistente_completo"] / full_period_metrics_current["plazas_barrio_anio"],
            "saldo_flujo": (full_period_metrics_current["n_D"] - full_period_metrics_current["n_B"]) / full_period_metrics_current["plazas_barrio_anio"],
            "rotacion_bruta": (full_period_metrics_current["n_B"] + 2 * full_period_metrics_current["n_C"] + full_period_metrics_current["n_D"]) / full_period_metrics_current["plazas_barrio_anio"],
        }
        for metric, values in full_period_metric_formulas.items():
            full_period_metrics_current[metric] = np.where(valid_capacity_current, values, np.nan)
        full_period_metrics_parts.append(full_period_metrics_current)

    full_period_expansion_summary_rows.append({
        "granularidad_min": granularidad_min,
        "n_rows_expandidas": len(full_period_expanded_current),
        "n_rows_agregadas": len(full_period_aggregated_current),
        "minutos_solapados_total": full_period_expanded_current["minutos_solapados"].sum() if not full_period_expanded_current.empty else 0.0,
        "n_barrios": full_period_expanded_current["barrio_key"].nunique(dropna=True) if not full_period_expanded_current.empty else 0,
        "intervalo_min": full_period_expanded_current["intervalo_inicio"].min() if not full_period_expanded_current.empty else pd.NaT,
        "intervalo_max": full_period_expanded_current["intervalo_fin"].max() if not full_period_expanded_current.empty else pd.NaT,
    })

    if not full_period_metrics_current.empty:
        ocupacion_current = full_period_metrics_current["ocupacion_pagada_proxy"]
        saldo_current = full_period_metrics_current["saldo_flujo"]
        n_metric_rows_current = int(ocupacion_current.notna().sum())
        full_period_signal_summary_rows.append({
            "granularidad_min": granularidad_min,
            "n_rows_ocupacion_gt_1": int((ocupacion_current > 1).sum()),
            "pct_rows_ocupacion_gt_1": (ocupacion_current > 1).sum() / n_metric_rows_current if n_metric_rows_current else np.nan,
            "n_rows_ocupacion_eq_0": int((ocupacion_current == 0).sum()),
            "pct_rows_ocupacion_eq_0": (ocupacion_current == 0).sum() / n_metric_rows_current if n_metric_rows_current else np.nan,
            "n_rows_saldo_positivo": int((saldo_current > 0).sum()),
            "n_rows_saldo_negativo": int((saldo_current < 0).sum()),
            "n_rows_saldo_cero": int((saldo_current == 0).sum()),
        })
    else:
        full_period_signal_summary_rows.append({
            "granularidad_min": granularidad_min,
            "n_rows_ocupacion_gt_1": 0,
            "pct_rows_ocupacion_gt_1": np.nan,
            "n_rows_ocupacion_eq_0": 0,
            "pct_rows_ocupacion_eq_0": np.nan,
            "n_rows_saldo_positivo": 0,
            "n_rows_saldo_negativo": 0,
            "n_rows_saldo_cero": 0,
        })

    observable_keys_current = full_period_observable_intervals[["granularidad_min", "intervalo_inicio"]].drop_duplicates()
    observable_check_current = (
        full_period_expanded_current[["granularidad_min", "intervalo_inicio"]]
        .merge(observable_keys_current.assign(_observable=True), on=["granularidad_min", "intervalo_inicio"], how="left")
        ["_observable"]
        .fillna(False)
        .all()
        if not full_period_expanded_current.empty
        else False
    )
    metric_values_current = full_period_metrics_current[full_period_metric_columns] if not full_period_metrics_current.empty else pd.DataFrame(columns=full_period_metric_columns)
    finite_values_current = np.isfinite(metric_values_current.to_numpy(dtype=float)) | metric_values_current.isna().to_numpy()
    non_negative_cols_current = [metric for metric in full_period_metric_columns if metric != "saldo_flujo"]
    non_negative_values_current = full_period_metrics_current[non_negative_cols_current] if not full_period_metrics_current.empty else pd.DataFrame(columns=non_negative_cols_current)

    if not full_period_metrics_current.empty:
        saldo_expected_current = full_period_metrics_current["afluencia_tiques"] - full_period_metrics_current["liberacion_tiques"]
        rotacion_expected_current = full_period_metrics_current["afluencia_tiques"] + full_period_metrics_current["liberacion_tiques"]
        ocupacion_expected_current = full_period_metrics_current["minutos_solapados"] / (full_period_metrics_current["plazas_barrio_anio"] * full_period_metrics_current["granularidad_min"])
        saldo_ok_current = bool(np.nanmax(np.abs(full_period_metrics_current["saldo_flujo"] - saldo_expected_current)) <= 1e-10)
        rotacion_ok_current = bool(np.nanmax(np.abs(full_period_metrics_current["rotacion_bruta"] - rotacion_expected_current)) <= 1e-10)
        ocupacion_ok_current = bool(np.nanmax(np.abs(full_period_metrics_current["ocupacion_pagada_proxy"] - ocupacion_expected_current)) <= 1e-10)
    else:
        saldo_ok_current = False
        rotacion_ok_current = False
        ocupacion_ok_current = False

    add_full_period_check(granularidad_min, "filas_validas_leidas", len(full_period_tickets) > 0, f"filas válidas: {len(full_period_tickets)}")
    add_full_period_check(granularidad_min, "expansion_genero_filas", len(full_period_expanded_current) > 0, f"filas expandidas observables: {len(full_period_expanded_current)}")
    add_full_period_check(granularidad_min, "minutos_solapados_positivos", (full_period_expanded_current["minutos_solapados"] > 0).all() if not full_period_expanded_current.empty else False, "todos los solapes conservados son positivos")
    add_full_period_check(granularidad_min, "minutos_solapados_no_superan_granularidad", (full_period_expanded_current["minutos_solapados"] <= full_period_expanded_current["granularidad_min"]).all() if not full_period_expanded_current.empty else False, "ningún solape supera la duración del intervalo")
    add_full_period_check(granularidad_min, "intervalos_ser_observables", observable_check_current, "todas las filas expandidas pertenecen a time_dimension_candidates")
    add_full_period_check(granularidad_min, "categoria_unica_abcd", (full_period_expanded_current["categoria_count"] == 1).all() if not full_period_expanded_current.empty else False, "cada fila expandida tiene una y solo una categoría A/B/C/D")
    add_full_period_check(granularidad_min, "agregacion_sin_barrio_nulo", full_period_aggregated_current["barrio_key"].notna().all() if not full_period_aggregated_current.empty else False, "la agregación conserva barrio_key no nulo")
    add_full_period_check(granularidad_min, "capacidad_no_nula_en_agregados", full_period_metrics_current["plazas_barrio_anio"].notna().all() if not full_period_metrics_current.empty else False, "todas las filas agregadas enlazan con capacidad barrio-año")
    add_full_period_check(granularidad_min, "metricas_finitas", finite_values_current.all() if not full_period_metrics_current.empty else False, "no hay inf ni -inf en las métricas calculadas")
    add_full_period_check(granularidad_min, "metricas_no_negativas_salvo_saldo", (non_negative_values_current.dropna(how="all") >= 0).all().all() if not full_period_metrics_current.empty else False, "todas las métricas excepto saldo_flujo son >= 0 cuando no son nulas")
    add_full_period_check(granularidad_min, "saldo_equivale_afluencia_menos_liberacion", saldo_ok_current, "saldo_flujo coincide con afluencia_tiques - liberacion_tiques")
    add_full_period_check(granularidad_min, "rotacion_equivale_afluencia_mas_liberacion", rotacion_ok_current, "rotacion_bruta coincide con afluencia_tiques + liberacion_tiques")
    add_full_period_check(granularidad_min, "ocupacion_usa_minutos_exactos", ocupacion_ok_current, "ocupacion_pagada_proxy coincide con minutos solapados / capacidad-tiempo")

full_period_expansion_summary = pd.DataFrame(full_period_expansion_summary_rows)
full_period_signal_summary = pd.DataFrame(full_period_signal_summary_rows)
full_period_checks = pd.DataFrame(full_period_check_rows)
full_period_failed_checks = full_period_checks.loc[
    ~full_period_checks["ok"],
    ["granularidad_min", "check", "ok", "critico", "detalle"],
].copy()
full_period_checks_compact = (
    full_period_checks
    .groupby(["check", "critico", "detalle"], dropna=False)
    .agg(
        ok=("ok", "all"),
        granularidades_validadas=(
            "granularidad_min",
            lambda values: ", ".join(str(value) for value in sorted(values.dropna().unique())),
        ),
        n_granularidades=("granularidad_min", "nunique"),
    )
    .reset_index()
    .loc[lambda df: df["ok"]]
    [["check", "ok", "critico", "granularidades_validadas", "n_granularidades", "detalle"]]
)

display(full_period_input_summary)
display(full_period_expansion_summary)
display(full_period_signal_summary)
display(full_period_checks_compact)
if not full_period_failed_checks.empty:
    display(full_period_failed_checks)
else:
    display(Markdown("No hay checks fallidos en la prueba de periodo completo."))

if full_period_failed_critical:
    raise ValueError(
        "Fallan checks críticos en la prueba de periodo completo: "
        f"{full_period_failed_critical}. Revisa `full_period_checks`."
    )

### Lectura de control del periodo completo

La prueba sobre el periodo completo `2025_q3` queda validada. Se procesan 61 ficheros Parquet del trimestre, con 9.857.726 tiques válidos, y se ejecuta el cálculo para las cuatro granularidades candidatas: 15, 30, 45 y 60 minutos.

La expansión, agregación inmediata, unión con capacidad y cálculo de métricas se completan sin fallos críticos. La prueba confirma que el procedimiento escala desde una muestra de ficheros seleccionados hasta un trimestre completo, manteniendo la lógica de solapes y los controles de coherencia ya validados en apartados anteriores.

Los checks compactos confirman que las cuatro granularidades superan las comprobaciones críticas: lectura válida, expansión no vacía, minutos solapados correctos, intervalos observables, categoría A/B/C/D única, agregación sin `barrio_key` nulo, capacidad disponible y métricas finitas.

## 16. Construcción completa del panel global SER barrio-intervalo

Tras validar el procedimiento en una muestra reducida, en `largest_part_per_period` y en un periodo completo, se activa la construcción completa del panel global `SER_barrio_intervalo`. Esta sección procesa todos los ficheros disponibles de `ser_tiques_barrio_base`, manteniendo las cuatro granularidades candidatas: 15, 30, 45 y 60 minutos.

La ejecución completa no carga todo el dataset simultáneamente. El procesamiento se organiza por `periodo_inicio` y granularidad: para cada bloque se leen los tiques necesarios, se calculan solapes con intervalos SER observables, se agregan inmediatamente por barrio-intervalo, se enlaza con capacidad barrio-año y se calculan las métricas proxy. La tabla expandida tique-intervalo sigue siendo un objeto intermedio temporal y no se conserva como salida.

A diferencia de las pruebas anteriores, esta sección incorpora la malla común barrio-intervalo. Por tanto, el panel resultante no contiene únicamente intervalos con tiques, sino también intervalos SER observables sin señal pagada. En esos casos, cuando exista capacidad válida, los conteos y minutos se fijan a cero y las métricas derivadas representan ausencia de señal pagada observada, no facilidad real para aparcar.

La salida se escribe de forma particionada para evitar una tabla monolítica y para facilitar posteriores diagnósticos de granularidad, estabilidad y target. Esta sección construye candidatos globales de señal pagada SER; no crea todavía clases de dificultad, `prob_aparcar_proxy` ni un modelo predictivo.


In [ ]:
RUN_FULL_DATASET_PANEL = False
WRITE_FULL_DATASET_PANEL = False
FULL_DATASET_GRANULARITIES_MIN = GRANULARITIES_MIN
FULL_DATASET_OUTPUT_DIR = OUTPUT_GLOBAL_CANDIDATES_DIR
FULL_DATASET_OVERWRITE_OUTPUTS = False

full_dataset_execution_rows = []
full_dataset_occupancy_rows = []
full_dataset_check_rows = []
full_dataset_failed_critical = []

full_dataset_count_columns = [
    "n_tiques_solapados",
    "minutos_solapados",
    "n_A",
    "n_B",
    "n_C",
    "n_D",
    "n_activo_inicio",
    "n_activo_fin",
    "n_entra_intervalo",
    "n_sale_intervalo",
    "n_persistente_completo",
]
full_dataset_read_columns = [
    "fecha_inicio",
    "fecha_fin",
    "barrio_key",
    "anio",
]

full_dataset_metric_columns = [
    "ocupacion_pagada_proxy",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]


def add_full_dataset_check(periodo_inicio: str, granularidad_min: int, check: str, ok: bool, detalle: str, critico: bool = True):
    full_dataset_check_rows.append({
        "periodo_inicio": periodo_inicio,
        "granularidad_min": granularidad_min,
        "check": check,
        "ok": bool(ok),
        "critico": critico,
        "detalle": detalle,
    })
    if critico and not ok:
        full_dataset_failed_critical.append((periodo_inicio, granularidad_min, check))


def parse_periodo_inicio_bounds(periodo_inicio: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    match = re.fullmatch(r"(\d{4})_q([1-4])", str(periodo_inicio))
    if not match:
        raise ValueError(f"periodo_inicio no reconocido: {periodo_inicio}")
    year = int(match.group(1))
    quarter = int(match.group(2))
    start_month = 3 * (quarter - 1) + 1
    start = pd.Timestamp(year=year, month=start_month, day=1)
    end_exclusive = start + pd.DateOffset(months=3)
    return start, end_exclusive


def classify_and_aggregate_expanded(expanded_df: pd.DataFrame) -> pd.DataFrame:
    if expanded_df.empty:
        return pd.DataFrame(columns=[
            "granularidad_min",
            "barrio_key",
            "intervalo_inicio",
            "intervalo_fin",
            "anio_intervalo",
            *full_dataset_count_columns,
        ])

    cond_A = (expanded_df["fecha_inicio"] < expanded_df["intervalo_inicio"]) & (expanded_df["fecha_fin"] > expanded_df["intervalo_fin"])
    cond_B = (expanded_df["fecha_inicio"] < expanded_df["intervalo_inicio"]) & (expanded_df["fecha_fin"] > expanded_df["intervalo_inicio"]) & (expanded_df["fecha_fin"] <= expanded_df["intervalo_fin"])
    cond_C = (expanded_df["fecha_inicio"] >= expanded_df["intervalo_inicio"]) & (expanded_df["fecha_inicio"] < expanded_df["intervalo_fin"]) & (expanded_df["fecha_fin"] <= expanded_df["intervalo_fin"])
    cond_D = (expanded_df["fecha_inicio"] >= expanded_df["intervalo_inicio"]) & (expanded_df["fecha_inicio"] < expanded_df["intervalo_fin"]) & (expanded_df["fecha_fin"] > expanded_df["intervalo_fin"])

    expanded_df["categoria_count"] = cond_A.astype(int) + cond_B.astype(int) + cond_C.astype(int) + cond_D.astype(int)
    expanded_df["categoria_solape"] = np.select([cond_A, cond_B, cond_C, cond_D], ["A", "B", "C", "D"], default=pd.NA)
    expanded_df["activo_inicio"] = (expanded_df["fecha_inicio"] < expanded_df["intervalo_inicio"]) & (expanded_df["fecha_fin"] > expanded_df["intervalo_inicio"])
    expanded_df["activo_fin"] = (expanded_df["fecha_inicio"] < expanded_df["intervalo_fin"]) & (expanded_df["fecha_fin"] > expanded_df["intervalo_fin"])
    expanded_df["entra_intervalo"] = (expanded_df["fecha_inicio"] >= expanded_df["intervalo_inicio"]) & (expanded_df["fecha_inicio"] < expanded_df["intervalo_fin"])
    expanded_df["sale_intervalo"] = (expanded_df["fecha_fin"] > expanded_df["intervalo_inicio"]) & (expanded_df["fecha_fin"] <= expanded_df["intervalo_fin"])
    expanded_df["persistente_completo"] = (expanded_df["fecha_inicio"] < expanded_df["intervalo_inicio"]) & (expanded_df["fecha_fin"] > expanded_df["intervalo_fin"])

    for category in ["A", "B", "C", "D"]:
        expanded_df[f"is_{category}"] = expanded_df["categoria_solape"].eq(category)

    return (
        expanded_df
        .groupby(["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"], dropna=False)
        .agg(
            n_tiques_solapados=("minutos_solapados", "size"),
            minutos_solapados=("minutos_solapados", "sum"),
            n_A=("is_A", "sum"),
            n_B=("is_B", "sum"),
            n_C=("is_C", "sum"),
            n_D=("is_D", "sum"),
            n_activo_inicio=("activo_inicio", "sum"),
            n_activo_fin=("activo_fin", "sum"),
            n_entra_intervalo=("entra_intervalo", "sum"),
            n_sale_intervalo=("sale_intervalo", "sum"),
            n_persistente_completo=("persistente_completo", "sum"),
        )
        .reset_index()
    )


def add_global_metrics(panel_df: pd.DataFrame) -> pd.DataFrame:
    valid_capacity = panel_df["plazas_barrio_anio"] > 0
    metric_formulas = {
        "ocupacion_pagada_proxy": panel_df["minutos_solapados"] / (panel_df["plazas_barrio_anio"] * panel_df["granularidad_min"]),
        "stock_inicio_proxy": panel_df["n_activo_inicio"] / panel_df["plazas_barrio_anio"],
        "stock_fin_proxy": panel_df["n_activo_fin"] / panel_df["plazas_barrio_anio"],
        "afluencia_tiques": panel_df["n_entra_intervalo"] / panel_df["plazas_barrio_anio"],
        "liberacion_tiques": panel_df["n_sale_intervalo"] / panel_df["plazas_barrio_anio"],
        "persistencia_completa": panel_df["n_persistente_completo"] / panel_df["plazas_barrio_anio"],
        "saldo_flujo": (panel_df["n_D"] - panel_df["n_B"]) / panel_df["plazas_barrio_anio"],
        "rotacion_bruta": (panel_df["n_B"] + 2 * panel_df["n_C"] + panel_df["n_D"]) / panel_df["plazas_barrio_anio"],
    }
    for metric, values in metric_formulas.items():
        panel_df[metric] = np.where(valid_capacity, values, np.nan)
    return panel_df

if not RUN_FULL_DATASET_PANEL:
    display(Markdown("La ejecución completa del panel global está desactivada (`RUN_FULL_DATASET_PANEL=False`)."))
else:
    import shutil

    if WRITE_FULL_DATASET_PANEL:
        output_has_files = FULL_DATASET_OUTPUT_DIR.exists() and any(path.is_file() for path in FULL_DATASET_OUTPUT_DIR.rglob("*"))
        if output_has_files and not FULL_DATASET_OVERWRITE_OUTPUTS:
            raise FileExistsError(
                f"La salida {relpath(FULL_DATASET_OUTPUT_DIR)} ya existe y contiene ficheros. "
                "Activa FULL_DATASET_OVERWRITE_OUTPUTS=True solo si quieres reemplazar esa salida."
            )
        if output_has_files and FULL_DATASET_OVERWRITE_OUTPUTS:
            shutil.rmtree(FULL_DATASET_OUTPUT_DIR)
        FULL_DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    full_dataset_periods = sorted(ticket_parts_inventory["periodo_inicio"].dropna().unique().tolist())
    capacity_barrios_by_year = (
        capacity_for_smoke[["anio", "barrio_key"]]
        .drop_duplicates()
        .rename(columns={"anio": "anio_intervalo"})
    )

    for periodo_inicio in full_dataset_periods:
        period_start, period_end_exclusive = parse_periodo_inicio_bounds(periodo_inicio)
        period_parts = ticket_parts_inventory.loc[
            ticket_parts_inventory["periodo_inicio"].eq(periodo_inicio)
        ].copy()

        period_ticket_parts = []
        for _, part in period_parts.iterrows():
            part_path = part.get("path_obj", pd.NA)
            if pd.isna(part_path):
                part_path = ROOT / part["path"]
            part_df = read_ticket_part_columns(Path(part_path), full_dataset_read_columns)
            part_df["source_periodo_inicio"] = periodo_inicio
            period_ticket_parts.append(part_df)

        period_tickets_raw = pd.concat(period_ticket_parts, ignore_index=True) if period_ticket_parts else pd.DataFrame(columns=full_dataset_read_columns + ["source_periodo_inicio"])
        period_tickets = period_tickets_raw.copy()
        period_tickets["fecha_inicio"] = pd.to_datetime(period_tickets["fecha_inicio"])
        period_tickets["fecha_fin"] = pd.to_datetime(period_tickets["fecha_fin"])
        period_tickets = period_tickets.loc[
            period_tickets["fecha_inicio"].notna()
            & period_tickets["fecha_fin"].notna()
            & (period_tickets["fecha_fin"] > period_tickets["fecha_inicio"])
            & period_tickets["barrio_key"].notna()
        ].copy().reset_index(drop=True)

        for granularidad_min in FULL_DATASET_GRANULARITIES_MIN:
            observed_intervals = time_dimension_candidates.loc[
                time_dimension_candidates["granularidad_min"].eq(granularidad_min)
                & (time_dimension_candidates["intervalo_inicio"] >= period_start)
                & (time_dimension_candidates["intervalo_inicio"] < period_end_exclusive),
                ["granularidad_min", "intervalo_inicio", "intervalo_fin", "anio_intervalo"],
            ].drop_duplicates()

            expanded_current = expand_tickets_to_intervals_smoke(period_tickets, granularidad_min)
            expanded_current = expanded_current.drop(columns=["intervalo_fin"]).merge(
                observed_intervals,
                on=["granularidad_min", "intervalo_inicio"],
                how="inner",
            )

            n_rows_expandidas = len(expanded_current)
            minutos_solapados_total = expanded_current["minutos_solapados"].sum() if not expanded_current.empty else 0.0
            intervalo_min = expanded_current["intervalo_inicio"].min() if not expanded_current.empty else pd.NaT
            intervalo_max = expanded_current["intervalo_fin"].max() if not expanded_current.empty else pd.NaT
            expanded_barrio_count = expanded_current["barrio_key"].nunique(dropna=True) if not expanded_current.empty else 0

            aggregated_observed = classify_and_aggregate_expanded(expanded_current)
            n_rows_agregadas_observadas = len(aggregated_observed)

            full_dataset_grid_period_granularity = observed_intervals.merge(
                capacity_barrios_by_year,
                on="anio_intervalo",
                how="inner",
            )[["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"]]

            panel_current = full_dataset_grid_period_granularity.merge(
                aggregated_observed,
                on=["granularidad_min", "barrio_key", "intervalo_inicio", "intervalo_fin", "anio_intervalo"],
                how="left",
            )
            for column in full_dataset_count_columns:
                panel_current[column] = panel_current[column].fillna(0)
            panel_current = panel_current.merge(
                capacity_for_smoke,
                left_on=["anio_intervalo", "barrio_key"],
                right_on=["anio", "barrio_key"],
                how="left",
            )
            panel_current = add_global_metrics(panel_current)
            panel_current["periodo_inicio"] = periodo_inicio

            output_path = pd.NA
            if WRITE_FULL_DATASET_PANEL:
                output_dir = FULL_DATASET_OUTPUT_DIR / f"granularidad_min={granularidad_min}" / f"periodo_inicio={periodo_inicio}"
                output_dir.mkdir(parents=True, exist_ok=True)
                output_file = output_dir / "part.parquet"
                panel_current.to_parquet(output_file, index=False)
                output_path = relpath(output_file)

            metric_values = panel_current[full_dataset_metric_columns]
            finite_values = np.isfinite(metric_values.to_numpy(dtype=float)) | metric_values.isna().to_numpy()
            non_negative_values = panel_current[[column for column in full_dataset_metric_columns if column != "saldo_flujo"]]
            saldo_expected = panel_current["afluencia_tiques"] - panel_current["liberacion_tiques"]
            rotacion_expected = panel_current["afluencia_tiques"] + panel_current["liberacion_tiques"]
            ocupacion_expected = panel_current["minutos_solapados"] / (panel_current["plazas_barrio_anio"] * panel_current["granularidad_min"])

            observable_check = (
                expanded_current[["granularidad_min", "intervalo_inicio"]]
                .merge(observed_intervals[["granularidad_min", "intervalo_inicio"]].assign(_observable=True), on=["granularidad_min", "intervalo_inicio"], how="left")
                ["_observable"]
                .fillna(False)
                .all()
                if not expanded_current.empty
                else False
            )
            output_written_ok = bool((not WRITE_FULL_DATASET_PANEL) or (not pd.isna(output_path) and (FULL_DATASET_OUTPUT_DIR / f"granularidad_min={granularidad_min}" / f"periodo_inicio={periodo_inicio}" / "part.parquet").exists()))

            add_full_dataset_check(periodo_inicio, granularidad_min, "filas_validas_leidas", len(period_tickets) > 0, f"filas válidas: {len(period_tickets)}")
            add_full_dataset_check(periodo_inicio, granularidad_min, "expansion_genero_filas", len(expanded_current) > 0, f"filas expandidas observables: {len(expanded_current)}")
            add_full_dataset_check(periodo_inicio, granularidad_min, "minutos_solapados_positivos", (expanded_current["minutos_solapados"] > 0).all() if not expanded_current.empty else False, "todos los solapes conservados son positivos")
            add_full_dataset_check(periodo_inicio, granularidad_min, "minutos_solapados_no_superan_granularidad", (expanded_current["minutos_solapados"] <= expanded_current["granularidad_min"]).all() if not expanded_current.empty else False, "ningún solape supera la duración del intervalo")
            add_full_dataset_check(periodo_inicio, granularidad_min, "intervalos_ser_observables", observable_check, "todas las filas expandidas pertenecen a la dimensión temporal observable")
            add_full_dataset_check(periodo_inicio, granularidad_min, "categoria_unica_abcd", (expanded_current["categoria_count"] == 1).all() if not expanded_current.empty else False, "cada fila expandida tiene una y solo una categoría A/B/C/D")
            add_full_dataset_check(periodo_inicio, granularidad_min, "agregacion_sin_barrio_nulo", aggregated_observed["barrio_key"].notna().all() if not aggregated_observed.empty else False, "la agregación observada conserva barrio_key no nulo")
            add_full_dataset_check(periodo_inicio, granularidad_min, "panel_sin_barrio_nulo", panel_current["barrio_key"].notna().all(), "el panel con malla común conserva barrio_key no nulo")
            add_full_dataset_check(periodo_inicio, granularidad_min, "capacidad_no_nula_en_panel", panel_current["plazas_barrio_anio"].notna().all(), "todas las filas del panel enlazan con capacidad barrio-año")
            add_full_dataset_check(periodo_inicio, granularidad_min, "metricas_finitas", finite_values.all(), "no hay inf ni -inf en las métricas calculadas")
            add_full_dataset_check(periodo_inicio, granularidad_min, "metricas_no_negativas_salvo_saldo", (non_negative_values.dropna(how="all") >= 0).all().all(), "todas las métricas excepto saldo_flujo son >= 0 cuando no son nulas")
            add_full_dataset_check(periodo_inicio, granularidad_min, "saldo_equivale_afluencia_menos_liberacion", bool(np.nanmax(np.abs(panel_current["saldo_flujo"] - saldo_expected)) <= 1e-10), "saldo_flujo coincide con afluencia_tiques - liberacion_tiques")
            add_full_dataset_check(periodo_inicio, granularidad_min, "rotacion_equivale_afluencia_mas_liberacion", bool(np.nanmax(np.abs(panel_current["rotacion_bruta"] - rotacion_expected)) <= 1e-10), "rotacion_bruta coincide con afluencia_tiques + liberacion_tiques")
            add_full_dataset_check(periodo_inicio, granularidad_min, "ocupacion_usa_minutos_exactos", bool(np.nanmax(np.abs(panel_current["ocupacion_pagada_proxy"] - ocupacion_expected)) <= 1e-10), "ocupacion_pagada_proxy coincide con minutos solapados / capacidad-tiempo")
            add_full_dataset_check(periodo_inicio, granularidad_min, "panel_contiene_malla_comun", len(panel_current) == len(full_dataset_grid_period_granularity), "el panel conserva todas las filas de la malla común barrio-intervalo")
            add_full_dataset_check(periodo_inicio, granularidad_min, "output_escrito_si_corresponde", output_written_ok, "la salida se escribe si WRITE_FULL_DATASET_PANEL=True")

            full_dataset_execution_rows.append({
                "periodo_inicio": periodo_inicio,
                "granularidad_min": granularidad_min,
                "n_files": len(period_parts),
                "n_rows_leidas": len(period_tickets_raw),
                "n_rows_validas": len(period_tickets),
                "n_rows_expandidas": n_rows_expandidas,
                "n_rows_agregadas_observadas": n_rows_agregadas_observadas,
                "n_rows_panel": len(panel_current),
                "n_rows_sin_senal_pagada": int((panel_current["n_tiques_solapados"] == 0).sum()),
                "minutos_solapados_total": minutos_solapados_total,
                "n_barrios": expanded_barrio_count,
                "intervalo_min": intervalo_min,
                "intervalo_max": intervalo_max,
                "output_path": output_path,
            })
            full_dataset_occupancy_rows.append({
                "granularidad_min": granularidad_min,
                "ocupacion_gt_1_total": int((panel_current["ocupacion_pagada_proxy"] > 1).sum()),
                "ocupacion_eq_0_total": int((panel_current["ocupacion_pagada_proxy"] == 0).sum()),
            })

            del expanded_current, aggregated_observed, full_dataset_grid_period_granularity, panel_current

        del period_tickets_raw, period_tickets, period_ticket_parts

    full_dataset_execution_summary = pd.DataFrame(full_dataset_execution_rows)
    full_dataset_summary_by_granularity = (
        full_dataset_execution_summary
        .groupby("granularidad_min", dropna=False)
        .agg(
            n_periodos=("periodo_inicio", "nunique"),
            n_files_total=("n_files", "sum"),
            n_rows_validas_total=("n_rows_validas", "sum"),
            n_rows_expandidas_total=("n_rows_expandidas", "sum"),
            n_rows_panel_total=("n_rows_panel", "sum"),
            n_rows_sin_senal_pagada_total=("n_rows_sin_senal_pagada", "sum"),
            minutos_solapados_total=("minutos_solapados_total", "sum"),
        )
        .reset_index()
    )
    occupancy_summary = (
        pd.DataFrame(full_dataset_occupancy_rows)
        .groupby("granularidad_min", as_index=False)
        .sum()
    )
    full_dataset_summary_by_granularity = full_dataset_summary_by_granularity.merge(
        occupancy_summary,
        on="granularidad_min",
        how="left",
    )

    full_dataset_checks = pd.DataFrame(full_dataset_check_rows)
    full_dataset_failed_checks = full_dataset_checks.loc[
        ~full_dataset_checks["ok"],
        ["periodo_inicio", "granularidad_min", "check", "ok", "critico", "detalle"],
    ].copy()
    full_dataset_checks_compact = (
        full_dataset_checks
        .groupby(["check", "critico", "detalle"], dropna=False)
        .agg(
            ok=("ok", "all"),
            periodos_validados=("periodo_inicio", lambda values: ", ".join(str(value) for value in sorted(values.dropna().unique()))),
            granularidades_validadas=("granularidad_min", lambda values: ", ".join(str(value) for value in sorted(values.dropna().unique()))),
            n_periodos=("periodo_inicio", "nunique"),
            n_granularidades=("granularidad_min", "nunique"),
        )
        .reset_index()
        .loc[lambda df: df["ok"]]
    )

    display(full_dataset_execution_summary)
    display(full_dataset_summary_by_granularity)
    display(full_dataset_checks_compact)
    if not full_dataset_failed_checks.empty:
        display(full_dataset_failed_checks)
    else:
        display(Markdown("No hay checks fallidos en la construcción completa del panel global."))

    if full_dataset_failed_critical:
        raise ValueError(
            "Fallan checks críticos en la construcción completa del panel global: "
            f"{full_dataset_failed_critical}. Revisa `full_dataset_checks`."
        )

### Lectura de control de la construcción completa del panel global

La construcción completa del panel global `SER_barrio_intervalo` queda validada. La ejecución procesa los 13 periodos disponibles entre `2023_q1` y `2026_q1`, cubriendo los 1002 ficheros Parquet de `ser_tiques_barrio_base` y las cuatro granularidades candidatas: 15, 30, 45 y 60 minutos.

El procedimiento mantiene la estrategia incremental definida en las secciones anteriores: lectura por bloque temporal, expansión temporal controlada, filtrado a intervalos SER observables, agregación inmediata por barrio-intervalo, enlace con capacidad barrio-año, cálculo de métricas proxy y escritura particionada. La tabla expandida tique-intervalo no se conserva como salida persistente.

Los tamaños finales del panel coinciden con la malla común estimada previamente: 2.584.296 filas para 15 minutos, 1.292.148 para 30 minutos, 861.432 para 45 minutos y 646.074 para 60 minutos. Esta coincidencia confirma que la salida no contiene únicamente intervalos con tiques, sino todos los intervalos SER observables por barrio y granularidad dentro del horizonte efectivo del panel.

La suma de minutos solapados se mantiene constante entre granularidades, lo que confirma que el reparto temporal de los tiques no altera el volumen total de señal pagada observada. La diferencia entre granularidades afecta al número de filas expandidas y al nivel de detalle temporal, no a la cantidad total de minutos pagados asignados al panel.

Los checks críticos se superan para todos los periodos y granularidades: hay filas válidas, la expansión genera solapes observables, los minutos solapados son positivos y no superan la duración del intervalo, cada solape queda asignado a una única categoría A/B/C/D, el panel conserva la malla común, las filas enlazan con capacidad barrio-año y las métricas calculadas son finitas y coherentes algebraicamente.

La presencia de filas con `ocupacion_pagada_proxy = 0` confirma que la malla común incorpora intervalos observables sin señal pagada. Estos casos deben interpretarse como ausencia de tiques pagados observados en el intervalo, no como facilidad real para aparcar. El panel sigue midiendo señal pagada SER, no ocupación real total de las plazas.

Con esta sección queda construido el candidato global completo del panel SER a escala barrio-intervalo. El siguiente paso metodológico no es entrenar todavía un modelo, sino diagnosticar las cuatro granularidades, seleccionar una granularidad final y fijar el target histórico global que alimentará el dataset de modelado posterior.


## 17. Diagnóstico de granularidad y selección de granularidad candidata final

Tras construir el panel global completo `SER_barrio_intervalo` para las cuatro granularidades candidatas, esta sección evalúa qué granularidad resulta más adecuada para construir el panel final `ser_barrio_intervalo_global_final`.

La métrica central para seleccionar la granularidad es `ocupacion_pagada_proxy`, porque constituye el target histórico candidato del panel global barrio-intervalo. Las demás métricas —stock, afluencia, liberación, persistencia, saldo y rotación— se conservan como métricas diagnósticas del panel, pero no se utilizan como criterio principal para fijar la unidad temporal del target.

El objetivo no es elegir automáticamente la granularidad con mayor variación, sino identificar una unidad temporal suficientemente fina para capturar cambios reales de la señal pagada observada, sin introducir resolución innecesaria, ruido o coste computacional excesivo. Una granularidad demasiado fina puede duplicar el tamaño del panel sin aportar información sustantiva adicional; una granularidad demasiado gruesa puede suavizar cambios intra-horarios relevantes para el posterior análisis y modelado.

Para cada granularidad $\Delta$ y barrio $b$, se calcula la variación temporal media del target entre intervalos consecutivos comparables:

$$
V_{\Delta,b}
=
\frac{1}{N^{*}_{\Delta,b}}
\sum_{k \in \mathcal{C}_{\Delta,b}}
\left|
y_{\Delta,b,k} - y_{\Delta,b,k-1}
\right|
$$

donde:

- $y_{\Delta,b,k}$ representa el valor de `ocupacion_pagada_proxy` en el intervalo $k$.
- $\mathcal{C}_{\Delta,b}$ es el conjunto de pares de intervalos consecutivos reales dentro de la misma ventana SER observable.
- $N^{*}_{\Delta,b}$ es el número total de pares comparables incluidos en $\mathcal{C}_{\Delta,b}$.

Operativamente, dos intervalos solo se consideran comparables si el cierre del intervalo anterior coincide exactamente con el inicio del intervalo actual:

$$
\texttt{prev\_intervalo\_fin}
=
\texttt{intervalo\_inicio}
$$

Esta restricción evita comparar el último intervalo de un día con el primero del día siguiente, así como saltos producidos por noches, domingos, festivos o periodos sin servicio SER. Esos saltos pueden ser relevantes en otros análisis descriptivos, pero no son adecuados para determinar la granularidad temporal del target.

Además de la variación contigua, se evalúa la ganancia marginal de resolución. Para ello se toma la granularidad de 15 minutos como referencia interna y se mide cuánta variación existe dentro de bloques de 30, 45 y 60 minutos. Si los subintervalos de 15 minutos dentro de un bloque más amplio presentan valores muy similares de `ocupacion_pagada_proxy`, la granularidad de 15 minutos aporta poca señal adicional frente a ese bloque. Si, por el contrario, la variación interna es elevada, una agregación más gruesa podría estar ocultando cambios relevantes.

La comparación de granularidades se organiza en tres niveles:

1. **Control mínimo del panel candidato.**  
   Se verifica que las cuatro granularidades están presentes, que conservan el mismo horizonte temporal, que no aparecen valores de `ocupacion_pagada_proxy` superiores a 1 y que el volumen relativo de filas es el esperado.

2. **Señal temporal útil.**  
   Se mide la variación contigua de `ocupacion_pagada_proxy` dentro de cada barrio y la variación interna de los subintervalos de 15 minutos dentro de bloques de 30, 45 y 60 minutos. Este bloque es el criterio principal para valorar si una granularidad añade resolución útil o solo aumenta el volumen del panel.

3. **Coste e interpretabilidad.**  
   Se compara el tamaño relativo de cada granularidad y su facilidad de interpretación posterior en memoria, mapas y modelado. La granularidad seleccionada debe ser defendible tanto técnica como narrativamente.

La decisión final deberá equilibrar detalle temporal y robustez. La granularidad seleccionada será aquella que conserve variación temporal suficiente para representar cambios en la señal pagada observada, sin generar un panel innecesariamente grande ni una lectura excesivamente ruidosa. El resultado de esta sección será una granularidad candidata final para construir `ser_barrio_intervalo_global_final`.


In [25]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

if "ROOT" not in globals():
    def find_repo_root_for_granularity_selection(start: Path | None = None) -> Path:
        current = Path.cwd().resolve() if start is None else Path(start).resolve()
        for candidate in [current, *current.parents]:
            if (candidate / "data_catalog.csv").exists():
                return candidate
        raise FileNotFoundError("No se ha encontrado data_catalog.csv al ascender desde el directorio actual.")

    ROOT = find_repo_root_for_granularity_selection()

if "OUTPUT_GLOBAL_CANDIDATES_DIR" not in globals():
    OUTPUT_GLOBAL_CANDIDATES_DIR = ROOT / "data/processed/core/ser/ser_barrio_intervalo_global_candidates"

if not OUTPUT_GLOBAL_CANDIDATES_DIR.exists():
    raise FileNotFoundError(f"No existe OUTPUT_GLOBAL_CANDIDATES_DIR: {OUTPUT_GLOBAL_CANDIDATES_DIR}")

candidate_path_pattern = str(OUTPUT_GLOBAL_CANDIDATES_DIR / "**/*.parquet")
candidate_parquet_files = sorted(OUTPUT_GLOBAL_CANDIDATES_DIR.glob("**/*.parquet"))
if not candidate_parquet_files:
    raise FileNotFoundError(f"No hay ficheros Parquet bajo {OUTPUT_GLOBAL_CANDIDATES_DIR}")

candidate_path_pattern_sql = candidate_path_pattern.replace("'", "''")
read_candidates_sql = f"read_parquet('{candidate_path_pattern_sql}', hive_partitioning=false)"
con = duckdb.connect(database=":memory:")

required_selection_columns = [
    "granularidad_min",
    "periodo_inicio",
    "barrio_key",
    "intervalo_inicio",
    "intervalo_fin",
    "minutos_solapados",
    "plazas_barrio_anio",
    "ocupacion_pagada_proxy",
]
source_columns = con.sql(f"select * from {read_candidates_sql} limit 0").df().columns.tolist()
missing_selection_columns = [column for column in required_selection_columns if column not in source_columns]
if missing_selection_columns:
    raise ValueError(f"Faltan columnas mínimas en los candidatos globales: {missing_selection_columns}")

granularity_core_control_summary = con.sql(f"""
    select
        granularidad_min,
        count(*) as n_rows,
        count(distinct periodo_inicio) as n_periodos,
        count(distinct barrio_key) as n_barrios,
        min(intervalo_inicio) as intervalo_min,
        max(intervalo_fin) as intervalo_max,
        sum(minutos_solapados) as minutos_solapados_total,
        avg(case when ocupacion_pagada_proxy = 0 then 1.0 else 0.0 end) as pct_ocupacion_eq_0,
        sum(case when ocupacion_pagada_proxy > 1 then 1 else 0 end) as n_ocupacion_gt_1,
        sum(case when plazas_barrio_anio is null then 1 else 0 end) as n_capacidad_nula
    from {read_candidates_sql}
    group by granularidad_min
    order by granularidad_min
""").df()
rows_by_granularity = granularity_core_control_summary.set_index("granularidad_min")["n_rows"]
n_rows_30 = rows_by_granularity.get(30, np.nan)
n_rows_60 = rows_by_granularity.get(60, np.nan)
granularity_core_control_summary["relative_rows_vs_30"] = granularity_core_control_summary["n_rows"] / n_rows_30
granularity_core_control_summary["relative_rows_vs_60"] = granularity_core_control_summary["n_rows"] / n_rows_60

expected_granularities = {15, 30, 45, 60}
present_granularities = set(granularity_core_control_summary["granularidad_min"].astype(int).tolist())
periods_by_granularity = granularity_core_control_summary.set_index("granularidad_min")["n_periodos"]
horizon_by_granularity = granularity_core_control_summary.set_index("granularidad_min")[["intervalo_min", "intervalo_max"]]
minutes_total = granularity_core_control_summary.set_index("granularidad_min")["minutos_solapados_total"]
minutes_constant = bool(np.isclose(minutes_total.max(), minutes_total.min(), rtol=1e-9, atol=1e-6)) if not minutes_total.empty else False

granularity_core_checks = pd.DataFrame([
    {
        "check": "todas_las_granularidades_presentes",
        "ok": present_granularities == expected_granularities,
        "detalle": f"presentes: {sorted(present_granularities)}",
    },
    {
        "check": "n_periodos_13_en_todas",
        "ok": bool((periods_by_granularity == 13).all()),
        "detalle": periods_by_granularity.to_dict(),
    },
    {
        "check": "horizonte_temporal_identico",
        "ok": bool(horizon_by_granularity.drop_duplicates().shape[0] == 1),
        "detalle": "intervalo_min e intervalo_max son iguales entre granularidades",
    },
    {
        "check": "minutos_totales_constantes",
        "ok": minutes_constant,
        "detalle": "los minutos solapados totales se conservan entre granularidades",
    },
    {
        "check": "sin_ocupacion_gt_1",
        "ok": bool(granularity_core_control_summary["n_ocupacion_gt_1"].sum() == 0),
        "detalle": "no hay filas con ocupacion_pagada_proxy > 1",
    },
    {
        "check": "sin_capacidad_nula",
        "ok": bool(granularity_core_control_summary["n_capacidad_nula"].sum() == 0),
        "detalle": "no hay capacidad nula en el panel candidato",
    },
    {
        "check": "controles_no_vacios",
        "ok": bool(not granularity_core_control_summary.empty),
        "detalle": "granularity_core_control_summary contiene filas",
    },
])
granularity_core_failed_checks = granularity_core_checks.loc[~granularity_core_checks["ok"]].copy()

granularity_contiguous_variability_by_barrio = con.sql(f"""
    with ordered as (
        select
            granularidad_min,
            barrio_key,
            intervalo_inicio,
            intervalo_fin,
            ocupacion_pagada_proxy,
            lag(intervalo_fin) over (
                partition by granularidad_min, barrio_key
                order by intervalo_inicio
            ) as prev_intervalo_fin,
            lag(ocupacion_pagada_proxy) over (
                partition by granularidad_min, barrio_key
                order by intervalo_inicio
            ) as prev_ocupacion_pagada_proxy
        from {read_candidates_sql}
    ), pairs as (
        select
            granularidad_min,
            barrio_key,
            case when prev_intervalo_fin is not null then 1 else 0 end as has_prev,
            case
                when prev_intervalo_fin = intervalo_inicio
                 and ocupacion_pagada_proxy is not null
                 and prev_ocupacion_pagada_proxy is not null
                then 1 else 0
            end as is_contiguous,
            case
                when prev_intervalo_fin = intervalo_inicio
                 and ocupacion_pagada_proxy is not null
                 and prev_ocupacion_pagada_proxy is not null
                then abs(ocupacion_pagada_proxy - prev_ocupacion_pagada_proxy)
                else null
            end as abs_delta_contiguous
        from ordered
    )
    select
        granularidad_min,
        barrio_key,
        sum(has_prev) as n_pairs_total,
        sum(is_contiguous) as n_pairs_contiguous,
        sum(is_contiguous)::double / nullif(sum(has_prev), 0) as pct_pairs_contiguous,
        avg(abs_delta_contiguous) as mean_abs_delta_contiguous,
        quantile_cont(abs_delta_contiguous, 0.50) as p50_abs_delta_contiguous,
        quantile_cont(abs_delta_contiguous, 0.90) as p90_abs_delta_contiguous,
        quantile_cont(abs_delta_contiguous, 0.95) as p95_abs_delta_contiguous
    from pairs
    group by granularidad_min, barrio_key
""").df()

granularity_contiguous_variability_summary = (
    granularity_contiguous_variability_by_barrio
    .groupby("granularidad_min", as_index=False)
    .agg(
        n_barrios=("barrio_key", "nunique"),
        n_pairs_contiguous_total=("n_pairs_contiguous", "sum"),
        mean_abs_delta_contiguous_mean_barrio=("mean_abs_delta_contiguous", "mean"),
        p50_abs_delta_contiguous_mean_barrio=("mean_abs_delta_contiguous", "median"),
        p90_abs_delta_contiguous_mean_barrio=("mean_abs_delta_contiguous", lambda s: s.quantile(0.90)),
        p95_abs_delta_contiguous_mean_barrio=("mean_abs_delta_contiguous", lambda s: s.quantile(0.95)),
        mean_pct_pairs_contiguous=("pct_pairs_contiguous", "mean"),
    )
)

candidate_15min_df = con.sql(f"""
    select
        barrio_key,
        intervalo_inicio,
        ocupacion_pagada_proxy
    from {read_candidates_sql}
    where granularidad_min = 15
      and ocupacion_pagada_proxy is not null
""").df()
candidate_15min_df["intervalo_inicio"] = pd.to_datetime(candidate_15min_df["intervalo_inicio"])
candidate_15min_df["fecha"] = candidate_15min_df["intervalo_inicio"].dt.normalize()
candidate_15min_df["block_30_start"] = candidate_15min_df["intervalo_inicio"].dt.floor("30min")
candidate_15min_df["block_60_start"] = candidate_15min_df["intervalo_inicio"].dt.floor("60min")
anchor_09 = candidate_15min_df["fecha"] + pd.Timedelta(hours=9)
minutes_from_anchor = (candidate_15min_df["intervalo_inicio"] - anchor_09).dt.total_seconds().div(60)
candidate_15min_df["block_45_start"] = anchor_09 + pd.to_timedelta(np.floor(minutes_from_anchor / 45) * 45, unit="min")

within_block_rows = []
for block_minutes, block_column, expected_subintervals in [
    (30, "block_30_start", 2),
    (45, "block_45_start", 3),
    (60, "block_60_start", 4),
]:
    block_stats = (
        candidate_15min_df
        .groupby(["barrio_key", block_column])["ocupacion_pagada_proxy"]
        .agg(
            n_subintervals="size",
            mean_ocupacion_15="mean",
            std_ocupacion_15="std",
            min_ocupacion_15="min",
            max_ocupacion_15="max",
        )
        .reset_index()
    )
    block_stats = block_stats.loc[block_stats["n_subintervals"].eq(expected_subintervals)].copy()
    if block_stats.empty:
        within_block_rows.append({
            "block_minutes": block_minutes,
            "n_blocks": 0,
            "mean_std_ocupacion_15": np.nan,
            "p50_std_ocupacion_15": np.nan,
            "p90_std_ocupacion_15": np.nan,
            "mean_range_ocupacion_15": np.nan,
            "p50_range_ocupacion_15": np.nan,
            "p90_range_ocupacion_15": np.nan,
            "mean_abs_dev_15": np.nan,
            "p50_abs_dev_15": np.nan,
            "p90_abs_dev_15": np.nan,
        })
        continue
    block_stats["range_ocupacion_15"] = block_stats["max_ocupacion_15"] - block_stats["min_ocupacion_15"]
    block_values = candidate_15min_df.merge(
        block_stats[["barrio_key", block_column, "mean_ocupacion_15"]],
        on=["barrio_key", block_column],
        how="inner",
    )
    abs_dev_by_block = (
        block_values
        .assign(abs_dev=lambda df: (df["ocupacion_pagada_proxy"] - df["mean_ocupacion_15"]).abs())
        .groupby(["barrio_key", block_column])["abs_dev"]
        .mean()
        .reset_index(name="mean_abs_dev_15")
    )
    block_stats = block_stats.merge(abs_dev_by_block, on=["barrio_key", block_column], how="left")
    within_block_rows.append({
        "block_minutes": block_minutes,
        "n_blocks": len(block_stats),
        "mean_std_ocupacion_15": block_stats["std_ocupacion_15"].mean(),
        "p50_std_ocupacion_15": block_stats["std_ocupacion_15"].quantile(0.50),
        "p90_std_ocupacion_15": block_stats["std_ocupacion_15"].quantile(0.90),
        "mean_range_ocupacion_15": block_stats["range_ocupacion_15"].mean(),
        "p50_range_ocupacion_15": block_stats["range_ocupacion_15"].quantile(0.50),
        "p90_range_ocupacion_15": block_stats["range_ocupacion_15"].quantile(0.90),
        "mean_abs_dev_15": block_stats["mean_abs_dev_15"].mean(),
        "p50_abs_dev_15": block_stats["mean_abs_dev_15"].quantile(0.50),
        "p90_abs_dev_15": block_stats["mean_abs_dev_15"].quantile(0.90),
    })
granularity_15min_within_block_variability = pd.DataFrame(within_block_rows)

reference_range = granularity_15min_within_block_variability["mean_range_ocupacion_15"].replace(0, np.nan)
low_variation_threshold = reference_range.quantile(0.33)
high_variation_threshold = reference_range.quantile(0.67)
reading_rows = []
for _, row in granularity_15min_within_block_variability.iterrows():
    block_minutes = int(row["block_minutes"])
    mean_range = row["mean_range_ocupacion_15"]
    if pd.isna(mean_range):
        lectura = "no hay bloques completos suficientes para evaluar la ganancia interna"
        implicacion = "no usar este bloque como criterio único"
    elif mean_range <= low_variation_threshold:
        lectura = "la variación interna de 15 minutos dentro del bloque es baja"
        implicacion = f"15 minutos aporta poca ganancia frente a {block_minutes} minutos según esta señal"
    elif mean_range >= high_variation_threshold:
        lectura = "la variación interna de 15 minutos dentro del bloque es elevada"
        implicacion = f"{block_minutes} minutos podría suavizar cambios intra-horarios relevantes"
    else:
        lectura = "la variación interna de 15 minutos dentro del bloque es intermedia"
        implicacion = "el criterio de coste e interpretabilidad debe ponderarse junto con la señal temporal"
    reading_rows.append({
        "block_minutes": block_minutes,
        "lectura": lectura,
        "implicacion_para_granularidad": implicacion,
    })
granularity_incremental_resolution_reading = pd.DataFrame(reading_rows)

contiguous_signal = granularity_contiguous_variability_summary.set_index("granularidad_min")["mean_abs_delta_contiguous_mean_barrio"]
contiguous_signal_median = contiguous_signal.median()
cost_lookup = granularity_core_control_summary.set_index("granularidad_min")
interpretability_map = {
    15: "alta resolución; mayor coste y posible ruido",
    30: "equilibrio entre detalle e interpretabilidad",
    45: "intermedia, menos estándar",
    60: "más estable e interpretable; menor detalle",
}
granularity_selection_compact_matrix = pd.DataFrame([
    {
        "granularidad_min": granularity,
        "detalle_temporal": {
            15: "muy alto",
            30: "alto",
            45: "medio",
            60: "menor",
        }.get(granularity),
        "senal_temporal_util": "alta" if contiguous_signal.get(granularity, np.nan) >= contiguous_signal_median else "moderada/menor",
        "coste_relativo": cost_lookup.loc[granularity, "relative_rows_vs_60"] if granularity in cost_lookup.index else np.nan,
        "interpretabilidad": interpretability_map.get(granularity),
        "lectura": "evidencia para decidir; no fija automáticamente la granularidad final",
    }
    for granularity in sorted(granularity_core_control_summary["granularidad_min"].astype(int).tolist())
])

delta_columns = [
    "mean_abs_delta_contiguous",
    "p50_abs_delta_contiguous",
    "p90_abs_delta_contiguous",
    "p95_abs_delta_contiguous",
]
granularity_selection_checks = pd.DataFrame([
    {
        "check": "hay_pares_contiguos",
        "ok": bool(granularity_contiguous_variability_summary["n_pairs_contiguous_total"].sum() > 0),
        "detalle": "existen pares contiguos reales dentro de ventanas SER observables",
    },
    {
        "check": "todas_las_granularidades_presentes",
        "ok": present_granularities == expected_granularities,
        "detalle": f"presentes: {sorted(present_granularities)}",
    },
    {
        "check": "sin_deltas_negativos",
        "ok": bool((granularity_contiguous_variability_by_barrio[delta_columns].dropna(how="all") >= 0).all().all()),
        "detalle": "las diferencias absolutas contiguas no son negativas",
    },
    {
        "check": "bloques_15min_generados",
        "ok": bool((granularity_15min_within_block_variability["n_blocks"] > 0).all()),
        "detalle": "hay bloques completos de 30, 45 y 60 minutos formados desde subintervalos de 15 minutos",
    },
    {
        "check": "diagnosticos_no_vacios",
        "ok": all([
            not granularity_core_control_summary.empty,
            not granularity_contiguous_variability_summary.empty,
            not granularity_15min_within_block_variability.empty,
            not granularity_incremental_resolution_reading.empty,
            not granularity_selection_compact_matrix.empty,
        ]),
        "detalle": "las tablas compactas de selección contienen filas",
    },
    {
        "check": "outputs_compactos",
        "ok": True,
        "detalle": "la sección muestra solo controles, variabilidad contigua, ganancia marginal y matriz compacta",
    },
])
granularity_selection_failed_checks = granularity_selection_checks.loc[~granularity_selection_checks["ok"]].copy()

granularity_core_control_compact = granularity_core_control_summary[[
    "granularidad_min",
    "n_rows",
    "pct_ocupacion_eq_0",
    "n_ocupacion_gt_1",
    "relative_rows_vs_30",
    "relative_rows_vs_60",
]].copy()

granularity_within_block_compact = granularity_15min_within_block_variability[[
    "block_minutes",
    "n_blocks",
    "mean_range_ocupacion_15",
    "p50_range_ocupacion_15",
    "p90_range_ocupacion_15",
    "mean_abs_dev_15",
]].copy()

within_block_lookup = granularity_within_block_compact.set_index("block_minutes")
cost_lookup_compact = granularity_core_control_summary.set_index("granularidad_min")
range_30 = within_block_lookup.loc[30, "mean_range_ocupacion_15"] if 30 in within_block_lookup.index else np.nan
range_60 = within_block_lookup.loc[60, "mean_range_ocupacion_15"] if 60 in within_block_lookup.index else np.nan
range_gap_30_60 = range_60 - range_30 if pd.notna(range_30) and pd.notna(range_60) else np.nan

granularity_final_decision_matrix = pd.DataFrame([
    {
        "granularidad_min": 15,
        "evidencia_temporal": (
            "máxima resolución temporal; la ganancia frente a 30 debe justificarse por la variación interna observada"
            if pd.notna(range_30) and range_30 > 0
            else "máxima resolución temporal; la ganancia frente a 30 parece limitada en la señal observada"
        ),
        "coste_relativo": cost_lookup_compact.loc[15, "relative_rows_vs_30"] if 15 in cost_lookup_compact.index else np.nan,
        "interpretabilidad": "menor por volumen y posible ruido de corto plazo",
        "decision_metodologica": "descartar como core y conservar como sensibilidad",
    },
    {
        "granularidad_min": 30,
        "evidencia_temporal": "equilibra variación contigua del target y reducción de resolución frente a 15 minutos",
        "coste_relativo": cost_lookup_compact.loc[30, "relative_rows_vs_30"] if 30 in cost_lookup_compact.index else np.nan,
        "interpretabilidad": "alta; unidad temporal estándar y legible",
        "decision_metodologica": "candidata principal para el panel final",
    },
    {
        "granularidad_min": 45,
        "evidencia_temporal": "intermedia; puede capturar parte de la señal intra-horaria pero con unidad menos convencional",
        "coste_relativo": cost_lookup_compact.loc[45, "relative_rows_vs_30"] if 45 in cost_lookup_compact.index else np.nan,
        "interpretabilidad": "menor; granularidad menos estándar para lectura y comunicación",
        "decision_metodologica": "descartar como core",
    },
    {
        "granularidad_min": 60,
        "evidencia_temporal": (
            "menor coste, pero la variación interna frente a 30 sugiere riesgo de suavizado intra-horario"
            if pd.notna(range_gap_30_60) and range_gap_30_60 > 0
            else "menor coste; si la diferencia frente a 30 es pequeña, gana peso la interpretabilidad"
        ),
        "coste_relativo": cost_lookup_compact.loc[60, "relative_rows_vs_30"] if 60 in cost_lookup_compact.index else np.nan,
        "interpretabilidad": "muy alta, aunque con menor detalle temporal",
        "decision_metodologica": "descartar como core por excesiva agregación",
    },
])

display(granularity_core_control_compact)
display(granularity_contiguous_variability_summary)
display(granularity_within_block_compact)
display(granularity_final_decision_matrix)

if granularity_core_failed_checks.empty and granularity_selection_failed_checks.empty:
    display(Markdown("No hay checks fallidos en el diagnóstico compacto de selección de granularidad."))
else:
    if not granularity_core_failed_checks.empty:
        display(granularity_core_failed_checks)
    if not granularity_selection_failed_checks.empty:
        display(granularity_selection_failed_checks)

,granularidad_min,n_rows,pct_ocupacion_eq_0,n_ocupacion_gt_1,relative_rows_vs_30,relative_rows_vs_60
0,15,2584296,0.038757,0.0,2.000000,4.000000
1,30,1292148,0.038619,0.0,1.000000,2.000000
2,45,861432,0.038521,0.0,0.666667,1.333333
3,60,646074,0.038475,0.0,0.500000,1.000000


,granularidad_min,n_barrios,n_pairs_contiguous_total,mean_abs_delta_contiguous_mean_barrio,p50_abs_delta_contiguous_mean_barrio,p90_abs_delta_contiguous_mean_barrio,p95_abs_delta_contiguous_mean_barrio,mean_pct_pairs_contiguous
0,15,65,2523206.0,0.004489,0.004501,0.006272,0.007051,0.976390
1,30,65,1231058.0,0.007074,0.007071,0.009884,0.010612,0.952778
2,45,65,800342.0,0.009028,0.009148,0.012654,0.013703,0.929166
3,60,65,584984.0,0.010842,0.010914,0.015341,0.016793,0.905552


,block_minutes,n_blocks,mean_range_ocupacion_15,p50_range_ocupacion_15,p90_range_ocupacion_15,mean_abs_dev_15
0,30,1292148,0.004885,0.003082,0.011367,0.002443
1,45,861432,0.009349,0.006230,0.021542,0.003515
2,60,646074,0.012789,0.008246,0.030584,0.004198


,granularidad_min,evidencia_temporal,coste_relativo,interpretabilidad,decision_metodologica
0,15,máxima resolución temporal; la ganancia frente a 30 debe justificarse por la variación interna observada,2.000000,menor por volumen y posible ruido de corto plazo,descartar como core y conservar como sensibilidad
1,30,equilibra variación contigua del target y reducción de resolución frente a 15 minutos,1.000000,alta; unidad temporal estándar y legible,candidata principal para el panel final
2,45,intermedia; puede capturar parte de la señal intra-horaria pero con unidad menos convencional,0.666667,menor; granularidad menos estándar para lectura y comunicación,descartar como core
3,60,"menor coste, pero la variación interna frente a 30 sugiere riesgo de suavizado intra-horario",0.500000,"muy alta, aunque con menor detalle temporal",descartar como core por excesiva agregación


No hay checks fallidos en el diagnóstico compacto de selección de granularidad.

### Lectura metodológica y selección de granularidad final

El diagnóstico compacto de granularidad permite seleccionar una unidad temporal final para el panel `ser_barrio_intervalo_global_final`. La decisión se apoya en `ocupacion_pagada_proxy`, que es la métrica central del target histórico global barrio-intervalo. Las demás métricas del panel se mantienen como diagnósticas, pero no se utilizan como criterio principal para fijar la granularidad temporal.

Los controles mínimos confirman que las cuatro granularidades candidatas —15, 30, 45 y 60 minutos— son técnicamente válidas. Todas conservan la malla barrio-intervalo esperada, no presentan valores de `ocupacion_pagada_proxy` superiores a 1 y mantienen un porcentaje muy similar de intervalos sin señal pagada observada, alrededor del 3,85 %. Por tanto, la selección no viene determinada por fallos técnicos, valores no plausibles ni diferencias relevantes de sparsity.

La comparación de coste muestra una diferencia clara de volumen. La granularidad de 15 minutos genera 2.584.296 filas, el doble que la granularidad de 30 minutos y cuatro veces el tamaño de 60 minutos. La granularidad de 30 minutos reduce el panel a 1.292.148 filas, manteniendo todavía una resolución intra-horaria suficiente para describir cambios dentro del régimen SER. Las granularidades de 45 y 60 minutos reducen más el tamaño, pero a costa de una mayor agregación temporal.

La variabilidad contigua de `ocupacion_pagada_proxy` confirma que existe señal temporal dentro de cada barrio. No obstante, esta métrica debe interpretarse con cautela: al aumentar la duración del intervalo, también aumenta la separación temporal entre observaciones consecutivas. Por ello, una mayor diferencia media entre intervalos no implica automáticamente que la granularidad sea más adecuada. La decisión no consiste en seleccionar la granularidad más variable, sino la que conserva señal útil sin introducir una resolución innecesaria.

La ganancia marginal de resolución es el criterio más informativo para esta decisión. Al tomar la granularidad de 15 minutos como referencia interna, se observa que la variación media dentro de bloques de 30 minutos es limitada en comparación con la observada dentro de bloques de 45 y 60 minutos. El rango medio de `ocupacion_pagada_proxy` dentro de bloques de 30 minutos es 0,004885, mientras que aumenta a 0,009349 en bloques de 45 minutos y a 0,012789 en bloques de 60 minutos. Esto sugiere que pasar de 15 a 30 minutos apenas sacrifica señal intra-horaria, mientras que agregaciones más gruesas empiezan a suavizar cambios más relevantes.

Con esta evidencia, se selecciona la granularidad de 30 minutos como granularidad final del panel global SER. Esta opción ofrece el mejor equilibrio entre señal temporal útil, coste computacional, estabilidad e interpretabilidad. La granularidad de 15 minutos se descarta como core porque duplica el volumen del panel respecto a 30 minutos sin aportar una ganancia marginal suficiente para justificar ese coste adicional. La granularidad de 45 minutos queda descartada por su menor interpretabilidad y por ser una unidad temporal menos estándar. La granularidad de 60 minutos, aunque más ligera, se descarta como granularidad principal porque presenta mayor riesgo de suavizar variaciones intra-horarias relevantes.

La decisión queda fijada como:

```python
SELECTED_GRANULARITY_MIN = 30
```

A partir de esta selección, la siguiente sección construirá `ser_barrio_intervalo_global_final` filtrando el panel candidato a `granularidad_min = 30`, sin recalcular solapes ni volver a procesar tiques individuales.


## 18. Construcción de `ser_barrio_intervalo_global_final`

Una vez seleccionada la granularidad final de 30 minutos, esta sección construye la salida definitiva del panel global `SER_barrio_intervalo`. No se recalculan solapes ni se vuelven a leer tiques individuales. La sección parte exclusivamente de `ser_barrio_intervalo_global_candidates`, generado previamente para las cuatro granularidades candidatas.

La operación consiste en filtrar el panel candidato a:

```python
SELECTED_GRANULARITY_MIN = 30
```

y construir una tabla final con una fila por:

```text
barrio_key × intervalo_inicio
```

manteniendo la columna `granularidad_min` como trazabilidad metodológica. Esta salida conserva las métricas proxy calculadas previamente: `ocupacion_pagada_proxy`, stock de señal pagada al inicio y al final, afluencia, liberación, persistencia, saldo de flujo y rotación bruta.

La sección realiza tres tareas:

1. **Lectura segura del panel candidato.**
   Se lee `ser_barrio_intervalo_global_candidates` evitando la inferencia automática de particiones Hive, ya que las columnas de partición ya están persistidas en los ficheros Parquet.

2. **Filtrado y limpieza de la granularidad final.**
   Se conserva únicamente `granularidad_min = 30`, se ordena el panel por periodo, barrio e intervalo, y se revisan tipos de columnas para que los conteos queden como enteros y las métricas normalizadas como numéricas continuas.

3. **Validación y escritura de salida final.**
   Se comprueba que la salida final contiene los 13 periodos disponibles, el horizonte temporal esperado, los barrios con capacidad válida, ausencia de valores de `ocupacion_pagada_proxy > 1`, métricas finitas y coherencia de filas respecto al panel candidato de 30 minutos.

El resultado de esta sección será `ser_barrio_intervalo_global_final`, que constituye el panel histórico principal del bloque SER. Esta tabla será la base para el dataset de modelado, los análisis temporales y la representación cartográfica del proxy de dificultad SER en superficie.


In [33]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

if "ROOT" not in globals():
    def find_repo_root_for_global_final(start: Path | None = None) -> Path:
        current = Path.cwd().resolve() if start is None else Path(start).resolve()
        for candidate in [current, *current.parents]:
            if (candidate / "data_catalog.csv").exists():
                return candidate
        raise FileNotFoundError("No se ha encontrado data_catalog.csv al ascender desde el directorio actual.")

    ROOT = find_repo_root_for_global_final()

if "OUTPUT_GLOBAL_CANDIDATES_DIR" not in globals():
    OUTPUT_GLOBAL_CANDIDATES_DIR = ROOT / "data/processed/core/ser/ser_barrio_intervalo_global_candidates"

SELECTED_GRANULARITY_MIN = 30
if "OUTPUT_GLOBAL_FINAL_PATH" not in globals():
    OUTPUT_GLOBAL_FINAL_PATH = ROOT / "data/processed/core/ser/ser_barrio_intervalo_global_final.parquet"

BUILD_GLOBAL_FINAL_PANEL = True
WRITE_GLOBAL_FINAL_PANEL = False
OVERWRITE_GLOBAL_FINAL_PANEL = False

final_panel_columns = [
    "granularidad_min",
    "periodo_inicio",
    "barrio_key",
    "intervalo_inicio",
    "intervalo_fin",
    "anio_intervalo",
    "anio",
    "plazas_barrio_anio",
    "n_tiques_solapados",
    "minutos_solapados",
    "n_A",
    "n_B",
    "n_C",
    "n_D",
    "n_activo_inicio",
    "n_activo_fin",
    "n_entra_intervalo",
    "n_sale_intervalo",
    "n_persistente_completo",
    "ocupacion_pagada_proxy",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]
count_columns_final = [
    "n_tiques_solapados",
    "n_A",
    "n_B",
    "n_C",
    "n_D",
    "n_activo_inicio",
    "n_activo_fin",
    "n_entra_intervalo",
    "n_sale_intervalo",
    "n_persistente_completo",
]
continuous_columns_final = [
    "minutos_solapados",
    "ocupacion_pagada_proxy",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]
integer_columns_final = ["granularidad_min", "anio_intervalo", "anio", "plazas_barrio_anio"]
metric_columns_final = [
    "ocupacion_pagada_proxy",
    "stock_inicio_proxy",
    "stock_fin_proxy",
    "afluencia_tiques",
    "liberacion_tiques",
    "persistencia_completa",
    "saldo_flujo",
    "rotacion_bruta",
]

if not BUILD_GLOBAL_FINAL_PANEL:
    display(Markdown("`BUILD_GLOBAL_FINAL_PANEL=False`: no se construye el panel global final en esta ejecución."))
else:
    if not OUTPUT_GLOBAL_CANDIDATES_DIR.exists():
        raise FileNotFoundError(f"No existe OUTPUT_GLOBAL_CANDIDATES_DIR: {OUTPUT_GLOBAL_CANDIDATES_DIR}")
    candidate_path_pattern = str(OUTPUT_GLOBAL_CANDIDATES_DIR / "**/*.parquet")
    candidate_parquet_files = sorted(OUTPUT_GLOBAL_CANDIDATES_DIR.glob("**/*.parquet"))
    if not candidate_parquet_files:
        raise FileNotFoundError(f"No hay ficheros Parquet bajo {OUTPUT_GLOBAL_CANDIDATES_DIR}")

    candidate_path_pattern_sql = candidate_path_pattern.replace("'", "''")
    read_candidates_sql = f"read_parquet('{candidate_path_pattern_sql}', hive_partitioning=false)"
    con = duckdb.connect(database=":memory:")

    source_columns = con.sql(f"select * from {read_candidates_sql} limit 0").df().columns.tolist()
    missing_final_columns = [column for column in final_panel_columns if column not in source_columns]
    if missing_final_columns:
        raise ValueError(f"Faltan columnas mínimas en los candidatos globales: {missing_final_columns}")

    expected_rows_30 = con.sql(f"""
        select count(*) as n_rows
        from {read_candidates_sql}
        where granularidad_min = {SELECTED_GRANULARITY_MIN}
    """).df()["n_rows"].iloc[0]

    select_columns_sql = ",\n        ".join(final_panel_columns)
    ser_barrio_intervalo_global_final_df = con.sql(f"""
        select
            {select_columns_sql}
        from {read_candidates_sql}
        where granularidad_min = {SELECTED_GRANULARITY_MIN}
        order by periodo_inicio, barrio_key, intervalo_inicio
    """).df()

    for column in [*count_columns_final, *integer_columns_final]:
        if column in ser_barrio_intervalo_global_final_df.columns and ser_barrio_intervalo_global_final_df[column].notna().all():
            ser_barrio_intervalo_global_final_df[column] = ser_barrio_intervalo_global_final_df[column].astype("int64")
    for column in continuous_columns_final:
        ser_barrio_intervalo_global_final_df[column] = pd.to_numeric(
            ser_barrio_intervalo_global_final_df[column],
            errors="coerce",
        )
    ser_barrio_intervalo_global_final_df["intervalo_inicio"] = pd.to_datetime(ser_barrio_intervalo_global_final_df["intervalo_inicio"])
    ser_barrio_intervalo_global_final_df["intervalo_fin"] = pd.to_datetime(ser_barrio_intervalo_global_final_df["intervalo_fin"])

    n_rows_final = len(ser_barrio_intervalo_global_final_df)
    metric_values_final = ser_barrio_intervalo_global_final_df[metric_columns_final]
    finite_metrics_final = np.isfinite(metric_values_final.to_numpy(dtype=float))
    duplicates_final = ser_barrio_intervalo_global_final_df.duplicated(["barrio_key", "intervalo_inicio"]).sum()
    ordered_reference = ser_barrio_intervalo_global_final_df.sort_values(["periodo_inicio", "barrio_key", "intervalo_inicio"]).index
    expected_intervalo_min = pd.Timestamp("2023-01-02 09:00:00")
    expected_intervalo_max = pd.Timestamp("2026-03-31 21:00:00")

    output_exists = OUTPUT_GLOBAL_FINAL_PATH.exists()
    output_blocked_by_existing_file = bool(output_exists and not OVERWRITE_GLOBAL_FINAL_PANEL)
    output_written = False
    output_message = None

    global_final_panel_summary = pd.DataFrame([
        {
            "n_rows": n_rows_final,
            "n_periodos": ser_barrio_intervalo_global_final_df["periodo_inicio"].nunique(dropna=True),
            "n_barrios": ser_barrio_intervalo_global_final_df["barrio_key"].nunique(dropna=True),
            "granularidad_min_unica": sorted(ser_barrio_intervalo_global_final_df["granularidad_min"].dropna().unique().tolist()),
            "intervalo_min": ser_barrio_intervalo_global_final_df["intervalo_inicio"].min(),
            "intervalo_max": ser_barrio_intervalo_global_final_df["intervalo_fin"].max(),
            "minutos_solapados_total": ser_barrio_intervalo_global_final_df["minutos_solapados"].sum(),
            "n_ocupacion_eq_0": int((ser_barrio_intervalo_global_final_df["ocupacion_pagada_proxy"] == 0).sum()),
            "pct_ocupacion_eq_0": (ser_barrio_intervalo_global_final_df["ocupacion_pagada_proxy"] == 0).mean(),
            "n_ocupacion_gt_1": int((ser_barrio_intervalo_global_final_df["ocupacion_pagada_proxy"] > 1).sum()),
            "n_capacidad_nula": int(ser_barrio_intervalo_global_final_df["plazas_barrio_anio"].isna().sum()),
            "n_metricas_nulas": int(ser_barrio_intervalo_global_final_df[metric_columns_final].isna().sum().sum()),
            "output_path": str(OUTPUT_GLOBAL_FINAL_PATH.relative_to(ROOT)) if OUTPUT_GLOBAL_FINAL_PATH.is_relative_to(ROOT) else str(OUTPUT_GLOBAL_FINAL_PATH),
            "se_escribe_output": bool(WRITE_GLOBAL_FINAL_PANEL and not output_blocked_by_existing_file),
        }
    ])

    global_final_panel_checks = pd.DataFrame([
        {
            "check": "solo_granularidad_30",
            "ok": ser_barrio_intervalo_global_final_df["granularidad_min"].nunique(dropna=True) == 1
            and int(ser_barrio_intervalo_global_final_df["granularidad_min"].iloc[0]) == SELECTED_GRANULARITY_MIN,
            "critico": True,
            "detalle": "la salida final conserva únicamente granularidad_min=30",
        },
        {
            "check": "n_rows_esperado_30",
            "ok": n_rows_final == expected_rows_30,
            "critico": True,
            "detalle": f"filas finales: {n_rows_final}; filas candidatas 30 min: {expected_rows_30}",
        },
        {
            "check": "n_periodos_13",
            "ok": ser_barrio_intervalo_global_final_df["periodo_inicio"].nunique(dropna=True) == 13,
            "critico": True,
            "detalle": "la salida final conserva los 13 periodos disponibles",
        },
        {
            "check": "n_barrios_65",
            "ok": ser_barrio_intervalo_global_final_df["barrio_key"].nunique(dropna=True) == 65,
            "critico": True,
            "detalle": "la salida final conserva 65 barrios con capacidad SER",
        },
        {
            "check": "horizonte_temporal_esperado",
            "ok": ser_barrio_intervalo_global_final_df["intervalo_inicio"].min() == expected_intervalo_min
            and ser_barrio_intervalo_global_final_df["intervalo_fin"].max() == expected_intervalo_max,
            "critico": True,
            "detalle": "horizonte esperado: 2023-01-02 09:00:00 a 2026-03-31 21:00:00",
        },
        {
            "check": "sin_ocupacion_gt_1",
            "ok": bool((ser_barrio_intervalo_global_final_df["ocupacion_pagada_proxy"] <= 1).all()),
            "critico": True,
            "detalle": "no hay valores de ocupacion_pagada_proxy superiores a 1",
        },
        {
            "check": "sin_capacidad_nula",
            "ok": bool(ser_barrio_intervalo_global_final_df["plazas_barrio_anio"].notna().all()),
            "critico": True,
            "detalle": "todas las filas tienen denominador de capacidad",
        },
        {
            "check": "metricas_finitas",
            "ok": bool(finite_metrics_final.all()),
            "critico": True,
            "detalle": "no hay inf ni -inf en las métricas normalizadas",
        },
        {
            "check": "conteos_no_negativos",
            "ok": bool((ser_barrio_intervalo_global_final_df[count_columns_final] >= 0).all().all()),
            "critico": True,
            "detalle": "todos los conteos agregados son no negativos",
        },
        {
            "check": "intervalos_inicio_menor_fin",
            "ok": bool((ser_barrio_intervalo_global_final_df["intervalo_inicio"] < ser_barrio_intervalo_global_final_df["intervalo_fin"]).all()),
            "critico": True,
            "detalle": "todos los intervalos cumplen inicio < fin",
        },
        {
            "check": "sin_duplicados_barrio_intervalo",
            "ok": duplicates_final == 0,
            "critico": True,
            "detalle": f"duplicados barrio_key × intervalo_inicio: {duplicates_final}",
        },
        {
            "check": "orden_logico_intervalos",
            "ok": bool(ser_barrio_intervalo_global_final_df.index.equals(ordered_reference)),
            "critico": True,
            "detalle": "la tabla está ordenada por periodo_inicio, barrio_key e intervalo_inicio",
        },
    ])
    global_final_panel_failed_checks = global_final_panel_checks.loc[~global_final_panel_checks["ok"]].copy()

    if BUILD_GLOBAL_FINAL_PANEL and WRITE_GLOBAL_FINAL_PANEL:
        if output_blocked_by_existing_file:
            output_message = (
                f"`{OUTPUT_GLOBAL_FINAL_PATH}` ya existe y `OVERWRITE_GLOBAL_FINAL_PANEL=False`; "
                "no se sobrescribe la salida final."
            )
        elif global_final_panel_failed_checks.empty:
            OUTPUT_GLOBAL_FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)
            ser_barrio_intervalo_global_final_df.to_parquet(
                OUTPUT_GLOBAL_FINAL_PATH,
                index=False,
                compression="snappy",
            )
            output_written = True
            output_message = f"Salida final escrita en `{OUTPUT_GLOBAL_FINAL_PATH.relative_to(ROOT)}`."
        else:
            output_message = "No se escribe la salida final porque existen checks críticos fallidos."
    else:
        output_message = "`WRITE_GLOBAL_FINAL_PANEL=False`: la salida final se ha construido y validado en memoria, pero no se ha escrito en disco."

    global_final_panel_summary["se_escribe_output"] = output_written

    display(global_final_panel_summary)
    if global_final_panel_failed_checks.empty:
        display(Markdown("No hay checks fallidos en la construcción de ser_barrio_intervalo_global_final."))
    else:
        display(global_final_panel_failed_checks)
    display(Markdown(output_message))

,n_rows,n_periodos,n_barrios,granularidad_min_unica,intervalo_min,intervalo_max,minutos_solapados_total,n_ocupacion_eq_0,pct_ocupacion_eq_0,n_ocupacion_gt_1,n_capacidad_nula,n_metricas_nulas,output_path,se_escribe_output
0,1292148,13,65,[30],2023-01-02 09:00:00,2026-03-31 21:00:00,1.034175e+10,49902,0.038619,0,0,0,data/processed/core/ser/ser_barrio_intervalo_global_final.parquet,False


No hay checks fallidos en la construcción de ser_barrio_intervalo_global_final.

`WRITE_GLOBAL_FINAL_PANEL=False`: la salida final se ha construido y validado en memoria, pero no se ha escrito en disco.

## 19. Cierre metodológico del target global

El notebook deja construido `ser_barrio_intervalo_global_final` como panel histórico principal del bloque SER. La unidad final es:

```text
barrio_key × intervalo_inicio
```

con granularidad temporal de 30 minutos.

El target central es `ocupacion_pagada_proxy`. Esta variable resume la intensidad de uso pagado observado en SER durante cada intervalo, calculada a partir de minutos solapados y capacidad anual del barrio:

```text
ocupacion_pagada_proxy = minutos_solapados / (plazas_barrio_anio × granularidad_min)
```

Debe interpretarse como un proxy histórico de presión pagada en superficie, no como ocupación real observada de todas las plazas SER. Los tiques permiten medir demanda pagada registrada, pero no capturan vehículos exentos, usos no pagados, ocupación irregular ni disponibilidad real plaza a plaza.

La salida final conserva también métricas auxiliares de stock, afluencia, liberación, persistencia, saldo y rotación. Estas variables permiten describir la dinámica interna de la señal pagada, pero el target principal para el siguiente bloque será `ocupacion_pagada_proxy`.

En `04_03`, este panel se utilizará como base para preparar el dataset de modelado SER: incorporación de variables temporales y estructurales disponibles antes del intervalo objetivo, análisis de distribución del target, validación temporal y construcción de modelos o baselines predictivos si procede.
